In [1]:
#1
!pip -q install cadquery torch_geometric

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from OCP.Interface import Interface_Static

# Everything is saved on Drive, so it survives disconnects
PROJECT_DIR  = Path("/content/drive/MyDrive/CAD_GNN_Project")
RAW_STEP_DIR = PROJECT_DIR / "raw_step"
GRAPH_DIR    = PROJECT_DIR / "graphs"

for folder in [RAW_STEP_DIR, GRAPH_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Import every CAD file in millimetres
Interface_Static.SetCVal_s("xstep.cascade.unit", "MM")

print("Project folder:", PROJECT_DIR)
print("Unit setting:", Interface_Static.CVal_s("xstep.cascade.unit"))

Mounted at /content/drive
Project folder: /content/drive/MyDrive/CAD_GNN_Project
Unit setting: 


In [2]:
#2
from OCP.STEPControl import STEPControl_Reader
from OCP.Interface import Interface_Static

# Creating a reader registers the STEP parameters
_ = STEPControl_Reader()

ok = Interface_Static.SetCVal_s("xstep.cascade.unit", "MM")
print("SetCVal returned:", ok)
print("Unit setting:", Interface_Static.CVal_s("xstep.cascade.unit"))

SetCVal returned: True
Unit setting: MM


In [3]:
#3
import requests, gzip, time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

BASE_URL     = "https://concept-collection.github.io/abc-step-1000/"
MANIFEST_URL = BASE_URL + "index.json"

N_PILOT  = 20
N_BATCH2 = 120
MAX_WORKERS = 6

# 1. Load the manifest
resp = requests.get(MANIFEST_URL, timeout=60)
resp.raise_for_status()
files = resp.json()["files"]
print("Files listed in manifest:", len(files))

# 2. First file of each source document, in manifest order
selected, seen_docs = [], set()
for item in files:
    parts = item["name"].split("_")
    if len(parts) < 3:
        continue
    doc_id = parts[1]
    if doc_id in seen_docs:
        continue
    seen_docs.add(doc_id)
    selected.append(item)
    if len(selected) == N_PILOT + N_BATCH2:
        break

pilot_items  = selected[:N_PILOT]
batch2_items = selected[N_PILOT:]
print("Pilot:", len(pilot_items), "| Batch 2:", len(batch2_items))

# 3. Download one file (skips if already on Drive)
def download_step(item):
    name = item["name"]
    out  = RAW_STEP_DIR / name

    if out.exists() and out.stat().st_size == item["stepBytes"]:
        return {"filename": name, "status": "ALREADY_EXISTS"}

    try:
        r = requests.get(BASE_URL + item["path"], timeout=180)
        r.raise_for_status()
        data = gzip.decompress(r.content)

        if len(data) != item["stepBytes"]:
            raise ValueError(f"Size mismatch: {len(data)} vs {item['stepBytes']}")

        out.write_bytes(data)
        return {"filename": name, "status": "SUCCESS"}

    except Exception as e:
        return {"filename": name, "status": "FAILED", "error": str(e)}

# 4. Run both groups
def run_group(items, label):
    results, t0 = [], time.time()
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = [ex.submit(download_step, it) for it in items]
        for f in as_completed(futures):
            results.append(f.result())
    df = pd.DataFrame(results)
    print(f"\n{label}: {df['status'].value_counts().to_dict()} "
          f"({time.time()-t0:.0f}s)")
    return df

pilot_df  = run_group(pilot_items,  "PILOT")
batch2_df = run_group(batch2_items, "BATCH 2")

# 5. Save the reports and the group membership
pilot_df.to_csv(PROJECT_DIR / "download_manifest.csv", index=False)
batch2_df.to_csv(PROJECT_DIR / "download_manifest_batch2.csv", index=False)

pd.DataFrame(
    [{"filename": i["name"], "group": "pilot"}  for i in pilot_items] +
    [{"filename": i["name"], "group": "batch2"} for i in batch2_items]
).to_csv(PROJECT_DIR / "download_groups.csv", index=False)

print("\nSTEP files on Drive:", len(list(RAW_STEP_DIR.glob("*.step"))))

Files listed in manifest: 1000
Pilot: 20 | Batch 2: 120

PILOT: {'ALREADY_EXISTS': 20} (0s)

BATCH 2: {'ALREADY_EXISTS': 120} (0s)

STEP files on Drive: 140


In [4]:
#4
import hashlib, time
import numpy as np
import pandas as pd
from pathlib import Path

from OCP.STEPControl import STEPControl_Reader
from OCP.IFSelect import IFSelect_RetDone
from OCP.Interface import Interface_Static
from OCP.BRepCheck import BRepCheck_Analyzer
from OCP.BRepGProp import BRepGProp
from OCP.GProp import GProp_GProps
from OCP.Bnd import Bnd_Box
from OCP.BRepBndLib import BRepBndLib
from OCP.TopExp import TopExp
from OCP.TopTools import TopTools_IndexedMapOfShape
from OCP.TopAbs import (TopAbs_SOLID, TopAbs_SHELL, TopAbs_FACE,
                        TopAbs_EDGE, TopAbs_VERTEX)

PROJECT_DIR  = Path("/content/drive/MyDrive/CAD_GNN_Project")
RAW_STEP_DIR = PROJECT_DIR / "raw_step"
OUT_CSV      = PROJECT_DIR / "geometry_all_140.csv"

groups = pd.read_csv(PROJECT_DIR / "download_groups.csv")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def unique_count(shape, kind):
    m = TopTools_IndexedMapOfShape()
    TopExp.MapShapes_s(shape, kind, m)
    return m.Extent()

def process(filename):
    t0 = time.time()
    path = RAW_STEP_DIR / filename
    rec = {"filename": filename, "sha256": sha256(path),
           "classification": "read_failure", "error": ""}
    try:
        # Create the reader first, then set the unit (every read)
        reader = STEPControl_Reader()
        Interface_Static.SetCVal_s("xstep.cascade.unit", "MM")

        if reader.ReadFile(str(path)) != IFSelect_RetDone:
            raise RuntimeError("STEP could not be read")
        if reader.TransferRoots() == 0:
            raise RuntimeError("No roots transferred")
        shape = reader.OneShape()
        if shape.IsNull():
            raise RuntimeError("Null shape")

        rec["valid_brep"] = BRepCheck_Analyzer(shape).IsValid()
        rec["solids"]   = unique_count(shape, TopAbs_SOLID)
        rec["shells"]   = unique_count(shape, TopAbs_SHELL)
        rec["faces"]    = unique_count(shape, TopAbs_FACE)
        rec["edges"]    = unique_count(shape, TopAbs_EDGE)
        rec["vertices"] = unique_count(shape, TopAbs_VERTEX)

        if not rec["valid_brep"]:
            rec["classification"] = "invalid_brep"
            return rec

        vp = GProp_GProps(); BRepGProp.VolumeProperties_s(shape, vp)
        sp = GProp_GProps(); BRepGProp.SurfaceProperties_s(shape, sp)
        volume, area = abs(vp.Mass()), sp.Mass()

        box = Bnd_Box(); BRepBndLib.Add_s(shape, box)
        x0, y0, z0, x1, y1, z1 = box.Get()
        L, W, H = x1 - x0, y1 - y0, z1 - z0
        diag = float(np.sqrt(L**2 + W**2 + H**2))
        bvol = L * W * H

        vals = [volume, area, L, W, H, diag, bvol]
        if (min(rec["solids"], rec["faces"], rec["edges"], rec["vertices"]) <= 0
                or not np.isfinite(vals).all()
                or volume <= 0 or min(L, W, H) <= 0):
            rec["classification"] = "incomplete_topology"
            return rec

        rec.update({
            "volume": volume, "surface_area": area,
            "bbox_length": L, "bbox_width": W, "bbox_height": H,
            "bbox_diagonal": diag, "bbox_volume": bvol,
            "normalized_volume": volume / diag**3,
            "bbox_volume_fraction": volume / bvol,
            "classification": "primary_candidate",
        })
    except Exception as e:
        rec["error"] = str(e)
    finally:
        rec["time_seconds"] = round(time.time() - t0, 3)
    return rec

# Run on all 140 files
records = []
for i, name in enumerate(groups["filename"], start=1):
    records.append(process(name))
    if i % 20 == 0:
        print(f"{i}/{len(groups)} processed")

df = pd.DataFrame(records).merge(groups, on="filename")

# Exact-duplicate flag (first occurrence kept, in manifest order)
df["duplicate"] = df.duplicated("sha256", keep="first")

df.to_csv(OUT_CSV, index=False)

print("\n===== SUMMARY =====")
print(pd.crosstab(df["group"], df["classification"]))
print("\nExact duplicates:", int(df["duplicate"].sum()))
print("Saved:", OUT_CSV)

20/140 processed
40/140 processed
60/140 processed
80/140 processed
100/140 processed
120/140 processed
140/140 processed

===== SUMMARY =====
classification  incomplete_topology  invalid_brep  primary_candidate
group                                                               
batch2                            1             8                111
pilot                             0             1                 19

Exact duplicates: 0
Saved: /content/drive/MyDrive/CAD_GNN_Project/geometry_all_140.csv


In [5]:
#5
import numpy as np, pandas as pd
from pathlib import Path

df = pd.read_csv(Path("/content/drive/MyDrive/CAD_GNN_Project/geometry_all_140.csv"))
ok = df[df["classification"] == "primary_candidate"].copy()

checks = {
    "140 rows":                     len(df) == 140,
    "filenames unique":             df["filename"].is_unique,
    "values finite and positive":   bool(np.isfinite(ok[["volume","surface_area","bbox_diagonal"]]).all().all()
                                         and (ok[["volume","surface_area","bbox_diagonal"]] > 0).all().all()),
    "volume <= bbox volume":        bool((ok["bbox_volume_fraction"] <= 1 + 1e-6).all()),
    "diagonal >= longest side":     bool((ok["bbox_diagonal"] + 1e-9 >= ok[["bbox_length","bbox_width","bbox_height"]].max(axis=1)).all()),
    "isoperimetric A^3 >= 36*pi*V^2": bool((ok["surface_area"]**3 >= 36*np.pi*ok["volume"]**2 * (1 - 1e-6)).all()),
}

for name, passed in checks.items():
    print("PASS" if passed else "FAIL", "|", name)

bad = ok[ok["surface_area"]**3 < 36*np.pi*ok["volume"]**2 * (1 - 1e-6)]
if len(bad):
    print("\nModels breaking the isoperimetric test:")
    print(bad[["filename","volume","surface_area"]].to_string(index=False))

PASS | 140 rows
PASS | filenames unique
PASS | values finite and positive
PASS | volume <= bbox volume
PASS | diagonal >= longest side
PASS | isoperimetric A^3 >= 36*pi*V^2


In [6]:
#6
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_DIR  = Path("/content/drive/MyDrive/CAD_GNN_Project")
RAW_STEP_DIR = PROJECT_DIR / "raw_step"
OUT_CSV      = PROJECT_DIR / "dataset_100.csv"

N_NEW = 81

df = pd.read_csv(PROJECT_DIR / "geometry_all_140.csv")

# 1. Usable = passed all checks and is not a duplicate
usable = df[(df["classification"] == "primary_candidate") & (~df["duplicate"])].copy()

pilot_keep = usable[usable["group"] == "pilot"].sort_values("filename").reset_index(drop=True)
candidates = usable[usable["group"] == "batch2"].sort_values("filename").reset_index(drop=True)

print("Pilot models kept:", len(pilot_keep))
print("Batch 2 candidates:", len(candidates))

if len(candidates) < N_NEW:
    raise RuntimeError(f"Need {N_NEW} candidates, found {len(candidates)}")

# 2. Describe each candidate with 6 numbers
cols = ["solids", "shells", "faces", "edges", "vertices", "volume"]
features = candidates[cols].to_numpy(dtype=float)

features = np.log1p(features)                     # shrink huge values

fmin = features.min(axis=0)
frange = features.max(axis=0) - fmin
frange[frange == 0] = 1.0                         # avoid dividing by zero
scaled = (features - fmin) / frange               # every column now 0 to 1

# 3. Farthest-point sampling
centre = scaled.mean(axis=0)
first = int(np.argmin(((scaled - centre) ** 2).sum(axis=1)))

chosen = [first]
min_dist = np.linalg.norm(scaled - scaled[first], axis=1)

while len(chosen) < N_NEW:
    nxt = int(np.argmax(min_dist))                # most different from the chosen set
    chosen.append(nxt)
    min_dist = np.minimum(min_dist, np.linalg.norm(scaled - scaled[nxt], axis=1))
    min_dist[chosen] = -1                         # never pick the same one twice

new_models = candidates.iloc[chosen].copy()

# 4. Combine the 19 + 81
pilot_keep["role"] = "pilot"
new_models["role"] = "added"
dataset = pd.concat([pilot_keep, new_models], ignore_index=True)

# 5. Checks
total = len(dataset)
checks = {
    "exactly 100 models":            total == 100,
    "filenames unique":              dataset["filename"].is_unique,
    "19 pilot + 81 added":           (dataset["role"] == "pilot").sum() == 19
                                     and (dataset["role"] == "added").sum() == 81,
    "all STEP files exist":          all((RAW_STEP_DIR / f).exists() for f in dataset["filename"]),
    "no duplicate fingerprints":     dataset["sha256"].is_unique,
    "all volumes positive":          bool((dataset["volume"] > 0).all()),
}

for name, passed in checks.items():
    print("PASS" if passed else "FAIL", "|", name)

if not all(checks.values()):
    raise RuntimeError("A check failed. Nothing was saved.")

dataset.to_csv(OUT_CSV, index=False)
print("\nSaved:", OUT_CSV)
print(dataset.groupby("role")[["faces", "edges", "volume"]].describe().T.round(1))

Pilot models kept: 19
Batch 2 candidates: 111
PASS | exactly 100 models
PASS | filenames unique
PASS | 19 pilot + 81 added
PASS | all STEP files exist
PASS | no duplicate fingerprints
PASS | all volumes positive

Saved: /content/drive/MyDrive/CAD_GNN_Project/dataset_100.csv
role                 added         pilot
faces  count  8.100000e+01  1.900000e+01
       mean   2.603000e+02  3.622000e+02
       std    4.229000e+02  7.617000e+02
       min    3.000000e+00  3.000000e+00
       25%    2.400000e+01  4.250000e+01
       50%    1.190000e+02  8.200000e+01
       75%    3.400000e+02  2.335000e+02
       max    3.124000e+03  2.506000e+03
edges  count  8.100000e+01  1.900000e+01
       mean   6.582000e+02  1.157100e+03
       std    1.023500e+03  2.720800e+03
       min    3.000000e+00  3.000000e+00
       25%    6.000000e+01  1.020000e+02
       50%    2.880000e+02  1.960000e+02
       75%    8.520000e+02  5.820000e+02
       max    7.104000e+03  1.000800e+04
volume count  8.100000e+01  

In [8]:
#7
# =====================================================================
# CHUNK 5A: Build the 4 graphs for ONE model and test them
# =====================================================================
# Goal: turn one CAD model into four graphs (face, edge, vertex, hetero)
# whose numbers do NOT depend on how big the model is.
# =====================================================================


# ---------------------------------------------------------------------
# PART 1: Load the tools we need
# ---------------------------------------------------------------------
import numpy as np                 # maths on lists of numbers
import pandas as pd                # tables (reading our CSV)
import torch                       # the library GNNs are built on
from itertools import combinations # makes all pairs from a list: [1,2,3] -> (1,2),(1,3),(2,3)
from pathlib import Path           # folder locations
from torch_geometric.data import Data, HeteroData
# Data       = a normal graph (one kind of node)
# HeteroData = a graph with several kinds of nodes (faces, edges, vertices)

# --- Tools from OCP (the CAD library) ---
from OCP.STEPControl import STEPControl_Reader   # reads .step files
from OCP.IFSelect import IFSelect_RetDone        # the code meaning "read worked"
from OCP.Interface import Interface_Static       # CAD reader settings (units)
from OCP.TopAbs import TopAbs_FACE, TopAbs_EDGE, TopAbs_VERTEX  # labels for the 3 part types
from OCP.TopExp import TopExp                    # explores a shape's parts
from OCP.TopTools import TopTools_IndexedMapOfShape  # numbered list of unique parts
from OCP.TopoDS import TopoDS                    # converts a generic part to a face/edge/vertex
from OCP.BRepGProp import BRepGProp              # measures area, length, volume
from OCP.GProp import GProp_GProps               # a container that holds those measurements
from OCP.Bnd import Bnd_Box                      # a bounding box
from OCP.BRepBndLib import BRepBndLib            # fills a bounding box from a shape
from OCP.BRepAdaptor import BRepAdaptor_Surface, BRepAdaptor_Curve
# These two let us ask "what KIND of surface/curve is this?"
from OCP.GeomAbs import (GeomAbs_Plane, GeomAbs_Cylinder, GeomAbs_Cone,
                         GeomAbs_Sphere, GeomAbs_Torus,
                         GeomAbs_Line, GeomAbs_Circle)   # the surface/curve kinds
from OCP.BRep import BRep_Tool                   # gets a vertex's x, y, z position
from OCP.gp import gp_Trsf, gp_Pnt               # used only in the "2x bigger" test
from OCP.BRepBuilderAPI import BRepBuilderAPI_Transform  # scales a shape (test only)


# ---------------------------------------------------------------------
# PART 2: Settings
# ---------------------------------------------------------------------
PROJECT_DIR  = Path("/content/drive/MyDrive/CAD_GNN_Project")  # main folder on Drive
RAW_STEP_DIR = PROJECT_DIR / "raw_step"                         # where the .step files are

# How many numbers describe each kind of node.
# Face = 14 numbers, Edge = 4 numbers, Vertex = 3 numbers.
N_FACE_FEATURES, N_EDGE_FEATURES, N_VERTEX_FEATURES = 14, 4, 3


# ---------------------------------------------------------------------
# PART 3: Helper function 1 - read a STEP file into a 3D shape
# ---------------------------------------------------------------------
def load_shape(path):
    reader = STEPControl_Reader()                       # create the reader FIRST
    Interface_Static.SetCVal_s("xstep.cascade.unit", "MM")  # then set units to millimetres
    # ReadFile returns a status code. Anything other than "done" means failure.
    if reader.ReadFile(str(path)) != IFSelect_RetDone:
        raise RuntimeError("STEP could not be read")
    # TransferRoots converts the file's contents into a 3D shape. 0 = nothing converted.
    if reader.TransferRoots() == 0:
        raise RuntimeError("No roots transferred")
    shape = reader.OneShape()                           # the finished 3D shape
    if shape.IsNull():                                  # empty shape = something went wrong
        raise RuntimeError("Null shape")
    return shape


# ---------------------------------------------------------------------
# PART 4: Helper function 2 - list the UNIQUE faces / edges / vertices
# ---------------------------------------------------------------------
def unique_shapes(shape, kind):
    m = TopTools_IndexedMapOfShape()      # an empty numbered list
    TopExp.MapShapes_s(shape, kind, m)    # fill it with each unique part of that kind
    # CAD numbering starts at 1 (not 0), so we loop from 1 to Extent().
    # Returns: the numbered list itself, AND a normal Python list of the parts.
    return m, [m.FindKey(i) for i in range(1, m.Extent() + 1)]


# ---------------------------------------------------------------------
# PART 5: Helper function 3 - make connections go BOTH ways
# ---------------------------------------------------------------------
# A graph connection between A and B is stored as two arrows: A->B and B->A.
# 'pairs' is a list like [(0,1), (1,2)], 'feats' holds one feature list per pair.
def both_ways(pairs, feats, n_feat):
    if not pairs:   # no connections at all: return empty arrays of the right shape
        return (torch.empty((2, 0), dtype=torch.long),
                torch.empty((0, n_feat), dtype=torch.float))
    idx, attr = [], []
    for (a, b), f in zip(pairs, feats):   # go through each pair with its features
        idx += [(a, b), (b, a)]           # store both directions
        attr += [f, f]                    # both directions share the same features
    # Graph libraries want connections as a 2-row table: row 0 = from, row 1 = to.
    return (torch.tensor(idx, dtype=torch.long).T.contiguous(),
            torch.tensor(np.asarray(attr), dtype=torch.float))


# ---------------------------------------------------------------------
# PART 6: The main function - build all 4 graphs from one shape
# ---------------------------------------------------------------------
def build_graphs(shape, volume):

    # ===== Step A: measure the model's overall size =====
    box = Bnd_Box()                        # an empty bounding box
    BRepBndLib.Add_s(shape, box)           # stretch it to fit the shape
    x0, y0, z0, x1, y1, z1 = box.Get()     # the box corners (min and max for x, y, z)
    # Diagonal = the longest line from one corner of the box to the opposite corner.
    diag = float(np.sqrt((x1-x0)**2 + (y1-y0)**2 + (z1-z0)**2))
    if not np.isfinite(diag) or diag <= 0:   # a broken box would ruin every division below
        raise RuntimeError("Bad bounding box")
    # Centre of the box. Subtracting it moves every model to the same origin.
    center = np.array([(x0+x1)/2, (y0+y1)/2, (z0+z1)/2])

    # ===== Step B: list every unique face, edge and vertex =====
    # face_map etc. are the numbered lists, faces etc. are the plain Python lists.
    face_map,   faces    = unique_shapes(shape, TopAbs_FACE)
    edge_map,   edges    = unique_shapes(shape, TopAbs_EDGE)
    vertex_map, vertices = unique_shapes(shape, TopAbs_VERTEX)

    # ===== Step C: 14 numbers for every FACE =====
    # Order of the 14 numbers:
    #  [0-5]  surface type flags: plane, cylinder, cone, sphere, torus, other
    #  [6]    area / diag^2          (size-free area)
    #  [7]    radius / diag          (size-free radius)
    #  [8-10] axis direction x,y,z   (for cylinders and cones)
    #  [11-13] centre position x,y,z (size-free, around the model's centre)
    face_feats = []
    for fs in faces:
        face = TopoDS.Face_s(fs)               # treat the part as a face
        props = GProp_GProps()                 # a container for measurements
        BRepGProp.SurfaceProperties_s(face, props)   # measure this face
        area = props.Mass()                    # for surfaces, "Mass" means area
        c = props.CentreOfMass()               # the face's middle point
        # Move it so the model centre is (0,0,0), then divide by diag to remove size.
        centroid = (np.array([c.X(), c.Y(), c.Z()]) - center) / diag

        ad = BRepAdaptor_Surface(face, True)   # lets us ask what kind of surface it is
        t = ad.GetType()                       # the surface kind
        # Each flag is 1.0 if the face is that kind, otherwise 0.0.
        flags = [float(t == GeomAbs_Plane), float(t == GeomAbs_Cylinder),
                 float(t == GeomAbs_Cone),  float(t == GeomAbs_Sphere),
                 float(t == GeomAbs_Torus)]
        # "Other" (curvy B-spline surfaces etc.) = 1 if none of the five above matched.
        flags.append(1.0 - sum(flags))

        radius, axis = 0.0, np.zeros(3)        # defaults: no radius, no axis
        if t == GeomAbs_Cylinder:
            cyl = ad.Cylinder()                # get the cylinder's details
            radius = cyl.Radius()
            d = cyl.Axis().Direction()         # which way the cylinder points
            axis = np.array([d.X(), d.Y(), d.Z()])
        elif t == GeomAbs_Cone:
            d = ad.Cone().Axis().Direction()   # which way the cone points
            axis = np.array([d.X(), d.Y(), d.Z()])
        elif t == GeomAbs_Sphere:
            radius = ad.Sphere().Radius()

        # Join everything into one list of 14 numbers.
        face_feats.append(flags + [area / diag**2, radius / diag]
                          + list(axis) + list(centroid))
    face_x = np.asarray(face_feats, dtype=np.float32)   # list -> table of numbers

    # ===== Step D: 4 numbers for every EDGE =====
    # [0] length / diag, [1] is it a straight line, [2] is it a circle, [3] other curve
    edge_feats = []
    for es in edges:
        edge = TopoDS.Edge_s(es)               # treat the part as an edge
        props = GProp_GProps()
        BRepGProp.LinearProperties_s(edge, props)    # measure its length
        t = BRepAdaptor_Curve(edge).GetType()        # what kind of curve is it?
        is_line, is_circle = float(t == GeomAbs_Line), float(t == GeomAbs_Circle)
        edge_feats.append([props.Mass() / diag,      # for edges, "Mass" means length
                           is_line, is_circle,
                           1.0 - is_line - is_circle])   # "other" curve
    edge_x = np.asarray(edge_feats, dtype=np.float32)

    # ===== Step E: 3 numbers for every VERTEX =====
    # Just its position (x, y, z), centred and divided by diag.
    vertex_feats = []
    for vs in vertices:
        p = BRep_Tool.Pnt_s(TopoDS.Vertex_s(vs))     # the corner's 3D point
        vertex_feats.append((np.array([p.X(), p.Y(), p.Z()]) - center) / diag)
    vertex_x = np.asarray(vertex_feats, dtype=np.float32)

    # ===== Step F: work out WHO TOUCHES WHOM =====
    n_f, n_e, n_v = len(faces), len(edges), len(vertices)   # how many of each
    # Four lookup tables, all empty to begin with:
    edge_to_faces    = {i: set() for i in range(n_e)}   # which faces use this edge
    edge_to_vertices = {i: set() for i in range(n_e)}   # which corners end this edge
    vertex_to_edges  = {i: set() for i in range(n_v)}   # which edges meet at this corner
    face_to_edges    = [set() for _ in range(n_f)]      # which edges border this face

    # Fill them: for each face, find its edges.
    for fi, fs in enumerate(faces):                     # fi = face number (starts at 0)
        local, _ = unique_shapes(fs, TopAbs_EDGE)       # edges of just this face
        for j in range(1, local.Extent() + 1):
            # FindIndex gives the edge's number in the whole model (starts at 1), so subtract 1.
            ei = edge_map.FindIndex(local.FindKey(j)) - 1
            if ei >= 0:                                 # -1 would mean "not found"
                face_to_edges[fi].add(ei)
                edge_to_faces[ei].add(fi)

    # Fill them: for each edge, find its end corners.
    for ei, es in enumerate(edges):
        local, _ = unique_shapes(es, TopAbs_VERTEX)
        for j in range(1, local.Extent() + 1):
            vi = vertex_map.FindIndex(local.FindKey(j)) - 1
            if vi >= 0:
                edge_to_vertices[ei].add(vi)
                vertex_to_edges[vi].add(ei)

    # ===== Step G: FACE GRAPH (nodes = faces) =====
    # Two faces are connected if they share an edge.
    # The connection carries that shared edge's 4 numbers.
    pairs, feats = [], []
    for ei, fset in edge_to_faces.items():
        for a, b in combinations(sorted(fset), 2):      # every pair of faces on this edge
            pairs.append((a, b)); feats.append(edge_x[ei])
    f_index, f_attr = both_ways(pairs, feats, N_EDGE_FEATURES)

    # ===== Step H: EDGE GRAPH (nodes = edges) =====
    # Two edges are connected if they meet at a corner.
    # The connection carries that corner's 3 numbers.
    pairs, feats = [], []
    for vi in range(n_v):
        for a, b in combinations(sorted(vertex_to_edges[vi]), 2):
            pairs.append((a, b)); feats.append(vertex_x[vi])
    e_index, e_attr = both_ways(pairs, feats, N_VERTEX_FEATURES)

    # ===== Step I: VERTEX GRAPH (nodes = corners) =====
    # Each edge becomes a connection between its two end corners.
    # A closed circle edge has only 1 corner, so it is skipped (needs exactly 2).
    pairs, feats = [], []
    for ei, vset in edge_to_vertices.items():
        ends = sorted(vset)
        if len(ends) == 2:
            pairs.append((ends[0], ends[1])); feats.append(edge_x[ei])
    v_index, v_attr = both_ways(pairs, feats, N_EDGE_FEATURES)

    # ===== Step J: the target and extra labels =====
    # TARGET y = volume / diag^3. This number does not change if the part is scaled.
    y = torch.tensor([volume / diag**3], dtype=torch.float)
    # We also store the real volume and diag, but only as labels.
    # They are NOT inputs to the network; they let us convert predictions back later.
    labels = dict(y=y,
                  volume=torch.tensor([volume], dtype=torch.float),
                  diag=torch.tensor([diag], dtype=torch.float))

    # Pack each graph: x = node numbers, edge_index = who connects to whom,
    # edge_attr = numbers on each connection.
    face_g   = Data(x=torch.from_numpy(face_x),   edge_index=f_index, edge_attr=f_attr, **labels)
    edge_g   = Data(x=torch.from_numpy(edge_x),   edge_index=e_index, edge_attr=e_attr, **labels)
    vertex_g = Data(x=torch.from_numpy(vertex_x), edge_index=v_index, edge_attr=v_attr, **labels)

    # ===== Step K: HETERO GRAPH (all three node kinds together) =====
    # fe = list of (face, edge) pairs where the edge borders the face
    fe = [(f, e) for f, es in enumerate(face_to_edges) for e in sorted(es)]
    # ev = list of (edge, vertex) pairs where the corner ends the edge
    ev = [(e, v) for e, vs in edge_to_vertices.items() for v in sorted(vs)]
    # Small helper: turn a list of pairs into the 2-row table graph libraries want.
    to_idx = lambda p: (torch.tensor(p, dtype=torch.long).T.contiguous()
                        if p else torch.empty((2, 0), dtype=torch.long))
    fe_idx, ev_idx = to_idx(fe), to_idx(ev)

    hg = HeteroData()
    hg["face"].x   = torch.from_numpy(face_x)     # the three kinds of nodes
    hg["edge"].x   = torch.from_numpy(edge_x)
    hg["vertex"].x = torch.from_numpy(vertex_x)
    # Four kinds of connections: each one forwards, and each one backwards (.flip(0)
    # swaps the "from" row and the "to" row).
    hg["face", "boundary", "edge"].edge_index       = fe_idx
    hg["edge", "belongs_to", "face"].edge_index     = fe_idx.flip(0)
    hg["edge", "endpoint", "vertex"].edge_index     = ev_idx
    hg["vertex", "connected_by", "edge"].edge_index = ev_idx.flip(0)
    hg.y, hg.volume, hg.diag = y, labels["volume"], labels["diag"]

    # Return all four graphs plus the counts (used by the test below).
    return {"face": face_g, "edge": edge_g, "vertex": vertex_g, "hetero": hg,
            "counts": (n_f, n_e, n_v)}


# =====================================================================
# PART 7: TEST ON ONE MODEL
# =====================================================================
dataset = pd.read_csv(PROJECT_DIR / "dataset_100.csv")   # the 100 models from Chunk 4
row = dataset.iloc[0]                                    # take the first model
print("Test model:", row["filename"])

shape = load_shape(RAW_STEP_DIR / row["filename"])       # read it from Drive
out = build_graphs(shape, float(row["volume"]))          # build the 4 graphs

# ----- The scaling test -----
# Make a copy of the same model that is 2x bigger in every direction.
trsf = gp_Trsf()
trsf.SetScale(gp_Pnt(0, 0, 0), 2.0)                      # "scale by 2 around the origin"
big = BRepBuilderAPI_Transform(shape, trsf, True).Shape()
# Doubling every side makes the volume 2 x 2 x 2 = 8 times larger.
out2 = build_graphs(big, float(row["volume"]) * 8)

fg, eg, vg, hg = out["face"], out["edge"], out["vertex"], out["hetero"]

# 'same' checks that two tables hold the same numbers (small rounding allowed).
# We sort each column first, so the order of the nodes doesn't matter.
same = lambda a, b: np.allclose(np.sort(a.numpy(), axis=0),
                                np.sort(b.numpy(), axis=0), atol=1e-4)

# ----- The six checks -----
checks = {
    # 1. Our node counts agree with the counts saved earlier in the CSV.
    "counts match the CSV":   out["counts"] == (int(row["faces"]), int(row["edges"]), int(row["vertices"])),
    # 2. Each kind of node has the expected number of features.
    "feature sizes 14/4/3":   (fg.x.shape[1], eg.x.shape[1], vg.x.shape[1]) == (14, 4, 3),
    # 3. No NaN or infinity anywhere in the node numbers.
    "all numbers finite":     all(torch.isfinite(g.x).all() for g in (fg, eg, vg)),
    # 4. Every connection points to a node that exists.
    "edge indices in range":  all(g.edge_index.numel() == 0 or int(g.edge_index.max()) < g.x.shape[0] for g in (fg, eg, vg)),
    # 5. The 2x bigger model produces the SAME input numbers (the scale fix works).
    "2x model: same inputs":  same(fg.x, out2["face"].x) and same(eg.x, out2["edge"].x) and same(vg.x, out2["vertex"].x),
    # 6. The 2x bigger model produces the SAME target.
    "2x model: same target":  torch.allclose(fg.y, out2["face"].y, rtol=1e-4),
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "|", name)

# Finally, show the sizes so you can see what was built.
print("\nFaces / edges / vertices:", out["counts"])
print("Face graph   :", fg.x.shape, fg.edge_index.shape)    # (nodes, features), (2, connections)
print("Edge graph   :", eg.x.shape, eg.edge_index.shape)
print("Vertex graph :", vg.x.shape, vg.edge_index.shape)
print("Target (volume / diag^3):", float(fg.y))

Test model: 00000000_290a9120f9f249a7a05cfe9c_step_000.step
PASS | counts match the CSV
PASS | feature sizes 14/4/3
PASS | all numbers finite
PASS | edge indices in range
PASS | 2x model: same inputs
PASS | 2x model: same target

Faces / edges / vertices: (25, 58, 39)
Face graph   : torch.Size([25, 14]) torch.Size([2, 80])
Edge graph   : torch.Size([58, 4]) torch.Size([2, 186])
Vertex graph : torch.Size([39, 3]) torch.Size([2, 94])
Target (volume / diag^3): 0.010996286757290363


In [9]:
#8
# =====================================================================
# CHUNK 5B: Build graphs for ALL 100 models and save them to Drive
# =====================================================================
# Needs the 5A cell to have been run in this session
# (it defines load_shape and build_graphs).
# =====================================================================

import time
import numpy as np
import pandas as pd
import torch
from pathlib import Path

PROJECT_DIR  = Path("/content/drive/MyDrive/CAD_GNN_Project")
RAW_STEP_DIR = PROJECT_DIR / "raw_step"
GRAPH_DIR    = PROJECT_DIR / "graphs"
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

# A friendly stop if 5A was not run in this session
if "build_graphs" not in globals() or "load_shape" not in globals():
    raise RuntimeError("Run the Chunk 5A cell first, then this one.")

PARTIAL_FILE = GRAPH_DIR / "graphs_partial.pt"   # progress saved while running
SAVE_EVERY   = 20                                 # save progress every 20 models

dataset = pd.read_csv(PROJECT_DIR / "dataset_100.csv")
assert len(dataset) == 100, "dataset_100.csv should have 100 rows"

# ---------------------------------------------------------------------
# Resume: load whatever was finished in an earlier run
# ---------------------------------------------------------------------
if PARTIAL_FILE.exists():
    done = torch.load(PARTIAL_FILE, map_location="cpu", weights_only=False)
    print("Resuming. Already finished:", len(done))
else:
    done = {}          # {filename: {"face":..., "edge":..., "vertex":..., "hetero":...}}

failed = []            # (filename, reason) for any model that goes wrong


# ---------------------------------------------------------------------
# Checks run on EVERY model (returns a list of problems, empty = all good)
# ---------------------------------------------------------------------
def check_model(row, out):
    problems = []
    fg, eg, vg, hg = out["face"], out["edge"], out["vertex"], out["hetero"]

    # 1. Node counts agree with the counts saved in the CSV
    expected = (int(row["faces"]), int(row["edges"]), int(row["vertices"]))
    if out["counts"] != expected:
        problems.append(f"counts {out['counts']} != CSV {expected}")

    # 2. Feature sizes are 14 / 4 / 3
    if (fg.x.shape[1], eg.x.shape[1], vg.x.shape[1]) != (14, 4, 3):
        problems.append("wrong feature size")

    # 3. No NaN or infinity in any node or connection number
    for name, g in (("face", fg), ("edge", eg), ("vertex", vg)):
        if not torch.isfinite(g.x).all():
            problems.append(f"{name}: non-finite node features")
        if not torch.isfinite(g.edge_attr).all():
            problems.append(f"{name}: non-finite connection features")
        # 4. Every connection points at a node that exists
        if g.edge_index.numel() > 0:
            if int(g.edge_index.min()) < 0 or int(g.edge_index.max()) >= g.x.shape[0]:
                problems.append(f"{name}: connection index out of range")

    # 5. Hetero graph has the same node counts as the three simple graphs
    if (hg["face"].x.shape[0], hg["edge"].x.shape[0], hg["vertex"].x.shape[0]) != out["counts"]:
        problems.append("hetero node counts differ")

    # 6. The stored volume equals the CSV volume, and y = volume / diag^3
    vol_csv = float(row["volume"])
    if not np.isclose(float(fg.volume), vol_csv, rtol=1e-5):
        problems.append("stored volume differs from CSV")
    if not np.isclose(float(fg.y), vol_csv / float(fg.diag) ** 3, rtol=1e-4):
        problems.append("target is not volume / diag^3")

    # 7. All four graphs carry the same target
    for g in (eg, vg, hg):
        if not torch.allclose(fg.y, g.y):
            problems.append("targets differ between graph types")
            break

    return problems


# ---------------------------------------------------------------------
# Main loop over the 100 models
# ---------------------------------------------------------------------
start = time.time()

for i, row in dataset.iterrows():
    name = row["filename"]

    if name in done:                       # already built in an earlier run
        continue

    try:
        shape = load_shape(RAW_STEP_DIR / name)
        out = build_graphs(shape, float(row["volume"]))

        problems = check_model(row, out)
        if problems:
            raise RuntimeError("; ".join(problems))

        # keep the four graphs (drop the 'counts' helper)
        done[name] = {k: out[k] for k in ("face", "edge", "vertex", "hetero")}

    except Exception as e:
        failed.append((name, str(e)))
        print(f"FAILED {i+1}/100 | {name} | {e}")

    # progress message and checkpoint
    count = len(done) + len(failed)
    if count % SAVE_EVERY == 0:
        torch.save(done, PARTIAL_FILE)
        print(f"{count}/100 processed | {len(done)} good | "
              f"{time.time()-start:.0f}s | progress saved")

torch.save(done, PARTIAL_FILE)             # final checkpoint


# ---------------------------------------------------------------------
# Stop here if anything failed, so we never save an incomplete set
# ---------------------------------------------------------------------
print("\nGood models:", len(done), "| Failed:", len(failed))
if failed:
    for name, why in failed:
        print(" -", name, "|", why)
    raise RuntimeError("Some models failed. Nothing final was saved. Paste the list above.")


# ---------------------------------------------------------------------
# Write the four final files and re-load them to confirm
# ---------------------------------------------------------------------
master = set(dataset["filename"])
for kind in ("face", "edge", "vertex", "hetero"):
    graphs = {name: done[name][kind] for name in dataset["filename"]}
    path = GRAPH_DIR / f"{kind}_graphs_100.pt"
    torch.save(graphs, path)

    back = torch.load(path, map_location="cpu", weights_only=False)
    assert set(back.keys()) == master and len(back) == 100, f"{kind}: reload check failed"
    print(f"SAVED + RELOAD OK | {kind}: {len(back)} graphs | {path.stat().st_size/1e6:.1f} MB")


# ---------------------------------------------------------------------
# A short summary so you can see what was built
# ---------------------------------------------------------------------
rows = []
for name in dataset["filename"]:
    g = done[name]
    rows.append({
        "face_nodes":   g["face"].x.shape[0],
        "face_links":   g["face"].edge_index.shape[1],
        "edge_nodes":   g["edge"].x.shape[0],
        "edge_links":   g["edge"].edge_index.shape[1],
        "vertex_nodes": g["vertex"].x.shape[0],
        "vertex_links": g["vertex"].edge_index.shape[1],
        "target":       float(g["face"].y),
    })
summary = pd.DataFrame(rows)

print("\nGraph sizes across the 100 models:")
print(summary.describe().loc[["min", "50%", "max"]].round(4).T)
print("\nModels with an empty face graph (no connections):", int((summary["face_links"] == 0).sum()))
print("Models with an empty edge graph:", int((summary["edge_links"] == 0).sum()))
print("Models with an empty vertex graph:", int((summary["vertex_links"] == 0).sum()))
print("\nALL 100 MODELS BUILT, CHECKED AND SAVED.")

Resuming. Already finished: 100

Good models: 100 | Failed: 0
SAVED + RELOAD OK | face: 100 graphs | 6.1 MB
SAVED + RELOAD OK | edge: 100 graphs | 9.5 MB
SAVED + RELOAD OK | vertex: 100 graphs | 4.9 MB
SAVED + RELOAD OK | hetero: 100 graphs | 12.7 MB

Graph sizes across the 100 models:
                 min        50%         max
face_nodes    3.0000   102.0000   3124.0000
face_links    0.0000   458.0000  15018.0000
edge_nodes    3.0000   255.5000  10008.0000
edge_links    4.0000  1001.0000  39680.0000
vertex_nodes  2.0000   160.0000   7505.0000
vertex_links  2.0000   440.0000  15018.0000
target        0.0001     0.0106      0.1925

Models with an empty face graph (no connections): 1
Models with an empty edge graph: 0
Models with an empty vertex graph: 0

ALL 100 MODELS BUILT, CHECKED AND SAVED.


In [10]:
#9
# =====================================================================
# CHUNK 5C: Independent verification of the 100-model dataset
# READ-ONLY: nothing already saved is modified.
# Writes: verification_per_model_100.csv, verification_summary_100.csv,
#         environment_and_fingerprints.json
# =====================================================================

import json, hashlib, sys, platform
import importlib.metadata as md
from collections import Counter
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch_geometric                      # needed so saved graphs can be loaded
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

from OCP.STEPControl import STEPControl_Reader
from OCP.IFSelect import IFSelect_RetDone
from OCP.Interface import Interface_Static
from OCP.TopAbs import (TopAbs_FACE, TopAbs_EDGE, TopAbs_VERTEX,
                        TopAbs_SOLID, TopAbs_REVERSED)
from OCP.TopExp import TopExp, TopExp_Explorer
from OCP.TopTools import (TopTools_IndexedMapOfShape,
                          TopTools_IndexedDataMapOfShapeListOfShape)
from OCP.TopoDS import TopoDS
from OCP.BRep import BRep_Tool
from OCP.BRepGProp import BRepGProp
from OCP.GProp import GProp_GProps
from OCP.Bnd import Bnd_Box
from OCP.BRepBndLib import BRepBndLib


# ---------------------------------------------------------------------
# PART 1: Settings
# ---------------------------------------------------------------------
PROJECT_DIR  = Path("/content/drive/MyDrive/CAD_GNN_Project")
RAW_STEP_DIR = PROJECT_DIR / "raw_step"
GRAPH_DIR    = PROJECT_DIR / "graphs"

CUBE_MAX        = 1 / (3 * np.sqrt(3))   # largest possible volume / diag^3 (a cube)
DO_MESH         = True                   # set False to skip the mesh volume check
MESH_MAX_FACES  = 600                    # skip the mesh check for models bigger than this
MESH_DEFLECTION = 1e-3                   # mesh fineness, as a fraction of the diagonal

KINDS = ("face", "edge", "vertex", "hetero")


# ---------------------------------------------------------------------
# PART 2: Helper functions
# ---------------------------------------------------------------------
def load_shape(path):
    # Same safe reader as before: create reader first, then set millimetres.
    reader = STEPControl_Reader()
    Interface_Static.SetCVal_s("xstep.cascade.unit", "MM")
    if reader.ReadFile(str(path)) != IFSelect_RetDone:
        raise RuntimeError("STEP could not be read")
    if reader.TransferRoots() == 0:
        raise RuntimeError("No roots transferred")
    shape = reader.OneShape()
    if shape.IsNull():
        raise RuntimeError("Null shape")
    return shape


def unique(shape, kind):
    # Numbered list of unique faces / edges / vertices / solids,
    # plus the same parts as a normal Python list.
    m = TopTools_IndexedMapOfShape()
    TopExp.MapShapes_s(shape, kind, m)
    return m, [m.FindKey(i) for i in range(1, m.Extent() + 1)]


def bbox_diag(shape, tight):
    # Diagonal of the bounding box. tight=True asks OCC for the tightest box.
    box = Bnd_Box()
    if tight:
        BRepBndLib.AddOptimal_s(shape, box, False, False)
    else:
        BRepBndLib.Add_s(shape, box)
    x0, y0, z0, x1, y1, z1 = box.Get()
    return float(np.sqrt((x1-x0)**2 + (y1-y0)**2 + (z1-z0)**2))


def mesh_volume(shape, faces, diag):
    # Second, independent way to get volume: cut every face into triangles
    # and add up the signed volume of the tetrahedra they form with the origin.
    from OCP.BRepMesh import BRepMesh_IncrementalMesh
    from OCP.TopLoc import TopLoc_Location
    BRepMesh_IncrementalMesh(shape, MESH_DEFLECTION * diag, False, 0.3, True)
    total = 0.0
    for f in faces:
        face = TopoDS.Face_s(f)
        loc = TopLoc_Location()
        tri = BRep_Tool.Triangulation_s(face, loc)
        if tri is None:
            continue
        trsf = loc.Transformation()
        flip = (face.Orientation() == TopAbs_REVERSED)
        for i in range(1, tri.NbTriangles() + 1):
            n1, n2, n3 = tri.Triangle(i).Get()
            pts = [tri.Node(n).Transformed(trsf) for n in (n1, n2, n3)]
            if flip:
                pts[1], pts[2] = pts[2], pts[1]
            a, b, c = [np.array([p.X(), p.Y(), p.Z()]) for p in pts]
            total += float(np.dot(a, np.cross(b, c))) / 6.0
    return abs(total)


def pair_counter(edge_index):
    # Turn a 2-row connection table into a Counter of (from, to) pairs.
    return Counter(map(tuple, edge_index.T.tolist()))


def mismatch(expected, actual):
    # Number of pairs that are missing plus pairs that should not be there.
    return sum((expected - actual).values()) + sum((actual - expected).values())


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


# ---------------------------------------------------------------------
# PART 3: Verify ONE model against its CAD file
# ---------------------------------------------------------------------
def verify_model(name, volume_csv, g):
    shape = load_shape(RAW_STEP_DIR / name)
    rec = {"filename": name}

    face_map,   faces    = unique(shape, TopAbs_FACE)
    edge_map,   edges    = unique(shape, TopAbs_EDGE)
    vertex_map, vertices = unique(shape, TopAbs_VERTEX)
    n_e = len(edges)
    n_v = len(vertices)

    # ---- A. Rebuild the connections with a DIFFERENT method ----------
    # The extractor looked inside each face and each edge.
    # Here we ask OCC directly: "which faces use this edge?" and
    # "which edges meet at this vertex?" (ancestor maps).
    ef = TopTools_IndexedDataMapOfShapeListOfShape()
    TopExp.MapShapesAndAncestors_s(shape, TopAbs_EDGE, TopAbs_FACE, ef)
    ve = TopTools_IndexedDataMapOfShapeListOfShape()
    TopExp.MapShapesAndAncestors_s(shape, TopAbs_VERTEX, TopAbs_EDGE, ve)

    edge_faces = {i: set() for i in range(n_e)}
    for k in range(1, ef.Extent() + 1):
        ei = edge_map.FindIndex(ef.FindKey(k)) - 1
        for f in ef.FindFromIndex(k):
            fi = face_map.FindIndex(f) - 1
            if ei >= 0 and fi >= 0:
                edge_faces[ei].add(fi)

    edge_verts = {i: set() for i in range(n_e)}
    vert_edges = {i: set() for i in range(n_v)}
    for k in range(1, ve.Extent() + 1):
        vi = vertex_map.FindIndex(ve.FindKey(k)) - 1
        for e in ve.FindFromIndex(k):
            ei = edge_map.FindIndex(e) - 1
            if vi >= 0 and ei >= 0:
                edge_verts[ei].add(vi)
                vert_edges[vi].add(ei)

    exp_face, exp_edge, exp_vertex = Counter(), Counter(), Counter()
    for ei, fs in edge_faces.items():                 # face graph
        for a, b in combinations(sorted(fs), 2):
            exp_face[(a, b)] += 1; exp_face[(b, a)] += 1
    for vi, es in vert_edges.items():                 # edge graph
        for a, b in combinations(sorted(es), 2):
            exp_edge[(a, b)] += 1; exp_edge[(b, a)] += 1
    for ei, vs in edge_verts.items():                 # vertex graph
        vs = sorted(vs)
        if len(vs) == 2:
            exp_vertex[(vs[0], vs[1])] += 1; exp_vertex[(vs[1], vs[0])] += 1
    exp_fe = Counter((f, e) for e, fs in edge_faces.items() for f in fs)
    exp_ev = Counter((e, v) for e, vs in edge_verts.items() for v in vs)

    hg = g["hetero"]
    rec["bad_face_links"]   = mismatch(exp_face,   pair_counter(g["face"].edge_index))
    rec["bad_edge_links"]   = mismatch(exp_edge,   pair_counter(g["edge"].edge_index))
    rec["bad_vertex_links"] = mismatch(exp_vertex, pair_counter(g["vertex"].edge_index))
    rec["bad_face_edge"]    = mismatch(exp_fe, pair_counter(hg["face", "boundary", "edge"].edge_index))
    rec["bad_edge_vertex"]  = mismatch(exp_ev, pair_counter(hg["edge", "endpoint", "vertex"].edge_index))

    # ---- B. Is the shape a closed, manifold solid? ----------------------
    # In a closed solid every edge is used exactly twice by faces
    # (a seam edge is used twice by the SAME face). Count the uses.
    uses = Counter()
    for f in faces:
        ex = TopExp_Explorer(f, TopAbs_EDGE)
        while ex.More():
            uses[edge_map.FindIndex(ex.Current()) - 1] += 1
            ex.Next()
    free = nonmanifold = 0
    for ei, e in enumerate(edges):
        try:
            if BRep_Tool.Degenerated_s(TopoDS.Edge_s(e)):   # zero-length edges at poles
                continue
        except Exception:
            pass
        if uses[ei] < 2:
            free += 1            # open boundary
        elif uses[ei] > 2:
            nonmanifold += 1     # three or more faces on one edge
    rec["free_edges"] = free
    rec["nonmanifold_edges"] = nonmanifold

    # ---- C. Volume label integrity -------------------------------------
    props = GProp_GProps()
    BRepGProp.VolumeProperties_s(shape, props)
    signed_total = float(props.Mass())
    solid_map, solids = unique(shape, TopAbs_SOLID)
    sum_abs = 0.0
    for s in solids:
        p = GProp_GProps()
        BRepGProp.VolumeProperties_s(TopoDS.Solid_s(s), p)
        sum_abs += abs(float(p.Mass()))

    rec["n_solids"] = len(solids)
    rec["signed_volume_negative"] = signed_total < 0
    rec["vol_rel_diff_vs_csv"] = abs(abs(signed_total) - volume_csv) / max(volume_csv, 1e-30)
    rec["vol_rel_diff_vs_solid_sum"] = abs(abs(signed_total) - sum_abs) / max(abs(signed_total), 1e-30)

    # ---- D. Is the bounding box tight? ----------------------------------
    diag_loose = bbox_diag(shape, tight=False)
    try:
        diag_tight = bbox_diag(shape, tight=True)
        rec["bbox_inflation"] = diag_loose / diag_tight - 1.0
    except Exception:
        rec["bbox_inflation"] = np.nan

    # ---- E. Mesh-based volume (optional, skipped for big models) ----------
    rec["mesh_rel_diff"] = np.nan
    if DO_MESH and len(faces) <= MESH_MAX_FACES:
        try:
            mv = mesh_volume(shape, faces, diag_loose)
            rec["mesh_rel_diff"] = abs(mv - volume_csv) / max(volume_csv, 1e-30)
        except Exception:
            pass
    return rec


# ---------------------------------------------------------------------
# PART 4: Load the dataset and the saved graphs
# ---------------------------------------------------------------------
dataset = pd.read_csv(PROJECT_DIR / "dataset_100.csv")
names = dataset["filename"].tolist()

graphs = {}
for kind in KINDS:
    graphs[kind] = torch.load(GRAPH_DIR / f"{kind}_graphs_100.pt",
                              map_location="cpu", weights_only=False)

summary = []   # rows of (check, status, detail)
def add(check, status, detail=""):
    summary.append({"check": check, "status": status, "detail": detail})
    print(f"{status:5s} | {check} | {detail}")

def names_where(mask, limit=5):
    bad = per_model.loc[mask, "filename"].tolist()
    short = [b[:14] for b in bad[:limit]]
    return f"{len(bad)} model(s): {short}" if bad else "none"


# ---------------------------------------------------------------------
# PART 5: Run the per-model verification
# ---------------------------------------------------------------------
rows = []
for i, row in dataset.iterrows():
    name = row["filename"]
    try:
        g = {k: graphs[k][name] for k in KINDS}
        rows.append(verify_model(name, float(row["volume"]), g))
    except Exception as e:
        rows.append({"filename": name, "error": repr(e)})
    if (i + 1) % 20 == 0:
        print(f"{i+1}/100 verified")

per_model = pd.DataFrame(rows)
per_model.to_csv(PROJECT_DIR / "verification_per_model_100.csv", index=False)

print("\n================ RESULTS ================")

# ---------------------------------------------------------------------
# PART 6: Judge the results
# ---------------------------------------------------------------------
# 0. Did every model run?
if "error" in per_model.columns and per_model["error"].notna().any():
    add("Every model verified", "FAIL", names_where(per_model["error"].notna()))
else:
    add("Every model verified", "PASS", "100/100")

link_cols = ["bad_face_links", "bad_edge_links", "bad_vertex_links",
             "bad_face_edge", "bad_edge_vertex"]
total_bad = int(per_model[link_cols].fillna(0).sum().sum())
add("Graph connections match independent rebuild",
    "PASS" if total_bad == 0 else "FAIL",
    f"{total_bad} mismatching connections")

mask = (per_model["free_edges"].fillna(0) > 0) | (per_model["nonmanifold_edges"].fillna(0) > 0)
add("Closed manifold solid (no free or non-manifold edges)",
    "PASS" if not mask.any() else "WARN", names_where(mask))

mask = per_model["vol_rel_diff_vs_csv"].fillna(0) > 1e-6
add("Volume recomputed equals CSV volume",
    "PASS" if not mask.any() else "FAIL", names_where(mask))

mask = per_model["vol_rel_diff_vs_solid_sum"].fillna(0) > 1e-4
add("Volume equals sum of per-solid volumes",
    "PASS" if not mask.any() else "WARN", names_where(mask))

mask = per_model["signed_volume_negative"].fillna(False).astype(bool)
add("No inverted (negative-volume) solids",
    "PASS" if not mask.any() else "WARN", names_where(mask))

if per_model["mesh_rel_diff"].notna().any():
    mask = per_model["mesh_rel_diff"] > 0.05
    done = int(per_model["mesh_rel_diff"].notna().sum())
    add("Mesh volume agrees within 5%",
        "PASS" if not mask.any() else "WARN",
        f"{done} models checked; " + names_where(mask))
else:
    add("Mesh volume agrees within 5%", "SKIP", "not run or not supported here")

if per_model["bbox_inflation"].notna().any():
    mask = per_model["bbox_inflation"] > 0.01
    add("Bounding box tight (under 1% inflation)",
        "PASS" if not mask.any() else "WARN",
        f"max inflation {per_model['bbox_inflation'].max()*100:.2f}%; " + names_where(mask))
else:
    add("Bounding box tight (under 1% inflation)", "SKIP", "tight box not available")

# Checks that use the saved graphs directly
targets = pd.Series({n: float(graphs["face"][n].y) for n in names})
bad_target = targets[(targets <= 0) | (targets > CUBE_MAX * 1.001)]
add("Target within (0, cube maximum 0.19245]",
    "PASS" if bad_target.empty else "FAIL",
    f"min {targets.min():.5f}, max {targets.max():.5f}" +
    ("" if bad_target.empty else f"; outside: {[n[:14] for n in bad_target.index[:5]]}"))

keys_ok = all(set(graphs[k].keys()) == set(names) and len(graphs[k]) == 100 for k in KINDS)
add("All four graph files hold the same 100 models",
    "PASS" if keys_ok else "FAIL", "")

def n_components(gr):
    n = gr.x.shape[0]
    ei = gr.edge_index.numpy()
    if ei.shape[1] == 0:
        return n
    a = coo_matrix((np.ones(ei.shape[1]), (ei[0], ei[1])), shape=(n, n))
    return connected_components(a, directed=False)[0]

def isolated(gr):
    n = gr.x.shape[0]
    deg = np.zeros(n, dtype=int)
    for a in gr.edge_index[0].tolist():
        deg[a] += 1
    return int((deg == 0).sum())

empty_face = [n for n in names if graphs["face"][n].edge_index.shape[1] == 0]
add("No empty face graph",
    "PASS" if not empty_face else "WARN",
    "none" if not empty_face else
    f"{[(n[:14], graphs['face'][n].x.shape[0]) for n in empty_face]} (name, face count)")

iso = [n for n in names if isolated(graphs["face"][n]) > 0]
add("No isolated faces (every face has a neighbour)",
    "PASS" if not iso else "WARN", f"{len(iso)} model(s) have isolated faces")

multi = [n for n in names if n_components(graphs["face"][n]) > 1]
add("Face graph is one connected piece",
    "PASS" if not multi else "INFO",
    f"{len(multi)} model(s) have more than one piece (multi-body or unsewn)")

# Content duplicates: same counts, same target, same sorted face areas
by_counts = {}
for n in names:
    key = (graphs["face"][n].x.shape[0], graphs["edge"][n].x.shape[0],
           graphs["vertex"][n].x.shape[0])
    by_counts.setdefault(key, []).append(n)
dups = []
for group in by_counts.values():
    for a, b in combinations(group, 2):
        ya, yb = float(graphs["face"][a].y), float(graphs["face"][b].y)
        xa = np.sort(graphs["face"][a].x[:, 6].numpy())
        xb = np.sort(graphs["face"][b].x[:, 6].numpy())
        if abs(ya - yb) <= 1e-4 * max(ya, yb) and np.allclose(xa, xb, rtol=1e-3, atol=1e-6):
            dups.append((a[:14], b[:14]))
add("No same-shape duplicates (leakage risk)",
    "PASS" if not dups else "WARN", f"{len(dups)} pair(s): {dups[:5]}")


# ---------------------------------------------------------------------
# PART 7: Reproducibility record
# ---------------------------------------------------------------------
def version(pkg):
    try:
        return md.version(pkg)
    except Exception:
        return "unknown"

env = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torch_geometric": torch_geometric.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "cadquery": version("cadquery"),
    "cadquery-ocp": version("cadquery-ocp"),
    "settings": {"unit": "MM", "split_seed": 42, "n_models": 100},
    "fingerprints_sha256": {
        "dataset_100.csv": sha256(PROJECT_DIR / "dataset_100.csv"),
        "geometry_all_140.csv": sha256(PROJECT_DIR / "geometry_all_140.csv"),
        **{f"{k}_graphs_100.pt": sha256(GRAPH_DIR / f"{k}_graphs_100.pt") for k in KINDS},
    },
}
with open(PROJECT_DIR / "environment_and_fingerprints.json", "w") as f:
    json.dump(env, f, indent=2)

pd.DataFrame(summary).to_csv(PROJECT_DIR / "verification_summary_100.csv", index=False)

counts = Counter(r["status"] for r in summary)
print("\n=========================================")
print("PASS:", counts["PASS"], "| WARN:", counts["WARN"], "| FAIL:", counts["FAIL"],
      "| INFO:", counts["INFO"], "| SKIP:", counts["SKIP"])
print("Saved: verification_per_model_100.csv, verification_summary_100.csv,")
print("       environment_and_fingerprints.json")

20/100 verified
40/100 verified
60/100 verified
80/100 verified
100/100 verified

================ RESULTS ================
PASS  | Every model verified | 100/100
PASS  | Graph connections match independent rebuild | 0 mismatching connections
WARN  | Closed manifold solid (no free or non-manifold edges) | 3 model(s): ['00000036_b169a', '00000081_201da', '00000341_fe70b']
PASS  | Volume recomputed equals CSV volume | none
WARN  | Volume equals sum of per-solid volumes | 4 model(s): ['00000036_b169a', '00000081_201da', '00000084_9a051', '00000341_fe70b']
PASS  | No inverted (negative-volume) solids | none
WARN  | Mesh volume agrees within 5% | 88 models checked; 2 model(s): ['00000036_b169a', '00000286_09d7e']
WARN  | Bounding box tight (under 1% inflation) | max inflation 239.44%; 29 model(s): ['00000000_290a9', '00000001_1ffb8', '00000010_b4b99', '00000011_e909f', '00000012_f1688']
PASS  | Target within (0, cube maximum 0.19245] | min 0.00013, max 0.19245
PASS  | All four graph files h

In [11]:
#10

from pathlib import Path
import pandas as pd
import torch
import json
import hashlib

PROJECT_DIR = Path("/content/drive/MyDrive/CAD_GNN_Project")
RAW_STEP_DIR = PROJECT_DIR / "raw_step"
GRAPH_DIR = PROJECT_DIR / "graphs"

print("=" * 65)
print("CAD-GNN PROJECT PERSISTENCE CHECK")
print("=" * 65)

# 1. Required files
required_files = [
    PROJECT_DIR / "geometry_all_140.csv",
    PROJECT_DIR / "dataset_100.csv",
    PROJECT_DIR / "download_manifest.csv",
    PROJECT_DIR / "download_manifest_batch2.csv",
    PROJECT_DIR / "download_groups.csv",
    GRAPH_DIR / "face_graphs_100.pt",
    GRAPH_DIR / "edge_graphs_100.pt",
    GRAPH_DIR / "vertex_graphs_100.pt",
    GRAPH_DIR / "hetero_graphs_100.pt",
    PROJECT_DIR / "verification_per_model_100.csv",
    PROJECT_DIR / "verification_summary_100.csv",
    PROJECT_DIR / "environment_and_fingerprints.json",
]

print("\n--- FILE EXISTENCE AND READ ACCESS ---")
for path in required_files:
    if path.exists():
        try:
            size_mb = path.stat().st_size / (1024**2)
            with open(path, "rb") as f:
                f.read(1)
            print(f"PASS | {path.name} | {size_mb:.2f} MB | readable")
        except Exception as e:
            print(f"FAIL | {path.name} | read error: {e}")
    else:
        print(f"MISSING | {path.name}")

# 2. CSV readability and row counts
print("\n--- CSV CONTENT CHECK ---")
csv_files = [
    PROJECT_DIR / "geometry_all_140.csv",
    PROJECT_DIR / "dataset_100.csv",
    PROJECT_DIR / "download_manifest.csv",
    PROJECT_DIR / "download_manifest_batch2.csv",
    PROJECT_DIR / "download_groups.csv",
    PROJECT_DIR / "verification_per_model_100.csv",
    PROJECT_DIR / "verification_summary_100.csv",
]

for path in csv_files:
    if path.exists():
        try:
            df = pd.read_csv(path)
            print(f"PASS | {path.name} | rows={len(df)} | columns={len(df.columns)}")
        except Exception as e:
            print(f"FAIL | {path.name} | {e}")

# 3. Graph files: load and inspect keys/counts
print("\n--- GRAPH DATASET LOAD CHECK ---")
graph_files = {
    "face": GRAPH_DIR / "face_graphs_100.pt",
    "edge": GRAPH_DIR / "edge_graphs_100.pt",
    "vertex": GRAPH_DIR / "vertex_graphs_100.pt",
    "hetero": GRAPH_DIR / "hetero_graphs_100.pt",
}

loaded_graphs = {}

for name, path in graph_files.items():
    if not path.exists():
        print(f"MISSING | {name}")
        continue

    try:
        data = torch.load(path, map_location="cpu", weights_only=False)
        loaded_graphs[name] = data
        keys = set(data.keys()) if isinstance(data, dict) else set()
        print(f"PASS | {name} | {len(data)} graphs | {len(keys)} keys")
    except Exception as e:
        print(f"FAIL | {name} | {e}")

# 4. Compare model keys across graph files
print("\n--- CROSS-REPRESENTATION CONSISTENCY ---")
if len(loaded_graphs) == 4:
    key_sets = {
        name: set(data.keys())
        for name, data in loaded_graphs.items()
    }

    reference = key_sets["face"]
    for name, keys in key_sets.items():
        print(
            f"{'PASS' if keys == reference else 'FAIL'} | "
            f"{name} keys | {len(keys)} models"
        )

    print(f"All four datasets identical: {all(k == reference for k in key_sets.values())}")
else:
    print("SKIP | Not all four graph files loaded")

# 5. STEP file accessibility
print("\n--- STEP FILE ACCESSIBILITY ---")
step_files = list(RAW_STEP_DIR.glob("*.step")) if RAW_STEP_DIR.exists() else []
print(f"STEP files found: {len(step_files)}")

if step_files:
    readable = sum(
        1 for p in step_files
        if p.is_file() and p.stat().st_size > 0
    )
    print(f"Non-empty STEP files: {readable}/{len(step_files)}")

# 6. Verification metadata
print("\n--- VERIFICATION METADATA ---")
metadata_path = PROJECT_DIR / "environment_and_fingerprints.json"

if metadata_path.exists():
    try:
        with open(metadata_path, "r") as f:
            metadata = json.load(f)
        print("PASS | JSON readable")
        print("Top-level keys:", list(metadata.keys()))
    except Exception as e:
        print("FAIL | JSON:", e)

print("\n" + "=" * 65)
print("PERSISTENCE CHECK COMPLETE — NO FILES MODIFIED")
print("=" * 65)


CAD-GNN PROJECT PERSISTENCE CHECK

--- FILE EXISTENCE AND READ ACCESS ---
PASS | geometry_all_140.csv | 0.04 MB | readable
PASS | dataset_100.csv | 0.03 MB | readable
PASS | download_manifest.csv | 0.00 MB | readable
PASS | download_manifest_batch2.csv | 0.01 MB | readable
PASS | download_groups.csv | 0.01 MB | readable
PASS | face_graphs_100.pt | 5.77 MB | readable
PASS | edge_graphs_100.pt | 9.04 MB | readable
PASS | vertex_graphs_100.pt | 4.66 MB | readable
PASS | hetero_graphs_100.pt | 12.11 MB | readable
PASS | verification_per_model_100.csv | 0.01 MB | readable
PASS | verification_summary_100.csv | 0.00 MB | readable
PASS | environment_and_fingerprints.json | 0.00 MB | readable

--- CSV CONTENT CHECK ---
PASS | geometry_all_140.csv | rows=140 | columns=22
PASS | dataset_100.csv | rows=100 | columns=23
PASS | download_manifest.csv | rows=20 | columns=2
PASS | download_manifest_batch2.csv | rows=120 | columns=2
PASS | download_groups.csv | rows=140 | columns=2
PASS | verification_p

In [12]:
#11
# ============================================================
# CAD-GNN: RETRIEVE ORIGINAL GRAPH-BUILDING FUNCTIONS
# PURPOSE:
# Inspect the exact code used to create our graphs so that
# the independent feature audit uses the correct formulas.
#
# IMPORTANT:
# This cell does NOT modify, overwrite, or rebuild any files.
# ============================================================

# 1. Import Python's inspection module.
#    It allows us to retrieve the source code of existing functions.
import inspect


# 2. List the three functions whose original implementations
#    we need to inspect.
functions_to_inspect = [
    "load_shape",       # Reads STEP files and transfers B-Rep geometry.
    "unique_shapes",    # Extracts and indexes faces, edges and vertices.
    "build_graphs"      # Creates all four graph representations and features.
]


# 3. Inspect each function individually.
for name in functions_to_inspect:

    print("\n" + "=" * 75)
    print(f"FUNCTION: {name}")
    print("=" * 75)

    # Check whether the function currently exists in the Colab runtime.
    # globals() contains objects defined in the current notebook session.
    function_object = globals().get(name)

    if function_object is None:

        # If missing, we cannot retrieve its source from this runtime.
        print("NOT FOUND in current runtime.")

    else:

        try:
            # Retrieve and display the exact original Python source.
            # We need this to reproduce the feature definitions correctly.
            source_code = inspect.getsource(function_object)

            print(source_code)

        except Exception as error:

            # This can happen if the function was defined in a way
            # that prevents Python from retrieving its source.
            print("SOURCE RETRIEVAL ERROR:", error)


# ============================================================
# END
# No project files or graph datasets have been modified.
# ============================================================



FUNCTION: load_shape
def load_shape(path):
    # Same safe reader as before: create reader first, then set millimetres.
    reader = STEPControl_Reader()
    Interface_Static.SetCVal_s("xstep.cascade.unit", "MM")
    if reader.ReadFile(str(path)) != IFSelect_RetDone:
        raise RuntimeError("STEP could not be read")
    if reader.TransferRoots() == 0:
        raise RuntimeError("No roots transferred")
    shape = reader.OneShape()
    if shape.IsNull():
        raise RuntimeError("Null shape")
    return shape


FUNCTION: unique_shapes
def unique_shapes(shape, kind):
    m = TopTools_IndexedMapOfShape()      # an empty numbered list
    TopExp.MapShapes_s(shape, kind, m)    # fill it with each unique part of that kind
    # CAD numbering starts at 1 (not 0), so we loop from 1 to Extent().
    # Returns: the numbered list itself, AND a normal Python list of the parts.
    return m, [m.FindKey(i) for i in range(1, m.Extent() + 1)]


FUNCTION: build_graphs
def build_graphs(shape, volu

In [13]:
#12

# ================================================================
# CAD-GNN: INDEPENDENT B-REP TO GRAPH FEATURE CORRESPONDENCE AUDIT
#
# PURPOSE:
# 1. Reopen all 100 original STEP files.
# 2. Independently extract face, edge and vertex features.
# 3. Compare those features against all 400 saved graph objects.
# 4. Verify edge attributes against their B-Rep relationships.
#
# IMPORTANT:
# - Does NOT call build_graphs().
# - Does NOT rebuild or overwrite any graph.
# - Does NOT modify the original CSV or STEP files.
# - Prints discrepancies for investigation.
# ================================================================

import numpy as np
import pandas as pd
import torch

from pathlib import Path
from collections import Counter, defaultdict

# Import the OpenCascade classes used for independent extraction.
from OCP.Bnd import Bnd_Box
from OCP.BRepBndLib import BRepBndLib
from OCP.BRepGProp import BRepGProp
from OCP.GProp import GProp_GProps
from OCP.BRepAdaptor import BRepAdaptor_Surface, BRepAdaptor_Curve
from OCP.GeomAbs import (
    GeomAbs_Plane, GeomAbs_Cylinder, GeomAbs_Cone,
    GeomAbs_Sphere, GeomAbs_Torus,
    GeomAbs_Line, GeomAbs_Circle
)
from OCP.TopoDS import TopoDS
from OCP.BRep import BRep_Tool
from OCP.TopAbs import TopAbs_FACE, TopAbs_EDGE, TopAbs_VERTEX
from OCP.TopExp import TopExp
from OCP.TopTools import TopTools_IndexedMapOfShape


# ------------------------------------------------
# SECTION 1: LOAD THE EXISTING PROJECT FILES
# ------------------------------------------------

# Define the project locations.
PROJECT_DIR = Path("/content/drive/MyDrive/CAD_GNN_Project")
RAW_STEP_DIR = PROJECT_DIR / "raw_step"
GRAPH_DIR = PROJECT_DIR / "graphs"

# Read the selected 100-model dataset.
dataset = pd.read_csv(PROJECT_DIR / "dataset_100.csv")

# Load all four saved graph datasets into memory.
# weights_only=False is needed because these files contain PyG objects.
face_graphs = torch.load(
    GRAPH_DIR / "face_graphs_100.pt",
    map_location="cpu",
    weights_only=False
)

edge_graphs = torch.load(
    GRAPH_DIR / "edge_graphs_100.pt",
    map_location="cpu",
    weights_only=False
)

vertex_graphs = torch.load(
    GRAPH_DIR / "vertex_graphs_100.pt",
    map_location="cpu",
    weights_only=False
)

hetero_graphs = torch.load(
    GRAPH_DIR / "hetero_graphs_100.pt",
    map_location="cpu",
    weights_only=False
)

# Collect the four datasets under convenient names.
graph_sets = {
    "face": face_graphs,
    "edge": edge_graphs,
    "vertex": vertex_graphs,
    "hetero": hetero_graphs
}

print("Dataset rows:", len(dataset))
print("Graph dataset sizes:", {
    name: len(data) for name, data in graph_sets.items()
})


# ------------------------------------------------
# SECTION 2: INDEPENDENT B-REP ENTITY EXTRACTION
# ------------------------------------------------

def independent_entity_map(shape, kind):
    """
    Extract unique B-Rep entities using a fresh indexed map.

    This reproduces the intended entity enumeration independently,
    without calling the original unique_shapes() function.
    """
    indexed_map = TopTools_IndexedMapOfShape()
    TopExp.MapShapes_s(shape, kind, indexed_map)

    # OpenCascade uses indices starting at 1.
    entities = [
        indexed_map.FindKey(i)
        for i in range(1, indexed_map.Extent() + 1)
    ]

    return indexed_map, entities


def extract_reference_features(shape):
    """
    Independently calculate the expected numerical node features
    and B-Rep relationships for one STEP-derived shape.

    The formulas match the documented feature definitions in
    the original build_graphs() code.
    """

    # Calculate the same bounding box used by the original graph code.
    box = Bnd_Box()
    BRepBndLib.Add_s(shape, box)

    x0, y0, z0, x1, y1, z1 = box.Get()

    # Calculate the bounding-box diagonal and centre.
    diag = float(np.sqrt(
        (x1 - x0)**2 +
        (y1 - y0)**2 +
        (z1 - z0)**2
    ))

    if not np.isfinite(diag) or diag <= 0:
        raise RuntimeError("Invalid bounding-box diagonal")

    center = np.array([
        (x0 + x1) / 2,
        (y0 + y1) / 2,
        (z0 + z1) / 2
    ])

    # Independently enumerate all three B-Rep entity types.
    face_map, faces = independent_entity_map(shape, TopAbs_FACE)
    edge_map, edges = independent_entity_map(shape, TopAbs_EDGE)
    vertex_map, vertices = independent_entity_map(shape, TopAbs_VERTEX)

    # ============================================================
    # A. FACE FEATURES: 14 values per face
    # ============================================================

    face_features = []

    for face_shape in faces:

        face = TopoDS.Face_s(face_shape)

        # Calculate the face area and centre of mass.
        props = GProp_GProps()
        BRepGProp.SurfaceProperties_s(face, props)

        area = props.Mass()
        c = props.CentreOfMass()

        # Normalize the face centroid relative to the model centre.
        centroid = (
            np.array([c.X(), c.Y(), c.Z()]) - center
        ) / diag

        # Determine the geometric surface type.
        adaptor = BRepAdaptor_Surface(face, True)
        surface_type = adaptor.GetType()

        # Create the six surface-type indicators.
        flags = [
            float(surface_type == GeomAbs_Plane),
            float(surface_type == GeomAbs_Cylinder),
            float(surface_type == GeomAbs_Cone),
            float(surface_type == GeomAbs_Sphere),
            float(surface_type == GeomAbs_Torus)
        ]

        # Any other surface type is assigned to the sixth category.
        flags.append(1.0 - sum(flags))

        radius = 0.0
        axis = np.zeros(3)

        # Extract cylinder-specific geometric information.
        if surface_type == GeomAbs_Cylinder:

            cylinder = adaptor.Cylinder()
            radius = cylinder.Radius()

            direction = cylinder.Axis().Direction()
            axis = np.array([
                direction.X(),
                direction.Y(),
                direction.Z()
            ])

        # Extract cone-specific axis information.
        elif surface_type == GeomAbs_Cone:

            direction = adaptor.Cone().Axis().Direction()
            axis = np.array([
                direction.X(),
                direction.Y(),
                direction.Z()
            ])

        # Extract sphere-specific radius information.
        elif surface_type == GeomAbs_Sphere:

            radius = adaptor.Sphere().Radius()

        # Assemble the 14 face features in their original order.
        features = (
            flags +
            [area / diag**2, radius / diag] +
            list(axis) +
            list(centroid)
        )

        face_features.append(features)

    face_features = np.asarray(face_features, dtype=np.float32)

    # ============================================================
    # B. EDGE FEATURES: 4 values per edge
    # ============================================================

    edge_features = []

    for edge_shape in edges:

        edge = TopoDS.Edge_s(edge_shape)

        # Calculate the edge length.
        props = GProp_GProps()
        BRepGProp.LinearProperties_s(edge, props)

        length = props.Mass()

        # Determine the curve type.
        curve_type = BRepAdaptor_Curve(edge).GetType()

        is_line = float(curve_type == GeomAbs_Line)
        is_circle = float(curve_type == GeomAbs_Circle)

        # Assemble the four edge features.
        edge_features.append([
            length / diag,
            is_line,
            is_circle,
            1.0 - is_line - is_circle
        ])

    edge_features = np.asarray(edge_features, dtype=np.float32)

    # ============================================================
    # C. VERTEX FEATURES: 3 values per vertex
    # ============================================================

    vertex_features = []

    for vertex_shape in vertices:

        # Extract the original 3D vertex coordinate.
        point = BRep_Tool.Pnt_s(
            TopoDS.Vertex_s(vertex_shape)
        )

        coordinates = np.array([
            point.X(),
            point.Y(),
            point.Z()
        ])

        # Centre and normalize the coordinates.
        vertex_features.append(
            (coordinates - center) / diag
        )

    vertex_features = np.asarray(
        vertex_features,
        dtype=np.float32
    )

    # ============================================================
    # D. RECONSTRUCT B-REP INCIDENCE RELATIONSHIPS
    # ============================================================

    n_faces = len(faces)
    n_edges = len(edges)
    n_vertices = len(vertices)

    # Each edge records which faces use it.
    edge_to_faces = {
        i: set() for i in range(n_edges)
    }

    # Each edge records its endpoint vertices.
    edge_to_vertices = {
        i: set() for i in range(n_edges)
    }

    # Each vertex records the edges meeting there.
    vertex_to_edges = {
        i: set() for i in range(n_vertices)
    }

    # Each face records its boundary edges.
    face_to_edges = [
        set() for _ in range(n_faces)
    ]

    # Reconstruct face-edge relationships.
    for face_index, face_shape in enumerate(faces):

        local_map, _ = independent_entity_map(
            face_shape, TopAbs_EDGE
        )

        for j in range(1, local_map.Extent() + 1):

            local_edge = local_map.FindKey(j)

            # Convert the local edge into its global model index.
            edge_index = edge_map.FindIndex(local_edge) - 1

            if edge_index >= 0:

                face_to_edges[face_index].add(edge_index)
                edge_to_faces[edge_index].add(face_index)

    # Reconstruct edge-vertex relationships.
    for edge_index, edge_shape in enumerate(edges):

        local_map, _ = independent_entity_map(
            edge_shape, TopAbs_VERTEX
        )

        for j in range(1, local_map.Extent() + 1):

            local_vertex = local_map.FindKey(j)

            # Convert the local vertex into its global model index.
            vertex_index = vertex_map.FindIndex(local_vertex) - 1

            if vertex_index >= 0:

                edge_to_vertices[edge_index].add(vertex_index)
                vertex_to_edges[vertex_index].add(edge_index)

    # Return all reference information for this model.
    return {
        "face_x": face_features,
        "edge_x": edge_features,
        "vertex_x": vertex_features,
        "edge_to_faces": edge_to_faces,
        "edge_to_vertices": edge_to_vertices,
        "vertex_to_edges": vertex_to_edges,
        "face_to_edges": face_to_edges,
        "counts": (n_faces, n_edges, n_vertices),
        "diag": diag
    }


# ------------------------------------------------
# SECTION 3: COMPARISON HELPERS
# ------------------------------------------------

def compare_node_features(reference, stored, label):
    """
    Compare an independently extracted feature matrix with
    the feature matrix stored in a graph.

    Check:
    - Same matrix dimensions.
    - All values finite.
    - Numerical agreement within tolerance.
    """

    reference = np.asarray(reference, dtype=np.float32)
    stored = stored.detach().cpu().numpy()

    if reference.shape != stored.shape:
        return {
            "status": "SHAPE_MISMATCH",
            "max_error": np.nan
        }

    if not np.isfinite(stored).all():
        return {
            "status": "NONFINITE",
            "max_error": np.nan
        }

    difference = np.abs(reference - stored)
    max_error = float(difference.max()) if difference.size else 0.0

    # Allow a small floating-point tolerance.
    passed = np.allclose(
        reference,
        stored,
        rtol=1e-5,
        atol=1e-6,
        equal_nan=False
    )

    return {
        "status": "PASS" if passed else "VALUE_MISMATCH",
        "max_error": max_error
    }


def feature_key(values):
    """
    Convert a feature vector into a rounded tuple.

    This lets us compare edge attributes without depending
    on the order in which the graph builder stored connections.
    """
    values = np.asarray(values, dtype=np.float32)
    return tuple(np.round(values, 6).tolist())


def expected_connection_counter(pairs_and_features):
    """
    Build a multiset of directed connections and their features.

    A Counter is used because multiple B-Rep entities can produce
    connections between the same pair of graph nodes.
    """
    result = Counter()

    for source, target, features in pairs_and_features:

        result[(source, target, feature_key(features))] += 1

    return result


def actual_connection_counter(graph):
    """
    Read the stored graph connections and their edge attributes
    and convert them into the same Counter representation.
    """
    result = Counter()

    edge_index = graph.edge_index.detach().cpu().numpy()
    edge_attr = graph.edge_attr.detach().cpu().numpy()

    for i in range(edge_index.shape[1]):

        source = int(edge_index[0, i])
        target = int(edge_index[1, i])

        result[
            (source, target, feature_key(edge_attr[i]))
        ] += 1

    return result


# ------------------------------------------------
# SECTION 4: AUDIT ALL 100 MODELS
# ------------------------------------------------

results = []

print("\n" + "=" * 75)
print("STARTING INDEPENDENT FEATURE AUDIT")
print("=" * 75)

for row_number, row in dataset.iterrows():

    filename = row["filename"]
    step_path = RAW_STEP_DIR / filename

    print(
        f"\n[{row_number + 1}/100] "
        f"Auditing {filename}"
    )

    try:

        # Reopen the original STEP file and transfer its B-Rep shape.
        shape = load_shape(step_path)

        # Extract reference features independently.
        reference = extract_reference_features(shape)

        # Retrieve the four saved graph objects for this model.
        fg = face_graphs[filename]
        eg = edge_graphs[filename]
        vg = vertex_graphs[filename]
        hg = hetero_graphs[filename]

        # --------------------------------------------------------
        # A. Compare face, edge and vertex node features.
        # --------------------------------------------------------

        face_check = compare_node_features(
            reference["face_x"],
            fg.x,
            "face"
        )

        edge_check = compare_node_features(
            reference["edge_x"],
            eg.x,
            "edge"
        )

        vertex_check = compare_node_features(
            reference["vertex_x"],
            vg.x,
            "vertex"
        )

        # --------------------------------------------------------
        # B. Compare the heterogeneous graph's node features.
        # --------------------------------------------------------

        hetero_face_check = compare_node_features(
            reference["face_x"],
            hg["face"].x,
            "hetero face"
        )

        hetero_edge_check = compare_node_features(
            reference["edge_x"],
            hg["edge"].x,
            "hetero edge"
        )

        hetero_vertex_check = compare_node_features(
            reference["vertex_x"],
            hg["vertex"].x,
            "hetero vertex"
        )

        # --------------------------------------------------------
        # C. Independently construct expected FACE graph edges.
        #
        # Two faces are connected for every B-Rep edge they share.
        # The connection carries that B-Rep edge's four features.
        # --------------------------------------------------------

        expected_face_connections = []

        for edge_index, face_set in enumerate(
            reference["edge_to_faces"]
        ):

            face_list = sorted(face_set)

            for i in range(len(face_list)):
                for j in range(i + 1, len(face_list)):

                    a = face_list[i]
                    b = face_list[j]

                    features = reference["edge_x"][edge_index]

                    # Both directions are expected.
                    expected_face_connections.append(
                        (a, b, features)
                    )
                    expected_face_connections.append(
                        (b, a, features)
                    )

        # --------------------------------------------------------
        # D. Independently construct expected EDGE graph edges.
        #
        # Two B-Rep edges are connected when they share a vertex.
        # The connection carries that vertex's three coordinates.
        # --------------------------------------------------------

        expected_edge_connections = []

        for vertex_index, edge_set in enumerate(
            reference["vertex_to_edges"].values()
        ):

            edge_list = sorted(edge_set)

            for i in range(len(edge_list)):
                for j in range(i + 1, len(edge_list)):

                    a = edge_list[i]
                    b = edge_list[j]

                    features = reference["vertex_x"][vertex_index]

                    expected_edge_connections.append(
                        (a, b, features)
                    )
                    expected_edge_connections.append(
                        (b, a, features)
                    )

        # --------------------------------------------------------
        # E. Independently construct expected VERTEX graph edges.
        #
        # Every B-Rep edge with exactly two endpoint vertices
        # creates a connection between those two graph nodes.
        # Its edge attributes are the B-Rep edge's four features.
        # --------------------------------------------------------

        expected_vertex_connections = []

        for edge_index, vertex_set in enumerate(
            reference["edge_to_vertices"].values()
        ):

            endpoints = sorted(vertex_set)

            if len(endpoints) == 2:

                a, b = endpoints
                features = reference["edge_x"][edge_index]

                expected_vertex_connections.append(
                    (a, b, features)
                )
                expected_vertex_connections.append(
                    (b, a, features)
                )

        # --------------------------------------------------------
        # F. Compare expected connection-feature multisets.
        # --------------------------------------------------------

        face_attr_pass = (
            expected_connection_counter(expected_face_connections)
            ==
            actual_connection_counter(fg)
        )

        edge_attr_pass = (
            expected_connection_counter(expected_edge_connections)
            ==
            actual_connection_counter(eg)
        )

        vertex_attr_pass = (
            expected_connection_counter(expected_vertex_connections)
            ==
            actual_connection_counter(vg)
        )

        # --------------------------------------------------------
        # G. Check node counts in all four representations.
        # --------------------------------------------------------

        expected_counts = reference["counts"]

        count_checks = (
            fg.x.shape[0] == expected_counts[0] and
            eg.x.shape[0] == expected_counts[1] and
            vg.x.shape[0] == expected_counts[2] and
            hg["face"].x.shape[0] == expected_counts[0] and
            hg["edge"].x.shape[0] == expected_counts[1] and
            hg["vertex"].x.shape[0] == expected_counts[2]
        )

        # --------------------------------------------------------
        # H. Record the result for this model.
        # --------------------------------------------------------

        checks = [
            face_check["status"],
            edge_check["status"],
            vertex_check["status"],
            hetero_face_check["status"],
            hetero_edge_check["status"],
            hetero_vertex_check["status"]
        ]

        feature_pass = all(x == "PASS" for x in checks)

        connection_pass = (
            face_attr_pass and
            edge_attr_pass and
            vertex_attr_pass
        )

        model_pass = (
            feature_pass and
            connection_pass and
            count_checks
        )

        results.append({

            "filename": filename,

            "face_features": face_check["status"],
            "edge_features": edge_check["status"],
            "vertex_features": vertex_check["status"],

            "hetero_face_features": hetero_face_check["status"],
            "hetero_edge_features": hetero_edge_check["status"],
            "hetero_vertex_features": hetero_vertex_check["status"],

            "face_edge_attributes": face_attr_pass,
            "edge_edge_attributes": edge_attr_pass,
            "vertex_edge_attributes": vertex_attr_pass,

            "entity_counts": count_checks,

            "face_max_error": face_check["max_error"],
            "edge_max_error": edge_check["max_error"],
            "vertex_max_error": vertex_check["max_error"],

            "overall": "PASS" if model_pass else "FAIL"
        })

        print(
            "PASS" if model_pass else "CHECK REQUIRED",
            "| Features:", feature_pass,
            "| Connection attributes:", connection_pass,
            "| Counts:", count_checks
        )

    except Exception as error:

        # Preserve the failure information without stopping
        # the audit of the remaining models.
        results.append({
            "filename": filename,
            "overall": "ERROR",
            "error": str(error)
        })

        print("ERROR:", error)


# ------------------------------------------------
# SECTION 5: FINAL AUDIT REPORT
# ------------------------------------------------

audit_df = pd.DataFrame(results)

print("\n" + "=" * 75)
print("FINAL B-REP TO GRAPH FEATURE AUDIT")
print("=" * 75)

print("\nModels audited:", len(audit_df))

if "overall" in audit_df.columns:

    print("\nOverall status:")
    print(audit_df["overall"].value_counts(dropna=False))

    print("\nFeature comparison results:")
    for column in [
        "face_features",
        "edge_features",
        "vertex_features",
        "hetero_face_features",
        "hetero_edge_features",
        "hetero_vertex_features"
    ]:
        if column in audit_df.columns:
            print(f"{column}:")
            print(audit_df[column].value_counts(dropna=False))

    print("\nConnection attribute checks:")
    for column in [
        "face_edge_attributes",
        "edge_edge_attributes",
        "vertex_edge_attributes",
        "entity_counts"
    ]:
        if column in audit_df.columns:
            print(
                column,
                "| PASS:",
                int((audit_df[column] == True).sum()),
                "| FAIL:",
                int((audit_df[column] == False).sum())
            )

    # Display only models that need investigation.
    failed_models = audit_df[
        audit_df["overall"] != "PASS"
    ]

    print("\nModels requiring investigation:", len(failed_models))

    if len(failed_models):
        display(failed_models)

    # Display the maximum numerical discrepancies.
    print("\nMaximum numerical errors:")

    for column in [
        "face_max_error",
        "edge_max_error",
        "vertex_max_error"
    ]:
        if column in audit_df.columns:
            print(
                column,
                "|",
                audit_df[column].max()
            )

else:
    print("ERROR: Audit report could not be generated.")

print("\nAUDIT COMPLETE.")
print("No original project files or graph datasets were modified.")


Dataset rows: 100
Graph dataset sizes: {'face': 100, 'edge': 100, 'vertex': 100, 'hetero': 100}

STARTING INDEPENDENT FEATURE AUDIT

[1/100] Auditing 00000000_290a9120f9f249a7a05cfe9c_step_000.step
ERROR: 'int' object is not iterable

[2/100] Auditing 00000001_1ffb81a71e5b402e966b9341_step_000.step
ERROR: 'int' object is not iterable

[3/100] Auditing 00000005_d4fe04f0f5f84b52bd4f10e4_step_000.step
ERROR: 'int' object is not iterable

[4/100] Auditing 00000007_b33a147f86da49879455d286_step_000.step
ERROR: 'int' object is not iterable

[5/100] Auditing 00000008_9b3d6a97e8de4aa193b81000_step_000.step
ERROR: 'int' object is not iterable

[6/100] Auditing 00000010_b4b99d35e04b4277931f9a9c_step_000.step
ERROR: 'int' object is not iterable

[7/100] Auditing 00000011_e909f412cda24521865fac0f_step_000.step
ERROR: 'int' object is not iterable

[8/100] Auditing 00000012_f16882934f314832b639ffc0_step_000.step
ERROR: 'int' object is not iterable

[9/100] Auditing 00000013_c2f4d27f35ed4c138caf5c18_

KeyboardInterrupt: 

In [14]:
#12 redo

# ================================================================
# CAD-GNN: INDEPENDENT B-REP TO GRAPH FEATURE CORRESPONDENCE AUDIT
#
# PURPOSE:
# 1. Reopen all 100 original STEP files.
# 2. Independently extract face, edge and vertex features.
# 3. Compare those features against all 400 saved graph objects.
# 4. Verify graph edge attributes against B-Rep relationships.
# 5. Verify heterogeneous graph node features and entity counts.
#
# IMPORTANT:
# - Does NOT call build_graphs().
# - Does NOT rebuild or overwrite any graph.
# - Does NOT modify original CSV or STEP files.
# - Saves audit progress and reports in a NEW Drive folder.
# - A failed model does not stop the remaining audit.
# ================================================================


# ------------------------------------------------
# SECTION 0: IMPORT LIBRARIES
# ------------------------------------------------

import numpy as np
import pandas as pd
import torch
import json
import traceback

from pathlib import Path
from collections import Counter
from IPython.display import display

# OpenCascade geometry and topology classes.
from OCP.Bnd import Bnd_Box
from OCP.BRepBndLib import BRepBndLib
from OCP.BRepGProp import BRepGProp
from OCP.GProp import GProp_GProps
from OCP.BRepAdaptor import BRepAdaptor_Surface, BRepAdaptor_Curve

from OCP.GeomAbs import (
    GeomAbs_Plane,
    GeomAbs_Cylinder,
    GeomAbs_Cone,
    GeomAbs_Sphere,
    GeomAbs_Torus,
    GeomAbs_Line,
    GeomAbs_Circle
)

from OCP.TopoDS import TopoDS
from OCP.BRep import BRep_Tool
from OCP.TopAbs import (
    TopAbs_FACE,
    TopAbs_EDGE,
    TopAbs_VERTEX
)
from OCP.TopExp import TopExp
from OCP.TopTools import TopTools_IndexedMapOfShape

# STEP reader classes.
from OCP.STEPControl import STEPControl_Reader
from OCP.IFSelect import IFSelect_RetDone
from OCP.Interface import Interface_Static


# ------------------------------------------------
# SECTION 1: PROJECT PATHS AND EXISTING DATASETS
# ------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/CAD_GNN_Project")
RAW_STEP_DIR = PROJECT_DIR / "raw_step"
GRAPH_DIR = PROJECT_DIR / "graphs"

# All new audit outputs go into this separate folder.
# No original project file is written to or overwritten.
AUDIT_DIR = PROJECT_DIR / "audit_reports"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# Output files.
RESULTS_CSV = AUDIT_DIR / "feature_verification_100.csv"
ERRORS_CSV = AUDIT_DIR / "feature_verification_errors.csv"
SUMMARY_JSON = AUDIT_DIR / "feature_verification_summary.json"

# Load the original 100-model selection.
dataset = pd.read_csv(PROJECT_DIR / "dataset_100.csv")

# Load the four existing graph datasets.
face_graphs = torch.load(
    GRAPH_DIR / "face_graphs_100.pt",
    map_location="cpu",
    weights_only=False
)

edge_graphs = torch.load(
    GRAPH_DIR / "edge_graphs_100.pt",
    map_location="cpu",
    weights_only=False
)

vertex_graphs = torch.load(
    GRAPH_DIR / "vertex_graphs_100.pt",
    map_location="cpu",
    weights_only=False
)

hetero_graphs = torch.load(
    GRAPH_DIR / "hetero_graphs_100.pt",
    map_location="cpu",
    weights_only=False
)

# Organize the four graph datasets.
graph_sets = {
    "face": face_graphs,
    "edge": edge_graphs,
    "vertex": vertex_graphs,
    "hetero": hetero_graphs
}

print("Dataset rows:", len(dataset))

print(
    "Graph dataset sizes:",
    {name: len(data) for name, data in graph_sets.items()}
)

print("Audit output folder:", AUDIT_DIR)


# ------------------------------------------------
# SECTION 2: STEP FILE LOADER
# ------------------------------------------------

def load_audit_shape(step_path):
    """
    Independently reopen a STEP file and transfer its geometry.

    The unit setting matches the original project extraction.
    This function does not use or modify any saved graph.
    """

    reader = STEPControl_Reader()

    # Interpret STEP geometry in millimetres.
    Interface_Static.SetCVal_s(
        "xstep.cascade.unit",
        "MM"
    )

    # Read the STEP file.
    read_status = reader.ReadFile(str(step_path))

    if read_status != IFSelect_RetDone:
        raise RuntimeError("STEP file could not be read")

    # Transfer the STEP roots into an OpenCascade shape.
    transferred = reader.TransferRoots()

    if transferred == 0:
        raise RuntimeError("No STEP roots transferred")

    # Obtain the combined transferred shape.
    shape = reader.OneShape()

    if shape.IsNull():
        raise RuntimeError("Transferred shape is null")

    return shape


# ------------------------------------------------
# SECTION 3: INDEPENDENT ENTITY ENUMERATION
# ------------------------------------------------

def independent_entity_map(shape, kind):
    """
    Independently enumerate unique B-Rep entities.

    A fresh indexed map is constructed for each call.
    The returned entity ordering is used consistently when
    constructing the reference feature matrices and relationships.
    """

    indexed_map = TopTools_IndexedMapOfShape()

    # Map all entities of the requested topological type.
    TopExp.MapShapes_s(
        shape,
        kind,
        indexed_map
    )

    # OpenCascade indexed maps start at index 1.
    entities = [
        indexed_map.FindKey(i)
        for i in range(1, indexed_map.Extent() + 1)
    ]

    return indexed_map, entities


# ------------------------------------------------
# SECTION 4: INDEPENDENT NUMERICAL FEATURE EXTRACTION
# ------------------------------------------------

def extract_reference_features(shape):
    """
    Independently calculate the expected numerical features.

    Feature definitions and ordering are intentionally preserved
    from the original graph-building pipeline.
    """

    # ------------------------------------------------
    # A. Bounding box, diagonal and centre
    # ------------------------------------------------

    box = Bnd_Box()

    BRepBndLib.Add_s(
        shape,
        box
    )

    x0, y0, z0, x1, y1, z1 = box.Get()

    diag = float(np.sqrt(
        (x1 - x0)**2 +
        (y1 - y0)**2 +
        (z1 - z0)**2
    ))

    if not np.isfinite(diag) or diag <= 0:
        raise RuntimeError("Invalid bounding-box diagonal")

    center = np.array([
        (x0 + x1) / 2,
        (y0 + y1) / 2,
        (z0 + z1) / 2
    ])

    # ------------------------------------------------
    # B. Enumerate unique faces, edges and vertices
    # ------------------------------------------------

    face_map, faces = independent_entity_map(
        shape,
        TopAbs_FACE
    )

    edge_map, edges = independent_entity_map(
        shape,
        TopAbs_EDGE
    )

    vertex_map, vertices = independent_entity_map(
        shape,
        TopAbs_VERTEX
    )

    # ------------------------------------------------
    # C. FACE FEATURES: 14 values per face
    # ------------------------------------------------

    face_features = []

    for face_shape in faces:

        face = TopoDS.Face_s(face_shape)

        # Calculate surface area and centroid.
        props = GProp_GProps()

        BRepGProp.SurfaceProperties_s(
            face,
            props
        )

        area = props.Mass()
        c = props.CentreOfMass()

        # Normalize centroid coordinates.
        centroid = (
            np.array([c.X(), c.Y(), c.Z()]) - center
        ) / diag

        # Identify the geometric surface type.
        adaptor = BRepAdaptor_Surface(face, True)
        surface_type = adaptor.GetType()

        # Six mutually exclusive surface-type indicators.
        flags = [
            float(surface_type == GeomAbs_Plane),
            float(surface_type == GeomAbs_Cylinder),
            float(surface_type == GeomAbs_Cone),
            float(surface_type == GeomAbs_Sphere),
            float(surface_type == GeomAbs_Torus)
        ]

        # Sixth category: every other surface type.
        flags.append(1.0 - sum(flags))

        radius = 0.0
        axis = np.zeros(3)

        # Cylinder radius and axis direction.
        if surface_type == GeomAbs_Cylinder:

            cylinder = adaptor.Cylinder()
            radius = cylinder.Radius()

            direction = cylinder.Axis().Direction()

            axis = np.array([
                direction.X(),
                direction.Y(),
                direction.Z()
            ])

        # Cone axis direction.
        elif surface_type == GeomAbs_Cone:

            direction = adaptor.Cone().Axis().Direction()

            axis = np.array([
                direction.X(),
                direction.Y(),
                direction.Z()
            ])

        # Sphere radius.
        elif surface_type == GeomAbs_Sphere:

            radius = adaptor.Sphere().Radius()

        # Preserve the original 14-feature ordering.
        features = (
            flags +
            [area / diag**2, radius / diag] +
            list(axis) +
            list(centroid)
        )

        face_features.append(features)

    face_features = np.asarray(
        face_features,
        dtype=np.float32
    ).reshape(-1, 14)

    # ------------------------------------------------
    # D. EDGE FEATURES: 4 values per edge
    # ------------------------------------------------

    edge_features = []

    for edge_shape in edges:

        edge = TopoDS.Edge_s(edge_shape)

        # Calculate edge length.
        props = GProp_GProps()

        BRepGProp.LinearProperties_s(
            edge,
            props
        )

        length = props.Mass()

        # Identify the geometric curve type.
        curve_type = BRepAdaptor_Curve(edge).GetType()

        is_line = float(curve_type == GeomAbs_Line)
        is_circle = float(curve_type == GeomAbs_Circle)

        # Preserve the original four-feature ordering.
        edge_features.append([
            length / diag,
            is_line,
            is_circle,
            1.0 - is_line - is_circle
        ])

    edge_features = np.asarray(
        edge_features,
        dtype=np.float32
    ).reshape(-1, 4)

    # ------------------------------------------------
    # E. VERTEX FEATURES: 3 values per vertex
    # ------------------------------------------------

    vertex_features = []

    for vertex_shape in vertices:

        # Obtain the original vertex coordinate.
        point = BRep_Tool.Pnt_s(
            TopoDS.Vertex_s(vertex_shape)
        )

        coordinates = np.array([
            point.X(),
            point.Y(),
            point.Z()
        ])

        # Centre and normalize the coordinates.
        vertex_features.append(
            (coordinates - center) / diag
        )

    vertex_features = np.asarray(
        vertex_features,
        dtype=np.float32
    ).reshape(-1, 3)

    # ------------------------------------------------
    # F. Reconstruct B-Rep incidence relationships
    # ------------------------------------------------

    n_faces = len(faces)
    n_edges = len(edges)
    n_vertices = len(vertices)

    # Dictionary keys are global zero-based entity indices.
    edge_to_faces = {
        i: set() for i in range(n_edges)
    }

    edge_to_vertices = {
        i: set() for i in range(n_edges)
    }

    vertex_to_edges = {
        i: set() for i in range(n_vertices)
    }

    face_to_edges = [
        set() for _ in range(n_faces)
    ]

    # ------------------------------------------------
    # F1. Face-to-edge relationships
    # ------------------------------------------------

    for face_index, face_shape in enumerate(faces):

        local_map, _ = independent_entity_map(
            face_shape,
            TopAbs_EDGE
        )

        for j in range(1, local_map.Extent() + 1):

            local_edge = local_map.FindKey(j)

            # Convert local edge identity to global model index.
            edge_index = edge_map.FindIndex(local_edge) - 1

            if edge_index >= 0:

                face_to_edges[face_index].add(edge_index)

                edge_to_faces[edge_index].add(face_index)

    # ------------------------------------------------
    # F2. Edge-to-vertex relationships
    # ------------------------------------------------

    for edge_index, edge_shape in enumerate(edges):

        local_map, _ = independent_entity_map(
            edge_shape,
            TopAbs_VERTEX
        )

        for j in range(1, local_map.Extent() + 1):

            local_vertex = local_map.FindKey(j)

            # Convert local vertex identity to global model index.
            vertex_index = vertex_map.FindIndex(local_vertex) - 1

            if vertex_index >= 0:

                edge_to_vertices[edge_index].add(vertex_index)

                vertex_to_edges[vertex_index].add(edge_index)

    # Return all independently extracted reference information.
    return {
        "face_x": face_features,
        "edge_x": edge_features,
        "vertex_x": vertex_features,

        "edge_to_faces": edge_to_faces,
        "edge_to_vertices": edge_to_vertices,
        "vertex_to_edges": vertex_to_edges,
        "face_to_edges": face_to_edges,

        "counts": (
            n_faces,
            n_edges,
            n_vertices
        ),

        "diag": diag
    }


# ------------------------------------------------
# SECTION 5: NUMERICAL COMPARISON HELPERS
# ------------------------------------------------

def compare_node_features(reference, stored):
    """
    Compare independently extracted features with saved graph
    node features.

    Uses the original numerical tolerances:
    rtol = 1e-5
    atol = 1e-6
    """

    reference = np.asarray(
        reference,
        dtype=np.float32
    )

    stored = stored.detach().cpu().numpy()

    if reference.shape != stored.shape:

        return {
            "status": "SHAPE_MISMATCH",
            "max_error": np.nan
        }

    if not np.isfinite(reference).all():

        return {
            "status": "REFERENCE_NONFINITE",
            "max_error": np.nan
        }

    if not np.isfinite(stored).all():

        return {
            "status": "STORED_NONFINITE",
            "max_error": np.nan
        }

    difference = np.abs(reference - stored)

    max_error = (
        float(difference.max())
        if difference.size
        else 0.0
    )

    passed = np.allclose(
        reference,
        stored,
        rtol=1e-5,
        atol=1e-6,
        equal_nan=False
    )

    return {
        "status": "PASS" if passed else "VALUE_MISMATCH",
        "max_error": max_error
    }


# ------------------------------------------------
# SECTION 6: EDGE-ATTRIBUTE COMPARISON HELPERS
# ------------------------------------------------

def feature_key(values):
    """
    Convert an edge-attribute vector into a rounded tuple.

    Rounding to six decimal places is retained from the original
    audit to avoid insignificant floating-point differences.
    """

    values = np.asarray(
        values,
        dtype=np.float32
    )

    return tuple(
        np.round(values, 6).tolist()
    )


def expected_connection_counter(pairs_and_features):
    """
    Construct a multiset of expected directed graph connections.

    Each entry contains:
    source index, target index and associated edge features.

    Counter preserves multiplicity if multiple B-Rep entities
    generate connections between the same node pair.
    """

    result = Counter()

    for source, target, features in pairs_and_features:

        result[
            (source, target, feature_key(features))
        ] += 1

    return result


def actual_connection_counter(graph):
    """
    Extract directed connections and edge attributes from
    an existing homogeneous graph.
    """

    result = Counter()

    edge_index = graph.edge_index.detach().cpu().numpy()
    edge_attr = graph.edge_attr.detach().cpu().numpy()

    if edge_index.shape[1] != edge_attr.shape[0]:
        raise RuntimeError(
            "Graph edge_index and edge_attr counts disagree"
        )

    for i in range(edge_index.shape[1]):

        source = int(edge_index[0, i])
        target = int(edge_index[1, i])

        result[
            (source, target, feature_key(edge_attr[i]))
        ] += 1

    return result


# ------------------------------------------------
# SECTION 7: INCREMENTAL REPORT SAVING
# ------------------------------------------------

def save_progress(results, errors):
    """
    Save current results after each completed model.

    These are new report files inside audit_reports.
    The original project datasets remain untouched.
    """

    pd.DataFrame(results).to_csv(
        RESULTS_CSV,
        index=False
    )

    pd.DataFrame(errors).to_csv(
        ERRORS_CSV,
        index=False
    )


# ------------------------------------------------
# SECTION 8: AUDIT ALL 100 MODELS
# ------------------------------------------------

results = []
errors = []

print("\n" + "=" * 75)
print("STARTING INDEPENDENT FEATURE AUDIT")
print("=" * 75)

for row_number, row in dataset.iterrows():

    filename = row["filename"]
    step_path = RAW_STEP_DIR / filename

    print(
        f"\n[{row_number + 1}/{len(dataset)}] "
        f"Auditing {filename}"
    )

    try:

        # ------------------------------------------------
        # A. Reopen the original STEP file.
        # ------------------------------------------------

        shape = load_audit_shape(step_path)

        # ------------------------------------------------
        # B. Independently extract numerical features.
        # ------------------------------------------------

        reference = extract_reference_features(shape)

        # ------------------------------------------------
        # C. Retrieve the four existing graph objects.
        # ------------------------------------------------

        fg = face_graphs[filename]
        eg = edge_graphs[filename]
        vg = vertex_graphs[filename]
        hg = hetero_graphs[filename]

        # ------------------------------------------------
        # D. Compare homogeneous node features.
        # ------------------------------------------------

        face_check = compare_node_features(
            reference["face_x"],
            fg.x
        )

        edge_check = compare_node_features(
            reference["edge_x"],
            eg.x
        )

        vertex_check = compare_node_features(
            reference["vertex_x"],
            vg.x
        )

        # ------------------------------------------------
        # E. Compare heterogeneous node features.
        # ------------------------------------------------

        hetero_face_check = compare_node_features(
            reference["face_x"],
            hg["face"].x
        )

        hetero_edge_check = compare_node_features(
            reference["edge_x"],
            hg["edge"].x
        )

        hetero_vertex_check = compare_node_features(
            reference["vertex_x"],
            hg["vertex"].x
        )

        # ------------------------------------------------
        # F. Expected FACE graph connections.
        #
        # FIX:
        # edge_to_faces is a dictionary.
        # .items() returns (edge_index, face_set).
        # Iterating directly over the dictionary returns
        # integer keys and caused the original error.
        # ------------------------------------------------

        expected_face_connections = []

        for edge_index, face_set in reference[
            "edge_to_faces"
        ].items():

            face_list = sorted(face_set)

            # Every pair of faces sharing this B-Rep edge
            # receives a connection in both directions.
            for i in range(len(face_list)):

                for j in range(i + 1, len(face_list)):

                    a = face_list[i]
                    b = face_list[j]

                    features = reference["edge_x"][edge_index]

                    expected_face_connections.append(
                        (a, b, features)
                    )

                    expected_face_connections.append(
                        (b, a, features)
                    )

        # ------------------------------------------------
        # G. Expected EDGE graph connections.
        #
        # Two B-Rep edges are connected when they share
        # a B-Rep vertex.
        #
        # The connection carries that vertex's coordinates.
        # ------------------------------------------------

        expected_edge_connections = []

        for vertex_index, edge_set in reference[
            "vertex_to_edges"
        ].items():

            edge_list = sorted(edge_set)

            for i in range(len(edge_list)):

                for j in range(i + 1, len(edge_list)):

                    a = edge_list[i]
                    b = edge_list[j]

                    features = reference["vertex_x"][vertex_index]

                    expected_edge_connections.append(
                        (a, b, features)
                    )

                    expected_edge_connections.append(
                        (b, a, features)
                    )

        # ------------------------------------------------
        # H. Expected VERTEX graph connections.
        #
        # Each B-Rep edge with exactly two distinct
        # endpoint vertices creates a bidirectional link.
        #
        # Its edge attributes are the four edge features.
        # ------------------------------------------------

        expected_vertex_connections = []

        for edge_index, vertex_set in reference[
            "edge_to_vertices"
        ].items():

            endpoints = sorted(vertex_set)

            if len(endpoints) == 2:

                a, b = endpoints

                features = reference["edge_x"][edge_index]

                expected_vertex_connections.append(
                    (a, b, features)
                )

                expected_vertex_connections.append(
                    (b, a, features)
                )

        # ------------------------------------------------
        # I. Compare expected and stored connection multisets.
        # ------------------------------------------------

        face_attr_pass = (
            expected_connection_counter(
                expected_face_connections
            )
            ==
            actual_connection_counter(fg)
        )

        edge_attr_pass = (
            expected_connection_counter(
                expected_edge_connections
            )
            ==
            actual_connection_counter(eg)
        )

        vertex_attr_pass = (
            expected_connection_counter(
                expected_vertex_connections
            )
            ==
            actual_connection_counter(vg)
        )

        # ------------------------------------------------
        # J. Verify entity counts across all representations.
        # ------------------------------------------------

        expected_counts = reference["counts"]

        count_checks = (
            fg.x.shape[0] == expected_counts[0] and
            eg.x.shape[0] == expected_counts[1] and
            vg.x.shape[0] == expected_counts[2] and

            hg["face"].x.shape[0] == expected_counts[0] and
            hg["edge"].x.shape[0] == expected_counts[1] and
            hg["vertex"].x.shape[0] == expected_counts[2]
        )

        # ------------------------------------------------
        # K. Determine model-level result.
        # ------------------------------------------------

        feature_checks = [
            face_check["status"],
            edge_check["status"],
            vertex_check["status"],

            hetero_face_check["status"],
            hetero_edge_check["status"],
            hetero_vertex_check["status"]
        ]

        feature_pass = all(
            status == "PASS"
            for status in feature_checks
        )

        connection_pass = (
            face_attr_pass and
            edge_attr_pass and
            vertex_attr_pass
        )

        model_pass = (
            feature_pass and
            connection_pass and
            count_checks
        )

        # ------------------------------------------------
        # L. Store detailed results.
        # ------------------------------------------------

        model_result = {

            "filename": filename,

            "face_features": face_check["status"],
            "edge_features": edge_check["status"],
            "vertex_features": vertex_check["status"],

            "hetero_face_features": hetero_face_check["status"],
            "hetero_edge_features": hetero_edge_check["status"],
            "hetero_vertex_features": hetero_vertex_check["status"],

            "face_edge_attributes": face_attr_pass,
            "edge_edge_attributes": edge_attr_pass,
            "vertex_edge_attributes": vertex_attr_pass,

            "entity_counts": count_checks,

            "face_max_error": face_check["max_error"],
            "edge_max_error": edge_check["max_error"],
            "vertex_max_error": vertex_check["max_error"],

            "hetero_face_max_error": hetero_face_check["max_error"],
            "hetero_edge_max_error": hetero_edge_check["max_error"],
            "hetero_vertex_max_error": hetero_vertex_check["max_error"],

            "overall": "PASS" if model_pass else "FAIL"
        }

        results.append(model_result)

        # Save progress immediately after each successful model.
        save_progress(results, errors)

        print(
            model_result["overall"],
            "| Features:", feature_pass,
            "| Connection attributes:", connection_pass,
            "| Counts:", count_checks
        )

    except Exception as error:

        # Record the error and its full traceback.
        error_record = {
            "filename": filename,
            "error": str(error),
            "traceback": traceback.format_exc()
        }

        errors.append(error_record)

        # Also include the failed model in the main results.
        results.append({
            "filename": filename,
            "overall": "ERROR",
            "error": str(error)
        })

        # Save immediately so interrupted runs retain progress.
        save_progress(results, errors)

        print("ERROR:", error)
        print("Full traceback saved in the error report.")

# ------------------------------------------------
# SECTION 9: FINAL AUDIT SUMMARY
# ------------------------------------------------

audit_df = pd.DataFrame(results)
error_df = pd.DataFrame(errors)

print("\n" + "=" * 75)
print("FINAL B-REP TO GRAPH FEATURE AUDIT")
print("=" * 75)

print("\nModels processed:", len(audit_df))
print("Models with extraction errors:", len(error_df))

if not audit_df.empty:

    print("\nOverall status:")
    print(
        audit_df["overall"].value_counts(dropna=False)
    )

    # Report each numerical feature comparison.
    feature_columns = [
        "face_features",
        "edge_features",
        "vertex_features",
        "hetero_face_features",
        "hetero_edge_features",
        "hetero_vertex_features"
    ]

    print("\nFeature comparison results:")

    for column in feature_columns:

        if column in audit_df.columns:

            print(f"\n{column}:")
            print(
                audit_df[column].value_counts(dropna=False)
            )

    # Report graph connection attribute checks.
    connection_columns = [
        "face_edge_attributes",
        "edge_edge_attributes",
        "vertex_edge_attributes",
        "entity_counts"
    ]

    print("\nConnection and count checks:")

    for column in connection_columns:

        if column in audit_df.columns:

            print(
                column,
                "| PASS:",
                int((audit_df[column] == True).sum()),
                "| FAIL:",
                int((audit_df[column] == False).sum())
            )

    # Show every model that did not pass.
    failed_models = audit_df[
        audit_df["overall"] != "PASS"
    ]

    print(
        "\nModels requiring investigation:",
        len(failed_models)
    )

    if not failed_models.empty:
        display(failed_models)

    # Report maximum numerical errors.
    error_columns = [
        "face_max_error",
        "edge_max_error",
        "vertex_max_error",
        "hetero_face_max_error",
        "hetero_edge_max_error",
        "hetero_vertex_max_error"
    ]

    print("\nMaximum numerical errors:")

    for column in error_columns:

        if column in audit_df.columns:

            print(
                column,
                "|",
                audit_df[column].max()
            )

# ------------------------------------------------
# SECTION 10: SAVE FINAL SUMMARY
# ------------------------------------------------

summary = {

    "models_in_dataset": int(len(dataset)),

    "models_processed": int(len(audit_df)),

    "models_passed": int(
        (audit_df["overall"] == "PASS").sum()
    ) if not audit_df.empty else 0,

    "models_failed": int(
        (audit_df["overall"] == "FAIL").sum()
    ) if not audit_df.empty else 0,

    "models_with_errors": int(
        (audit_df["overall"] == "ERROR").sum()
    ) if not audit_df.empty else 0,

    "feature_tolerance": {
        "rtol": 1e-5,
        "atol": 1e-6
    },

    "edge_attribute_rounding_decimals": 6,

    "original_graphs_modified": False,

    "original_dataset_modified": False,

    "results_csv": str(RESULTS_CSV),

    "errors_csv": str(ERRORS_CSV)
}

with open(SUMMARY_JSON, "w") as file:
    json.dump(summary, file, indent=4)

# Save the final reports once more.
save_progress(results, errors)

print("\n" + "=" * 75)
print("AUDIT COMPLETE")
print("=" * 75)

print("\nSaved files:")
print("Results:", RESULTS_CSV)
print("Errors:", ERRORS_CSV)
print("Summary:", SUMMARY_JSON)

print("\nOriginal graph datasets were not modified.")
print("Original master CSV and STEP files were not modified.")


Dataset rows: 100
Graph dataset sizes: {'face': 100, 'edge': 100, 'vertex': 100, 'hetero': 100}
Audit output folder: /content/drive/MyDrive/CAD_GNN_Project/audit_reports

STARTING INDEPENDENT FEATURE AUDIT

[1/100] Auditing 00000000_290a9120f9f249a7a05cfe9c_step_000.step
PASS | Features: True | Connection attributes: True | Counts: True

[2/100] Auditing 00000001_1ffb81a71e5b402e966b9341_step_000.step
PASS | Features: True | Connection attributes: True | Counts: True

[3/100] Auditing 00000005_d4fe04f0f5f84b52bd4f10e4_step_000.step
PASS | Features: True | Connection attributes: True | Counts: True

[4/100] Auditing 00000007_b33a147f86da49879455d286_step_000.step
PASS | Features: True | Connection attributes: True | Counts: True

[5/100] Auditing 00000008_9b3d6a97e8de4aa193b81000_step_000.step
PASS | Features: True | Connection attributes: True | Counts: True

[6/100] Auditing 00000010_b4b99d35e04b4277931f9a9c_step_000.step
PASS | Features: True | Connection attributes: True | Counts: T

In [15]:
#13

# ================================================================
# CAD-GNN: FINAL GEOMETRY NORMALIZATION AND DUPLICATE AUDIT
#
# PART A:
# Compare the original Add_s bounding box against AddOptimal_s.
#
# PART B:
# Independently calculate geometric descriptors for all models.
#
# PART C:
# Check exact STEP-file duplicates and screen possible geometric
# duplicates using topology and numerical geometry descriptors.
#
# IMPORTANT:
# - Does NOT rebuild or modify any graph.
# - Does NOT modify the master dataset or STEP files.
# - Does NOT change the original normalization method.
# - Saves all reports in audit_reports.
# ================================================================

import numpy as np
import pandas as pd
import hashlib
import json

from pathlib import Path
from itertools import combinations

from OCP.Bnd import Bnd_Box
from OCP.BRepBndLib import BRepBndLib
from OCP.BRepGProp import BRepGProp
from OCP.GProp import GProp_GProps
from OCP.TopAbs import TopAbs_FACE, TopAbs_EDGE, TopAbs_VERTEX
from OCP.TopExp import TopExp
from OCP.TopTools import TopTools_IndexedMapOfShape
from OCP.TopoDS import TopoDS
from OCP.BRepAdaptor import BRepAdaptor_Curve


# ------------------------------------------------
# SECTION 1: PATHS AND OUTPUT FILES
# ------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/CAD_GNN_Project")
RAW_STEP_DIR = PROJECT_DIR / "raw_step"
AUDIT_DIR = PROJECT_DIR / "audit_reports"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

BBOX_CSV = AUDIT_DIR / "bbox_comparison_100.csv"
DUPLICATE_CSV = AUDIT_DIR / "duplicate_assessment.csv"
DUPLICATE_HASH_CSV = AUDIT_DIR / "exact_duplicate_hashes.csv"
SUMMARY_JSON = AUDIT_DIR / "bbox_duplicate_summary.json"

# Use the already established 100-model selection.
dataset = pd.read_csv(PROJECT_DIR / "dataset_100.csv")

print("Models to audit:", len(dataset))
print("Output directory:", AUDIT_DIR)


# ------------------------------------------------
# SECTION 2: INDEPENDENT ENTITY EXTRACTION
# ------------------------------------------------

def get_entities(shape, kind):
    """
    Return unique B-Rep entities in indexed-map order.

    The ordering is used consistently for the geometric
    descriptors calculated below.
    """

    indexed_map = TopTools_IndexedMapOfShape()

    TopExp.MapShapes_s(
        shape,
        kind,
        indexed_map
    )

    return [
        indexed_map.FindKey(i)
        for i in range(1, indexed_map.Extent() + 1)
    ]


# ------------------------------------------------
# SECTION 3: EXACT FILE FINGERPRINT
# ------------------------------------------------

def sha256_file(path):
    """
    Calculate the SHA-256 fingerprint of the complete STEP file.

    Identical hashes mean the files are byte-for-byte identical.
    Different hashes do not necessarily mean different geometry.
    """

    digest = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):

            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------
# SECTION 4: EXTRACT GEOMETRY AND BOTH BOUNDING BOXES
# ------------------------------------------------

def inspect_model(filename, stored_row):
    """
    Independently inspect one STEP model.

    Calculate:
    - Original Add_s bounding box.
    - Tight AddOptimal_s bounding box.
    - Bounding-box diagonals and volumes.
    - Whole-shape volume and surface area.
    - Face, edge and vertex counts.
    - Sorted face-area and edge-length distributions.
    - Exact STEP file hash.
    """

    path = RAW_STEP_DIR / filename

    if not path.exists():
        raise FileNotFoundError(path)

    # Load the original STEP geometry.
    shape = load_audit_shape(path)

    # ------------------------------------------------
    # A. Original bounding box used by graph builder
    # ------------------------------------------------

    loose_box = Bnd_Box()

    BRepBndLib.Add_s(
        shape,
        loose_box
    )

    lx0, ly0, lz0, lx1, ly1, lz1 = loose_box.Get()

    loose_dimensions = np.array([
        lx1 - lx0,
        ly1 - ly0,
        lz1 - lz0
    ], dtype=float)

    loose_diagonal = float(
        np.linalg.norm(loose_dimensions)
    )

    loose_bbox_volume = float(
        np.prod(loose_dimensions)
    )

    # ------------------------------------------------
    # B. Tighter bounding box
    # ------------------------------------------------

    tight_box = Bnd_Box()

    BRepBndLib.AddOptimal_s(
        shape,
        tight_box,
        False,
        False
    )

    tx0, ty0, tz0, tx1, ty1, tz1 = tight_box.Get()

    tight_dimensions = np.array([
        tx1 - tx0,
        ty1 - ty0,
        tz1 - tz0
    ], dtype=float)

    tight_diagonal = float(
        np.linalg.norm(tight_dimensions)
    )

    tight_bbox_volume = float(
        np.prod(tight_dimensions)
    )

    if loose_diagonal <= 0 or tight_diagonal <= 0:
        raise RuntimeError("Invalid bounding-box diagonal")

    # ------------------------------------------------
    # C. Whole-shape geometric properties
    # ------------------------------------------------

    volume_props = GProp_GProps()

    BRepGProp.VolumeProperties_s(
        shape,
        volume_props
    )

    independent_volume = float(
        volume_props.Mass()
    )

    area_props = GProp_GProps()

    BRepGProp.SurfaceProperties_s(
        shape,
        area_props
    )

    independent_area = float(
        area_props.Mass()
    )

    # ------------------------------------------------
    # D. Enumerate topological entities
    # ------------------------------------------------

    faces = get_entities(shape, TopAbs_FACE)
    edges = get_entities(shape, TopAbs_EDGE)
    vertices = get_entities(shape, TopAbs_VERTEX)

    # ------------------------------------------------
    # E. Calculate sorted face-area distribution
    # ------------------------------------------------

    face_areas = []

    for face_shape in faces:

        face = TopoDS.Face_s(face_shape)

        props = GProp_GProps()

        BRepGProp.SurfaceProperties_s(
            face,
            props
        )

        face_areas.append(
            float(props.Mass())
        )

    face_areas = np.sort(
        np.asarray(face_areas, dtype=float)
    )

    # ------------------------------------------------
    # F. Calculate sorted edge-length distribution
    # ------------------------------------------------

    edge_lengths = []

    for edge_shape in edges:

        edge = TopoDS.Edge_s(edge_shape)

        props = GProp_GProps()

        BRepGProp.LinearProperties_s(
            edge,
            props
        )

        edge_lengths.append(
            float(props.Mass())
        )

    edge_lengths = np.sort(
        np.asarray(edge_lengths, dtype=float)
    )

    # ------------------------------------------------
    # G. Stored dataset values
    # ------------------------------------------------

    stored_volume = float(stored_row["volume"])
    stored_area = float(stored_row["surface_area"])
    stored_diagonal = float(stored_row["bbox_diagonal"])

    # ------------------------------------------------
    # H. Normalization impact
    # ------------------------------------------------

    # The original target is V / loose_diagonal^3.
    y_loose = independent_volume / loose_diagonal**3

    # This is a diagnostic alternative, not a replacement target.
    y_tight = independent_volume / tight_diagonal**3

    # Relative increase in the original bounding-box volume
    # compared with the tight bounding-box volume.
    bbox_inflation_percent = (
        (loose_bbox_volume / tight_bbox_volume) - 1
    ) * 100

    # Relative diagonal difference.
    diagonal_inflation_percent = (
        (loose_diagonal / tight_diagonal) - 1
    ) * 100

    # Effect on normalized volume target if the tight diagonal
    # were used instead of the original loose diagonal.
    target_change_percent = (
        (y_tight / y_loose) - 1
    ) * 100

    return {

        "filename": filename,

        "sha256": sha256_file(path),

        "faces": len(faces),
        "edges": len(edges),
        "vertices": len(vertices),

        "independent_volume": independent_volume,
        "stored_volume": stored_volume,

        "independent_surface_area": independent_area,
        "stored_surface_area": stored_area,

        "loose_x": loose_dimensions[0],
        "loose_y": loose_dimensions[1],
        "loose_z": loose_dimensions[2],

        "tight_x": tight_dimensions[0],
        "tight_y": tight_dimensions[1],
        "tight_z": tight_dimensions[2],

        "loose_diagonal": loose_diagonal,
        "tight_diagonal": tight_diagonal,

        "stored_diagonal": stored_diagonal,

        "loose_bbox_volume": loose_bbox_volume,
        "tight_bbox_volume": tight_bbox_volume,

        "bbox_inflation_percent": bbox_inflation_percent,

        "diagonal_inflation_percent": diagonal_inflation_percent,

        "original_normalized_volume": y_loose,
        "tight_normalized_volume_diagnostic": y_tight,

        "target_change_percent_if_tight": target_change_percent,

        # Keep sorted geometric distributions for pairwise
        # duplicate screening. These are not model inputs.
        "_face_areas": face_areas,
        "_edge_lengths": edge_lengths,

        # Preserve dimensions as arrays for pairwise comparison.
        "_loose_dimensions": loose_dimensions,
        "_tight_dimensions": tight_dimensions
    }


# ------------------------------------------------
# SECTION 5: AUDIT ALL 100 MODELS
# ------------------------------------------------

model_records = []
errors = []

print("\n" + "=" * 75)
print("STARTING BOUNDING-BOX AND GEOMETRY AUDIT")
print("=" * 75)

for index, row in dataset.iterrows():

    filename = row["filename"]

    print(
        f"\n[{index + 1}/{len(dataset)}] {filename}"
    )

    try:

        record = inspect_model(
            filename,
            row
        )

        model_records.append(record)

        # Save progress after each successful model.
        temporary_df = pd.DataFrame([
            {
                key: value
                for key, value in item.items()
                if not key.startswith("_")
            }
            for item in model_records
        ])

        temporary_df.to_csv(
            BBOX_CSV,
            index=False
        )

        print(
            "PASS | Faces:", record["faces"],
            "| Edges:", record["edges"],
            "| Vertices:", record["vertices"],
            "| BBox inflation:",
            round(record["bbox_inflation_percent"], 3), "%"
        )

    except Exception as error:

        errors.append({
            "filename": filename,
            "error": str(error)
        })

        print("ERROR:", error)


# ------------------------------------------------
# SECTION 6: BOUNDING-BOX COMPARISON REPORT
# ------------------------------------------------

bbox_df = pd.DataFrame([
    {
        key: value
        for key, value in item.items()
        if not key.startswith("_")
    }
    for item in model_records
])

print("\n" + "=" * 75)
print("BOUNDING-BOX AUDIT SUMMARY")
print("=" * 75)

print("Successfully inspected:", len(bbox_df))
print("Errors:", len(errors))

if not bbox_df.empty:

    print("\nBounding-box inflation percentage:")
    print(
        bbox_df["bbox_inflation_percent"].describe()
    )

    print("\nDiagonal inflation percentage:")
    print(
        bbox_df["diagonal_inflation_percent"].describe()
    )

    print("\nChange in normalized volume if tight diagonal used:")
    print(
        bbox_df["target_change_percent_if_tight"].describe()
    )

    print("\nLargest bounding-box inflation cases:")

    display(
        bbox_df.sort_values(
            "bbox_inflation_percent",
            ascending=False
        ).head(15)
    )

    print("\nLargest normalization-impact cases:")

    display(
        bbox_df.sort_values(
            "target_change_percent_if_tight",
            ascending=False
        ).head(15)
    )


# ------------------------------------------------
# SECTION 7: EXACT DUPLICATE DETECTION
# ------------------------------------------------

print("\n" + "=" * 75)
print("EXACT FILE DUPLICATE CHECK")
print("=" * 75)

hash_groups = {}

for record in model_records:

    hash_groups.setdefault(
        record["sha256"],
        []
    ).append(record["filename"])

exact_duplicate_groups = [
    {
        "sha256": digest,
        "filenames": filenames,
        "count": len(filenames)
    }
    for digest, filenames in hash_groups.items()
    if len(filenames) > 1
]

exact_duplicate_rows = []

for group in exact_duplicate_groups:

    for filename in group["filenames"]:

        exact_duplicate_rows.append({
            "sha256": group["sha256"],
            "filename": filename,
            "group_size": group["count"]
        })

exact_hash_df = pd.DataFrame(
    exact_duplicate_rows
)

exact_hash_df.to_csv(
    DUPLICATE_HASH_CSV,
    index=False
)

print(
    "Exact duplicate groups:",
    len(exact_duplicate_groups)
)

if exact_duplicate_groups:
    display(pd.DataFrame(exact_duplicate_groups))
else:
    print("No byte-for-byte duplicate STEP files found.")


# ------------------------------------------------
# SECTION 8: GEOMETRIC DUPLICATE SCREENING
# ------------------------------------------------

def relative_difference(a, b):
    """
    Calculate symmetric relative difference.

    The denominator is the larger absolute magnitude,
    avoiding division by zero for small values.
    """

    denominator = max(
        abs(float(a)),
        abs(float(b)),
        1e-12
    )

    return abs(float(a) - float(b)) / denominator


def distribution_difference(a, b):
    """
    Compare two sorted arrays of face areas or edge lengths.

    Arrays must have the same number of elements.
    Returns the maximum symmetric relative difference.
    """

    if len(a) != len(b):
        return np.inf

    if len(a) == 0:
        return 0.0

    differences = [
        relative_difference(x, y)
        for x, y in zip(a, b)
    ]

    return float(max(differences))


# Conservative screening thresholds.
# These identify close candidates, not confirmed duplicates.
VOLUME_TOL = 1e-4
AREA_TOL = 1e-4
BBOX_TOL = 1e-4

# The topology counts must match exactly.
# Face-area and edge-length distributions are additional evidence.
DISTRIBUTION_TOL = 1e-3

duplicate_candidates = []

for first, second in combinations(model_records, 2):

    # ------------------------------------------------
    # A. Topological count agreement
    # ------------------------------------------------

    same_topology_counts = (
        first["faces"] == second["faces"] and
        first["edges"] == second["edges"] and
        first["vertices"] == second["vertices"]
    )

    if not same_topology_counts:
        continue

    # ------------------------------------------------
    # B. Volume and surface-area agreement
    # ------------------------------------------------

    volume_diff = relative_difference(
        first["independent_volume"],
        second["independent_volume"]
    )

    area_diff = relative_difference(
        first["independent_surface_area"],
        second["independent_surface_area"]
    )

    # ------------------------------------------------
    # C. Compare sorted bounding-box dimensions.
    #
    # Sorting makes this comparison independent of
    # the orientation of the model's coordinate axes.
    # ------------------------------------------------

    first_dims = np.sort(
        first["_tight_dimensions"]
    )

    second_dims = np.sort(
        second["_tight_dimensions"]
    )

    bbox_diffs = [
        relative_difference(a, b)
        for a, b in zip(first_dims, second_dims)
    ]

    bbox_diff = float(max(bbox_diffs))

    # ------------------------------------------------
    # D. Compare sorted face areas and edge lengths.
    # ------------------------------------------------

    face_distribution_diff = distribution_difference(
        first["_face_areas"],
        second["_face_areas"]
    )

    edge_distribution_diff = distribution_difference(
        first["_edge_lengths"],
        second["_edge_lengths"]
    )

    # ------------------------------------------------
    # E. Conservative candidate decision
    # ------------------------------------------------

    candidate = (
        volume_diff <= VOLUME_TOL and
        area_diff <= AREA_TOL and
        bbox_diff <= BBOX_TOL
    )

    if candidate:

        duplicate_candidates.append({

            "filename_1": first["filename"],
            "filename_2": second["filename"],

            "same_topology_counts": True,

            "volume_relative_difference": volume_diff,
            "area_relative_difference": area_diff,
            "bbox_relative_difference": bbox_diff,

            "face_area_distribution_difference":
                face_distribution_diff,

            "edge_length_distribution_difference":
                edge_distribution_diff,

            "face_distribution_close":
                face_distribution_diff <= DISTRIBUTION_TOL,

            "edge_distribution_close":
                edge_distribution_diff <= DISTRIBUTION_TOL,

            "exact_file_match":
                first["sha256"] == second["sha256"],

            "assessment": (
                "HIGH_SIMILARITY_CANDIDATE"
                if (
                    face_distribution_diff <= DISTRIBUTION_TOL and
                    edge_distribution_diff <= DISTRIBUTION_TOL
                )
                else "GEOMETRICALLY_SIMILAR_CANDIDATE"
            )
        })


# ------------------------------------------------
# SECTION 9: SAVE DUPLICATE ASSESSMENT
# ------------------------------------------------

duplicate_df = pd.DataFrame(
    duplicate_candidates
)

duplicate_df.to_csv(
    DUPLICATE_CSV,
    index=False
)

print("\n" + "=" * 75)
print("GEOMETRIC DUPLICATE SCREENING")
print("=" * 75)

print(
    "Candidate pairs found:",
    len(duplicate_df)
)

if not duplicate_df.empty:

    display(
        duplicate_df.sort_values(
            "volume_relative_difference"
        )
    )

else:

    print(
        "No pairs met the conservative numerical screening criteria."
    )


# ------------------------------------------------
# SECTION 10: CHECK THE THREE PREVIOUSLY SUSPECTED MODELS
# ------------------------------------------------

suspected_prefixes = [
    "00000011_e909f",
    "00000012_f1688",
    "00000207_6edad"
]

print("\n" + "=" * 75)
print("PREVIOUSLY SUSPECTED MODEL PAIRS")
print("=" * 75)

suspected_names = {
    record["filename"]: record
    for record in model_records
    if any(
        record["filename"].startswith(prefix)
        for prefix in suspected_prefixes
    )
}

for name in suspected_names:
    print("Found:", name)

suspected_pairs = []

for first, second in combinations(
    list(suspected_names.values()),
    2
):

    suspected_pairs.append({
        "filename_1": first["filename"],
        "filename_2": second["filename"],

        "volume_relative_difference":
            relative_difference(
                first["independent_volume"],
                second["independent_volume"]
            ),

        "area_relative_difference":
            relative_difference(
                first["independent_surface_area"],
                second["independent_surface_area"]
            ),

        "tight_bbox_dimension_difference":
            float(max(
                relative_difference(a, b)
                for a, b in zip(
                    np.sort(first["_tight_dimensions"]),
                    np.sort(second["_tight_dimensions"])
                )
            )),

        "same_face_count":
            first["faces"] == second["faces"],

        "same_edge_count":
            first["edges"] == second["edges"],

        "same_vertex_count":
            first["vertices"] == second["vertices"],

        "same_face_area_distribution":
            distribution_difference(
                first["_face_areas"],
                second["_face_areas"]
            ) <= DISTRIBUTION_TOL,

        "same_edge_length_distribution":
            distribution_difference(
                first["_edge_lengths"],
                second["_edge_lengths"]
            ) <= DISTRIBUTION_TOL,

        "exact_file_match":
            first["sha256"] == second["sha256"]
    })

suspected_df = pd.DataFrame(suspected_pairs)

if not suspected_df.empty:
    display(suspected_df)
else:
    print("Insufficient suspected models were successfully inspected.")


# ------------------------------------------------
# SECTION 11: SAVE FINAL SUMMARY
# ------------------------------------------------

summary = {

    "models_selected": int(len(dataset)),

    "models_inspected": int(len(model_records)),

    "inspection_errors": errors,

    "exact_duplicate_groups": int(
        len(exact_duplicate_groups)
    ),

    "geometric_candidate_pairs": int(
        len(duplicate_candidates)
    ),

    "bbox_inflation": (
        bbox_df["bbox_inflation_percent"].describe().to_dict()
        if not bbox_df.empty else {}
    ),

    "diagonal_inflation": (
        bbox_df["diagonal_inflation_percent"].describe().to_dict()
        if not bbox_df.empty else {}
    ),

    "normalized_target_change_if_tight": (
        bbox_df["target_change_percent_if_tight"].describe().to_dict()
        if not bbox_df.empty else {}
    ),

    "original_bbox_method": "BRepBndLib.Add_s",

    "comparison_bbox_method": "BRepBndLib.AddOptimal_s",

    "original_graphs_modified": False,

    "master_dataset_modified": False,

    "step_files_modified": False,

    "bbox_report": str(BBOX_CSV),

    "duplicate_report": str(DUPLICATE_CSV),

    "exact_hash_report": str(DUPLICATE_HASH_CSV)
}

with open(SUMMARY_JSON, "w") as file:

    json.dump(
        summary,
        file,
        indent=4,
        default=str
    )

print("\n" + "=" * 75)
print("FINAL AUDIT FILES")
print("=" * 75)

print("Bounding-box report:", BBOX_CSV)
print("Duplicate candidates:", DUPLICATE_CSV)
print("Exact hash report:", DUPLICATE_HASH_CSV)
print("Summary:", SUMMARY_JSON)

print("\nNo original graph, dataset or STEP file was modified.")


Models to audit: 100
Output directory: /content/drive/MyDrive/CAD_GNN_Project/audit_reports

STARTING BOUNDING-BOX AND GEOMETRY AUDIT

[1/100] 00000000_290a9120f9f249a7a05cfe9c_step_000.step
PASS | Faces: 25 | Edges: 58 | Vertices: 39 | BBox inflation: 64.078 %

[2/100] 00000001_1ffb81a71e5b402e966b9341_step_000.step
PASS | Faces: 103 | Edges: 256 | Vertices: 160 | BBox inflation: 24.322 %

[3/100] 00000005_d4fe04f0f5f84b52bd4f10e4_step_000.step
PASS | Faces: 60 | Edges: 120 | Vertices: 80 | BBox inflation: 0.0 %

[4/100] 00000007_b33a147f86da49879455d286_step_000.step
PASS | Faces: 3 | Edges: 3 | Vertices: 2 | BBox inflation: 0.0 %

[5/100] 00000008_9b3d6a97e8de4aa193b81000_step_000.step
PASS | Faces: 21 | Edges: 41 | Vertices: 27 | BBox inflation: 0.0 %

[6/100] 00000010_b4b99d35e04b4277931f9a9c_step_000.step
PASS | Faces: 75 | Edges: 179 | Vertices: 111 | BBox inflation: 107.114 %

[7/100] 00000011_e909f412cda24521865fac0f_step_000.step
PASS | Faces: 253 | Edges: 666 | Vertices: 394

,filename,sha256,faces,edges,vertices,independent_volume,stored_volume,independent_surface_area,stored_surface_area,loose_x,...,loose_diagonal,tight_diagonal,stored_diagonal,loose_bbox_volume,tight_bbox_volume,bbox_inflation_percent,diagonal_inflation_percent,original_normalized_volume,tight_normalized_volume_diagnostic,target_change_percent_if_tight
7,00000012_f16882934f314832b639ffc0_step_000.step,29da96fd0b52e262036807b162b8c5d02cb74ab640dbef...,253,666,394,3.797250e+05,3.797250e+05,72304.348705,72304.348705,455.338953,...,537.794823,223.193526,537.794823,1.826717e+07,1.909599e+06,856.597098,140.954490,0.002441,0.034153,1298.959272
98,00000207_6edad88f1b7749c498b67f86_step_000.step,559d3febac984cbf990fb2514871345dcc0c341d0d8d6e...,253,666,394,3.797250e+05,3.797250e+05,72304.348374,72304.348374,455.338953,...,537.794823,223.193526,537.794823,1.826717e+07,1.909599e+06,856.597098,140.954490,0.002441,0.034153,1298.959272
6,00000011_e909f412cda24521865fac0f_step_000.step,490701d7fca1db71dc912bc6e1d7a3a1ef1792aeb030cf...,253,666,394,3.797250e+05,3.797250e+05,72304.348132,72304.348132,455.338953,...,537.794823,223.193526,537.794823,1.826717e+07,1.909599e+06,856.597098,140.954490,0.002441,0.034153,1298.959272
75,00000234_38a16b120d49465b86d5bd82_step_000.step,61c7885a8467c904e701e7bd0f63e0849319da523356be...,1391,3493,2087,3.913692e+05,3.913692e+05,141638.251390,141638.251390,836.697151,...,878.320643,258.756839,878.320643,2.763882e+07,3.212452e+06,760.365188,239.438619,0.000578,0.022590,3810.963442
74,00000206_27417003e762409185cb2b69_step_000.step,e40e2fbf5c9fe2f9ad8c24d2ed176c928cfe73f63b3c56...,51,104,56,3.267682e+04,3.267682e+04,21573.485934,21573.485934,158.750000,...,230.182026,116.570504,230.182026,1.280239e+06,2.580963e+05,396.031746,97.461637,0.002679,0.020629,669.924601
5,00000010_b4b99d35e04b4277931f9a9c_step_000.step,79a1916b709d95e5aa5b79694f9d988bcd97c1c1c2e9ab...,75,179,111,4.683109e+04,4.683109e+04,51604.104195,51604.104195,103.162077,...,234.459836,172.295896,234.459836,2.019162e+06,9.749022e+05,107.114333,36.079756,0.003634,0.009156,151.988411
9,00000014_5b1c2f8a8c6f40fdaae1e69d_step_000.step,7c7190f868acd93951fd6302313d1f818828f44da6a571...,214,498,307,6.963039e+03,6.963039e+03,11872.777968,11872.777968,14.680394,...,152.846557,150.413169,152.846557,4.160246e+04,2.346099e+04,77.326093,1.617803,0.001950,0.002046,4.932350
97,00000100_5ed74bccca6f4e89829bcb5e_step_000.step,709c695d000203525a2ba7fdf39414356e81eac9d06526...,227,580,366,1.792154e+04,1.792154e+04,16474.978245,16474.978245,63.253312,...,182.623044,181.019775,182.623044,3.017942e+05,1.833703e+05,64.581796,0.885687,0.002942,0.003021,2.680664
0,00000000_290a9120f9f249a7a05cfe9c_step_000.step,73411ebbcc0afa5fa742b331ce1be4450b34c9b367b75a...,25,58,39,3.166280e+06,3.166280e+06,404864.616471,404864.616471,220.985593,...,660.340231,610.399631,660.340231,3.741135e+07,2.280095e+07,64.078019,8.181624,0.010996,0.013922,26.607806
10,00000015_b08aa818955948c690fd9b6d_step_000.step,4460d2d68643db57c164e1ede21817a037fcb9881a9749...,42,98,55,4.216787e+05,4.216787e+05,79743.332431,79743.332431,216.246451,...,280.508191,258.128206,280.508191,3.376936e+06,2.474881e+06,36.448412,8.670104,0.019105,0.024517,28.330608



Largest normalization-impact cases:


,filename,sha256,faces,edges,vertices,independent_volume,stored_volume,independent_surface_area,stored_surface_area,loose_x,...,loose_diagonal,tight_diagonal,stored_diagonal,loose_bbox_volume,tight_bbox_volume,bbox_inflation_percent,diagonal_inflation_percent,original_normalized_volume,tight_normalized_volume_diagnostic,target_change_percent_if_tight
75,00000234_38a16b120d49465b86d5bd82_step_000.step,61c7885a8467c904e701e7bd0f63e0849319da523356be...,1391,3493,2087,3.913692e+05,3.913692e+05,141638.251390,141638.251390,836.697151,...,878.320643,258.756839,878.320643,2.763882e+07,3.212452e+06,760.365188,239.438619,0.000578,0.022590,3810.963442
7,00000012_f16882934f314832b639ffc0_step_000.step,29da96fd0b52e262036807b162b8c5d02cb74ab640dbef...,253,666,394,3.797250e+05,3.797250e+05,72304.348705,72304.348705,455.338953,...,537.794823,223.193526,537.794823,1.826717e+07,1.909599e+06,856.597098,140.954490,0.002441,0.034153,1298.959272
98,00000207_6edad88f1b7749c498b67f86_step_000.step,559d3febac984cbf990fb2514871345dcc0c341d0d8d6e...,253,666,394,3.797250e+05,3.797250e+05,72304.348374,72304.348374,455.338953,...,537.794823,223.193526,537.794823,1.826717e+07,1.909599e+06,856.597098,140.954490,0.002441,0.034153,1298.959272
6,00000011_e909f412cda24521865fac0f_step_000.step,490701d7fca1db71dc912bc6e1d7a3a1ef1792aeb030cf...,253,666,394,3.797250e+05,3.797250e+05,72304.348132,72304.348132,455.338953,...,537.794823,223.193526,537.794823,1.826717e+07,1.909599e+06,856.597098,140.954490,0.002441,0.034153,1298.959272
74,00000206_27417003e762409185cb2b69_step_000.step,e40e2fbf5c9fe2f9ad8c24d2ed176c928cfe73f63b3c56...,51,104,56,3.267682e+04,3.267682e+04,21573.485934,21573.485934,158.750000,...,230.182026,116.570504,230.182026,1.280239e+06,2.580963e+05,396.031746,97.461637,0.002679,0.020629,669.924601
5,00000010_b4b99d35e04b4277931f9a9c_step_000.step,79a1916b709d95e5aa5b79694f9d988bcd97c1c1c2e9ab...,75,179,111,4.683109e+04,4.683109e+04,51604.104195,51604.104195,103.162077,...,234.459836,172.295896,234.459836,2.019162e+06,9.749022e+05,107.114333,36.079756,0.003634,0.009156,151.988411
10,00000015_b08aa818955948c690fd9b6d_step_000.step,4460d2d68643db57c164e1ede21817a037fcb9881a9749...,42,98,55,4.216787e+05,4.216787e+05,79743.332431,79743.332431,216.246451,...,280.508191,258.128206,280.508191,3.376936e+06,2.474881e+06,36.448412,8.670104,0.019105,0.024517,28.330608
68,00000147_d9a2aa6d24764b809c265460_step_000.step,89334cfb2d9a5819be14f82a26dda4bf2a0ed6ce40e153...,717,1717,1017,8.964638e+04,8.964638e+04,50509.520714,50509.520714,174.215416,...,223.180257,205.692433,223.180257,1.619290e+06,1.198812e+06,35.074480,8.501929,0.008064,0.010301,27.735724
0,00000000_290a9120f9f249a7a05cfe9c_step_000.step,73411ebbcc0afa5fa742b331ce1be4450b34c9b367b75a...,25,58,39,3.166280e+06,3.166280e+06,404864.616471,404864.616471,220.985593,...,660.340231,610.399631,660.340231,3.741135e+07,2.280095e+07,64.078019,8.181624,0.010996,0.013922,26.607806
55,00000098_72cf0f9ca5324822a4d41a80_step_000.step,d6b18a044faf965acdace83eb70fd042235776173dfe41...,137,363,240,9.758674e+02,9.758674e+02,799.348677,799.348677,21.648907,...,30.783066,28.464715,30.783066,1.500259e+03,1.280000e+03,17.207711,8.144648,0.033455,0.042313,26.478031



EXACT FILE DUPLICATE CHECK
Exact duplicate groups: 0
No byte-for-byte duplicate STEP files found.

GEOMETRIC DUPLICATE SCREENING
Candidate pairs found: 3


,filename_1,filename_2,same_topology_counts,volume_relative_difference,area_relative_difference,bbox_relative_difference,face_area_distribution_difference,edge_length_distribution_difference,face_distribution_close,edge_distribution_close,exact_file_match,assessment
2,00000012_f16882934f314832b639ffc0_step_000.step,00000207_6edad88f1b7749c498b67f86_step_000.step,True,1.084650e-08,4.574039e-09,0.000000e+00,0.000001,1.811743e-01,True,False,False,GEOMETRICALLY_SIMILAR_CANDIDATE
1,00000011_e909f412cda24521865fac0f_step_000.step,00000207_6edad88f1b7749c498b67f86_step_000.step,True,1.802207e-08,3.349772e-09,1.276483e-14,0.000002,1.811743e-01,True,False,False,GEOMETRICALLY_SIMILAR_CANDIDATE
0,00000011_e909f412cda24521865fac0f_step_000.step,00000012_f16882934f314832b639ffc0_step_000.step,True,2.886858e-08,7.923810e-09,1.276483e-14,0.000002,4.093072e-09,True,True,False,HIGH_SIMILARITY_CANDIDATE



PREVIOUSLY SUSPECTED MODEL PAIRS
Found: 00000011_e909f412cda24521865fac0f_step_000.step
Found: 00000012_f16882934f314832b639ffc0_step_000.step
Found: 00000207_6edad88f1b7749c498b67f86_step_000.step


,filename_1,filename_2,volume_relative_difference,area_relative_difference,tight_bbox_dimension_difference,same_face_count,same_edge_count,same_vertex_count,same_face_area_distribution,same_edge_length_distribution,exact_file_match
0,00000011_e909f412cda24521865fac0f_step_000.step,00000012_f16882934f314832b639ffc0_step_000.step,2.886858e-08,7.923810e-09,1.276483e-14,True,True,True,True,True,False
1,00000011_e909f412cda24521865fac0f_step_000.step,00000207_6edad88f1b7749c498b67f86_step_000.step,1.802207e-08,3.349772e-09,1.276483e-14,True,True,True,True,False,False
2,00000012_f16882934f314832b639ffc0_step_000.step,00000207_6edad88f1b7749c498b67f86_step_000.step,1.084650e-08,4.574039e-09,0.000000e+00,True,True,True,True,False,False



FINAL AUDIT FILES
Bounding-box report: /content/drive/MyDrive/CAD_GNN_Project/audit_reports/bbox_comparison_100.csv
Duplicate candidates: /content/drive/MyDrive/CAD_GNN_Project/audit_reports/duplicate_assessment.csv
Exact hash report: /content/drive/MyDrive/CAD_GNN_Project/audit_reports/exact_duplicate_hashes.csv
Summary: /content/drive/MyDrive/CAD_GNN_Project/audit_reports/bbox_duplicate_summary.json

No original graph, dataset or STEP file was modified.


In [16]:
#14
# ============================================================
# STEP 1: CREATE A SHARED TRAIN / VALIDATION / TEST SPLIT
# ============================================================
# Purpose:
#   1. Load the existing 100-model dataset.
#   2. Create a reproducible 70/15/15 split.
#   3. Keep the highly similar models 00000011 and 00000012
#      in the same partition.
#   4. Save the split to Google Drive.
#
# IMPORTANT:
#   - Does NOT modify any existing dataset or graph file.
#   - Uses model IDs (filenames) to prevent representation leakage.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. DEFINE PROJECT PATHS
# ------------------------------------------------------------

PROJECT = Path("/content/drive/MyDrive/CAD_GNN_Project")

DATASET_PATH = PROJECT / "dataset_100.csv"
OUTPUT_PATH = PROJECT / "audit_reports" / "shared_split_100.csv"

# ------------------------------------------------------------
# 2. LOAD THE MASTER 100-MODEL DATASET
# ------------------------------------------------------------

df = pd.read_csv(DATASET_PATH)

# The filename uniquely identifies each CAD model.
model_ids = df["filename"].astype(str).tolist()

assert len(model_ids) == 100, "Expected exactly 100 models."
assert len(set(model_ids)) == 100, "Duplicate model IDs detected."

# ------------------------------------------------------------
# 3. CREATE GROUPS
# ------------------------------------------------------------

# These two models are highly similar and must not be separated.
similar_pair = {
    "00000011_e909f",
    "00000012_f1688"
}

# Match the pair against the actual filenames.
pair_ids = [
    model_id for model_id in model_ids
    if any(model_id.startswith(prefix) for prefix in similar_pair)
]

assert len(pair_ids) == 2, (
    f"Expected to find both similar models, found: {pair_ids}"
)

# Every model initially forms its own group.
groups = [[model_id] for model_id in model_ids]

# Replace the two individual groups with one combined group.
groups = [
    group for group in groups
    if not any(model_id in pair_ids for model_id in group)
]

groups.append(pair_ids)

# ------------------------------------------------------------
# 4. REPRODUCIBLE GROUP SHUFFLING
# ------------------------------------------------------------

SEED = 42
rng = np.random.default_rng(SEED)

rng.shuffle(groups)

# ------------------------------------------------------------
# 5. ASSIGN GROUPS TO PARTITIONS
# ------------------------------------------------------------

# Target sizes: 70 training, 15 validation, 15 testing.
targets = {
    "train": 70,
    "validation": 15,
    "test": 15
}

partitions = {
    "train": [],
    "validation": [],
    "test": []
}

# Assign whole groups while approaching the target sizes.
for group in groups:

    group_size = len(group)

    # Select the partition with the largest remaining capacity.
    available = [
        name for name in targets
        if len(partitions[name]) + group_size <= targets[name]
    ]

    if not available:
        raise RuntimeError("Could not allocate all model groups.")

    selected = max(
        available,
        key=lambda name: targets[name] - len(partitions[name])
    )

    partitions[selected].extend(group)

# ------------------------------------------------------------
# 6. CREATE SPLIT TABLE
# ------------------------------------------------------------

split_map = {}

for split_name, ids in partitions.items():
    for model_id in ids:
        split_map[model_id] = split_name

split_df = pd.DataFrame({
    "filename": model_ids,
    "split": [split_map[model_id] for model_id in model_ids]
})

# ------------------------------------------------------------
# 7. FINAL BASIC CHECKS
# ------------------------------------------------------------

assert len(split_df) == 100
assert split_df["filename"].is_unique
assert split_df["split"].notna().all()

counts = split_df["split"].value_counts()

assert counts.get("train", 0) == 70
assert counts.get("validation", 0) == 15
assert counts.get("test", 0) == 15

# Confirm the similar pair is in the same partition.
pair_splits = split_df[
    split_df["filename"].isin(pair_ids)
]["split"].unique()

assert len(pair_splits) == 1

# ------------------------------------------------------------
# 8. SAVE WITHOUT OVERWRITING EXISTING FILES
# ------------------------------------------------------------

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

assert not OUTPUT_PATH.exists(), (
    "Split file already exists. Do not overwrite it."
)

split_df.to_csv(OUTPUT_PATH, index=False)

# ------------------------------------------------------------
# 9. DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 55)
print("SHARED DATASET SPLIT CREATED")
print("=" * 55)

print("\nPartition counts:")
print(split_df["split"].value_counts())

print("\nSimilar models:")
print(pair_ids)
print("Assigned partition:", pair_splits[0])

print("\nSaved to:")
print(OUTPUT_PATH)

print("\nAll checks passed.")


SHARED DATASET SPLIT CREATED

Partition counts:
split
train         70
test          15
validation    15
Name: count, dtype: int64

Similar models:
['00000011_e909f412cda24521865fac0f_step_000.step', '00000012_f16882934f314832b639ffc0_step_000.step']
Assigned partition: train

Saved to:
/content/drive/MyDrive/CAD_GNN_Project/audit_reports/shared_split_100.csv

All checks passed.


In [17]:
#15

# ============================================================
# STEP 2A: LOAD AND PARTITION ALL FOUR GRAPH REPRESENTATIONS
# ============================================================
# Each representation will be trained independently.
# They share only the model IDs assigned to each data split.
#
# No existing files will be modified.
# ============================================================

from pathlib import Path
import torch
import pandas as pd

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

PROJECT = Path("/content/drive/MyDrive/CAD_GNN_Project")
GRAPH_DIR = PROJECT / "graphs"
SPLIT_PATH = PROJECT / "audit_reports" / "shared_split_100.csv"

# ------------------------------------------------------------
# 2. LOAD THE FIXED SPLIT
# ------------------------------------------------------------

split_df = pd.read_csv(SPLIT_PATH)

# Convert the CSV into a dictionary:
# model filename -> train / validation / test
split_map = dict(zip(split_df["filename"], split_df["split"]))

assert len(split_map) == 100
assert set(split_map.values()) == {"train", "validation", "test"}

# ------------------------------------------------------------
# 3. LOAD THE FOUR EXISTING GRAPH DATASETS
# ------------------------------------------------------------

graph_files = {
    "face": GRAPH_DIR / "face_graphs_100.pt",
    "edge": GRAPH_DIR / "edge_graphs_100.pt",
    "vertex": GRAPH_DIR / "vertex_graphs_100.pt",
    "hetero": GRAPH_DIR / "hetero_graphs_100.pt"
}

graph_datasets = {}

for representation, path in graph_files.items():

    # Load the saved graph dictionary into memory.
    graph_datasets[representation] = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    print(
        f"{representation.upper():8s}: "
        f"{len(graph_datasets[representation])} graphs loaded"
    )

# ------------------------------------------------------------
# 4. CHECK THAT ALL REPRESENTATIONS HAVE THE SAME MODEL IDs
# ------------------------------------------------------------

reference_ids = set(graph_datasets["face"].keys())

for representation, dataset in graph_datasets.items():

    current_ids = set(dataset.keys())

    assert current_ids == reference_ids, (
        f"Model ID mismatch in {representation} dataset."
    )

    assert current_ids == set(split_map.keys()), (
        f"Split mismatch in {representation} dataset."
    )

print("\nAll four graph datasets contain the same 100 models.")

# ------------------------------------------------------------
# 5. PARTITION EACH REPRESENTATION
# ------------------------------------------------------------

# Structure:
#
# graph_splits["face"]["train"]
# graph_splits["face"]["validation"]
# graph_splits["face"]["test"]
#
# The same structure exists for edge, vertex and hetero.

graph_splits = {}

for representation, dataset in graph_datasets.items():

    graph_splits[representation] = {
        "train": {},
        "validation": {},
        "test": {}
    }

    for model_id, graph in dataset.items():

        partition = split_map[model_id]

        graph_splits[representation][partition][model_id] = graph

# ------------------------------------------------------------
# 6. VERIFY PARTITION COUNTS
# ------------------------------------------------------------

expected_counts = {
    "train": 70,
    "validation": 15,
    "test": 15
}

print("\n" + "=" * 55)
print("GRAPH PARTITION SUMMARY")
print("=" * 55)

for representation, partitions in graph_splits.items():

    print(f"\n{representation.upper()} GNN:")

    for partition, graphs in partitions.items():

        count = len(graphs)

        print(f"  {partition:12s}: {count}")

        assert count == expected_counts[partition]

# ------------------------------------------------------------
# 7. VERIFY TARGET AVAILABILITY
# ------------------------------------------------------------

for representation, partitions in graph_splits.items():

    for partition, graphs in partitions.items():

        for model_id, graph in graphs.items():

            # Every graph must have a stored normalized target.
            assert hasattr(graph, "y"), (
                f"Missing target: {representation}, {model_id}"
            )

            assert graph.y.numel() == 1, (
                f"Unexpected target shape: {representation}, {model_id}"
            )

# ------------------------------------------------------------
# 8. FINAL RESULT
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("DATA LOADING AND PARTITIONING COMPLETE")
print("=" * 55)

print("Face graphs:       70 / 15 / 15")
print("Edge graphs:       70 / 15 / 15")
print("Vertex graphs:     70 / 15 / 15")
print("Heterogeneous:     70 / 15 / 15")

print("\nAll four representations use the same model split.")
print("Original files remain unchanged.")


FACE    : 100 graphs loaded
EDGE    : 100 graphs loaded
VERTEX  : 100 graphs loaded
HETERO  : 100 graphs loaded

All four graph datasets contain the same 100 models.

GRAPH PARTITION SUMMARY

FACE GNN:
  train       : 70
  validation  : 15
  test        : 15

EDGE GNN:
  train       : 70
  validation  : 15
  test        : 15

VERTEX GNN:
  train       : 70
  validation  : 15
  test        : 15

HETERO GNN:
  train       : 70
  validation  : 15
  test        : 15

DATA LOADING AND PARTITIONING COMPLETE
Face graphs:       70 / 15 / 15
Edge graphs:       70 / 15 / 15
Vertex graphs:     70 / 15 / 15
Heterogeneous:     70 / 15 / 15

All four representations use the same model split.
Original files remain unchanged.


In [18]:
#16
# ============================================================
# GCN 1: FACE ADJACENCY GRAPH
# ============================================================
# Input:
#   Face nodes (14 features)
#   Face adjacency connections
#
# Output:
#   Predicted normalized volume (V / D^3)
#
# This model is completely independent of the other three.
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import GCNConv
from pathlib import Path

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

FACE_SEED = 42

FACE_INPUT_DIM = 14
FACE_HIDDEN_DIM = 64
FACE_OUTPUT_DIM = 1

FACE_LEARNING_RATE = 0.001
FACE_DROPOUT = 0.2

FACE_EPOCHS = 300
FACE_PATIENCE = 30

FACE_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

torch.manual_seed(FACE_SEED)

# ------------------------------------------------------------
# 2. DEFINE FACE GCN
# ------------------------------------------------------------

class FaceGCN(nn.Module):

    def __init__(self):

        super().__init__()

        # Three graph convolution layers.
        self.conv1 = GCNConv(FACE_INPUT_DIM, FACE_HIDDEN_DIM)
        self.conv2 = GCNConv(FACE_HIDDEN_DIM, FACE_HIDDEN_DIM)
        self.conv3 = GCNConv(FACE_HIDDEN_DIM, FACE_HIDDEN_DIM)

        # Regression head.
        self.regressor = nn.Sequential(
            nn.Linear(FACE_HIDDEN_DIM, 32),
            nn.ReLU(),
            nn.Linear(32, FACE_OUTPUT_DIM)
        )

    def forward(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(
            x,
            p=FACE_DROPOUT,
            training=self.training
        )

        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(
            x,
            p=FACE_DROPOUT,
            training=self.training
        )

        x = self.conv3(x, edge_index)
        x = F.relu(x)

        # One graph is processed at a time.
        graph_embedding = x.mean(dim=0, keepdim=True)

        return self.regressor(graph_embedding)


# ------------------------------------------------------------
# 3. INITIALIZE INDEPENDENT FACE MODEL AND OPTIMIZER
# ------------------------------------------------------------

face_model = FaceGCN().to(FACE_DEVICE)

face_optimizer = torch.optim.Adam(
    face_model.parameters(),
    lr=FACE_LEARNING_RATE
)

face_criterion = nn.MSELoss()

# ------------------------------------------------------------
# 4. DEFINE FACE GRAPH DATA ACCESS
# ------------------------------------------------------------

face_train_graphs = graph_splits["face"]["train"]
face_validation_graphs = graph_splits["face"]["validation"]
face_test_graphs = graph_splits["face"]["test"]

# ------------------------------------------------------------
# 5. DEFINE ONE-GRAPH TRAINING FUNCTION
# ------------------------------------------------------------

def train_face_epoch():

    face_model.train()

    total_loss = 0.0

    for model_id, graph in face_train_graphs.items():

        face_optimizer.zero_grad()

        x = graph.x.float().to(FACE_DEVICE)
        edge_index = graph.edge_index.long().to(FACE_DEVICE)

        target = graph.y.float().reshape(1, 1).to(FACE_DEVICE)

        prediction = face_model(x, edge_index)

        loss = face_criterion(prediction, target)

        loss.backward()

        face_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(face_train_graphs)


# ------------------------------------------------------------
# 6. DEFINE VALIDATION FUNCTION
# ------------------------------------------------------------

def validate_face():

    face_model.eval()

    total_loss = 0.0

    with torch.no_grad():

        for model_id, graph in face_validation_graphs.items():

            x = graph.x.float().to(FACE_DEVICE)
            edge_index = graph.edge_index.long().to(FACE_DEVICE)

            target = graph.y.float().reshape(1, 1).to(FACE_DEVICE)

            prediction = face_model(x, edge_index)

            loss = face_criterion(prediction, target)

            total_loss += loss.item()

    return total_loss / len(face_validation_graphs)


# ------------------------------------------------------------
# 7. DEFINE FACE CHECKPOINT LOCATION
# ------------------------------------------------------------

FACE_OUTPUT_DIR = (
    Path("/content/drive/MyDrive/CAD_GNN_Project")
    / "training_results"
    / "face_gcn"
)

FACE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FACE_CHECKPOINT = FACE_OUTPUT_DIR / "best_face_gcn.pt"

# ------------------------------------------------------------
# 8. DISPLAY CONFIGURATION
# ------------------------------------------------------------

print("=" * 55)
print("FACE GCN INITIALIZED")
print("=" * 55)

print("Device:", FACE_DEVICE)
print("Training graphs:", len(face_train_graphs))
print("Validation graphs:", len(face_validation_graphs))
print("Test graphs:", len(face_test_graphs))

print("Input dimension:", FACE_INPUT_DIM)
print("Hidden dimension:", FACE_HIDDEN_DIM)
print("GCN layers: 3")
print("Learning rate:", FACE_LEARNING_RATE)
print("Maximum epochs:", FACE_EPOCHS)
print("Early stopping patience:", FACE_PATIENCE)

print("\nModel parameters:",
      sum(p.numel() for p in face_model.parameters()))

print("\nCheckpoint location:", FACE_CHECKPOINT)

print("\nFaceGCN initialized. Training has NOT started.")


FACE GCN INITIALIZED
Device: cpu
Training graphs: 70
Validation graphs: 15
Test graphs: 15
Input dimension: 14
Hidden dimension: 64
GCN layers: 3
Learning rate: 0.001
Maximum epochs: 300
Early stopping patience: 30

Model parameters: 11393

Checkpoint location: /content/drive/MyDrive/CAD_GNN_Project/training_results/face_gcn/best_face_gcn.pt

FaceGCN initialized. Training has NOT started.


In [19]:
#17
# ============================================================
# STEP 2C: TRAIN FACE GCN
# ============================================================
# Uses the independently initialized FaceGCN.
# Training: 70 graphs
# Validation: 15 graphs
# Testing: NOT USED IN THIS CELL
#
# Saves:
#   best_face_gcn.pt
#   face_training_history.csv
# ============================================================

import copy
import time
import numpy as np
import pandas as pd
import torch

# ------------------------------------------------------------
# 1. PREPARE TRAINING SETTINGS
# ------------------------------------------------------------

best_face_val_loss = float("inf")
epochs_without_improvement = 0

face_history = []

start_time = time.time()

# Prevent accidental overwriting of a previous checkpoint.
assert not FACE_CHECKPOINT.exists(), (
    "Face checkpoint already exists. Stop to protect previous work."
)

FACE_HISTORY_PATH = (
    FACE_OUTPUT_DIR / "face_training_history.csv"
)

assert not FACE_HISTORY_PATH.exists(), (
    "Training history already exists. Stop to protect previous work."
)

# ------------------------------------------------------------
# 2. TRAINING LOOP
# ------------------------------------------------------------

for epoch in range(1, FACE_EPOCHS + 1):

    # Train for one complete pass through the 70 training graphs.
    train_loss = train_face_epoch()

    # Evaluate on validation graphs without updating weights.
    val_loss = validate_face()

    # Check that losses are finite.
    if not np.isfinite(train_loss) or not np.isfinite(val_loss):
        raise RuntimeError(
            f"Non-finite loss detected at epoch {epoch}. "
            "Training stopped."
        )

    # Store epoch results.
    face_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_loss": val_loss
    })

    # --------------------------------------------------------
    # 3. CHECK FOR IMPROVEMENT
    # --------------------------------------------------------

    if val_loss < best_face_val_loss:

        best_face_val_loss = val_loss
        epochs_without_improvement = 0

        # Preserve the best model parameters independently.
        best_face_state = copy.deepcopy(face_model.state_dict())

        # Save the best checkpoint to Google Drive.
        torch.save({
            "epoch": epoch,
            "model_state_dict": best_face_state,
            "optimizer_state_dict": face_optimizer.state_dict(),
            "best_validation_loss": best_face_val_loss,
            "seed": FACE_SEED,
            "input_dim": FACE_INPUT_DIM,
            "hidden_dim": FACE_HIDDEN_DIM,
            "num_layers": 3,
            "dropout": FACE_DROPOUT,
            "learning_rate": FACE_LEARNING_RATE
        }, FACE_CHECKPOINT)

    else:

        epochs_without_improvement += 1

    # --------------------------------------------------------
    # 4. DISPLAY PROGRESS
    # --------------------------------------------------------

    if epoch == 1 or epoch % 10 == 0:

        print(
            f"Epoch {epoch:03d}/{FACE_EPOCHS} | "
            f"Train Loss: {train_loss:.8f} | "
            f"Validation Loss: {val_loss:.8f} | "
            f"Best Validation: {best_face_val_loss:.8f}"
        )

    # --------------------------------------------------------
    # 5. EARLY STOPPING
    # --------------------------------------------------------

    if epochs_without_improvement >= FACE_PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch}. "
            f"No validation improvement for {FACE_PATIENCE} epochs."
        )
        break

# ------------------------------------------------------------
# 6. RESTORE THE BEST MODEL
# ------------------------------------------------------------

face_model.load_state_dict(best_face_state)
face_model.eval()

# ------------------------------------------------------------
# 7. SAVE TRAINING HISTORY
# ------------------------------------------------------------

face_history_df = pd.DataFrame(face_history)

face_history_df.to_csv(
    FACE_HISTORY_PATH,
    index=False
)

# ------------------------------------------------------------
# 8. DISPLAY FINAL TRAINING SUMMARY
# ------------------------------------------------------------

elapsed_time = time.time() - start_time

best_epoch = int(
    face_history_df.loc[
        face_history_df["validation_loss"].idxmin(),
        "epoch"
    ]
)

print("\n" + "=" * 60)
print("FACE GCN TRAINING COMPLETED")
print("=" * 60)

print("Epochs completed:", len(face_history_df))
print("Best epoch:", best_epoch)
print("Best validation loss:", best_face_val_loss)
print("Final training loss:", face_history_df.iloc[-1]["train_loss"])
print("Elapsed time (minutes):", round(elapsed_time / 60, 2))

print("\nBest checkpoint:")
print(FACE_CHECKPOINT)

print("\nTraining history:")
print(FACE_HISTORY_PATH)

print("\nBest FaceGCN weights restored.")
print("Test dataset has NOT been used.")


Epoch 001/300 | Train Loss: 0.00558970 | Validation Loss: 0.00035041 | Best Validation: 0.00035041
Epoch 010/300 | Train Loss: 0.00032092 | Validation Loss: 0.00049858 | Best Validation: 0.00032855
Epoch 020/300 | Train Loss: 0.00027671 | Validation Loss: 0.00048039 | Best Validation: 0.00032855
Epoch 030/300 | Train Loss: 0.00025625 | Validation Loss: 0.00055474 | Best Validation: 0.00032855

Early stopping at epoch 34. No validation improvement for 30 epochs.

FACE GCN TRAINING COMPLETED
Epochs completed: 34
Best epoch: 4
Best validation loss: 0.00032855491103115736
Final training loss: 0.0002560825910639965
Elapsed time (minutes): 0.29

Best checkpoint:
/content/drive/MyDrive/CAD_GNN_Project/training_results/face_gcn/best_face_gcn.pt

Training history:
/content/drive/MyDrive/CAD_GNN_Project/training_results/face_gcn/face_training_history.csv

Best FaceGCN weights restored.
Test dataset has NOT been used.


In [20]:
#18
# ============================================================
# STEP 2D: FACE GCN - FINAL TEST EVALUATION
# ============================================================
# Loads the best checkpoint (epoch 4).
# Evaluates the model on the 15 untouched test graphs.
#
# Saves:
#   face_test_predictions.csv
#   face_test_metrics.json
#
# Does NOT train or modify model weights.
# ============================================================

import json
import numpy as np
import pandas as pd
import torch

# ------------------------------------------------------------
# 1. DEFINE OUTPUT PATHS
# ------------------------------------------------------------

FACE_PREDICTIONS_PATH = (
    FACE_OUTPUT_DIR / "face_test_predictions.csv"
)

FACE_METRICS_PATH = (
    FACE_OUTPUT_DIR / "face_test_metrics.json"
)

# Protect existing results from accidental overwriting.
assert not FACE_PREDICTIONS_PATH.exists(), (
    "Test predictions already exist. Stop to protect previous results."
)

assert not FACE_METRICS_PATH.exists(), (
    "Test metrics already exist. Stop to protect previous results."
)

# ------------------------------------------------------------
# 2. LOAD THE BEST CHECKPOINT
# ------------------------------------------------------------

checkpoint = torch.load(
    FACE_CHECKPOINT,
    map_location=FACE_DEVICE,
    weights_only=False
)

face_model.load_state_dict(checkpoint["model_state_dict"])
face_model.eval()

print("Loaded checkpoint from epoch:", checkpoint["epoch"])
print("Best validation MSE:", checkpoint["best_validation_loss"])

# ------------------------------------------------------------
# 3. PREDICT ALL 15 TEST GRAPHS
# ------------------------------------------------------------

prediction_records = []

with torch.no_grad():

    for model_id, graph in face_test_graphs.items():

        # Retrieve face-node features and adjacency connections.
        x = graph.x.float().to(FACE_DEVICE)
        edge_index = graph.edge_index.long().to(FACE_DEVICE)

        # Predict normalized volume.
        predicted_y = face_model(x, edge_index)

        predicted_y = float(predicted_y.item())

        # Retrieve actual normalized volume and physical quantities.
        actual_y = float(graph.y.reshape(-1)[0].item())
        actual_volume = float(graph.volume.reshape(-1)[0].item())
        diagonal = float(graph.diag.reshape(-1)[0].item())

        # Convert predicted normalized volume into physical volume.
        predicted_volume = predicted_y * (diagonal ** 3)

        # Calculate absolute and percentage errors.
        absolute_error = abs(predicted_volume - actual_volume)

        relative_error_percent = (
            absolute_error / actual_volume
        ) * 100.0

        # Store one record for each CAD model.
        prediction_records.append({
            "filename": model_id,
            "actual_normalized_volume": actual_y,
            "predicted_normalized_volume": predicted_y,
            "normalized_absolute_error": abs(predicted_y - actual_y),
            "bounding_box_diagonal": diagonal,
            "actual_volume": actual_volume,
            "predicted_volume": predicted_volume,
            "absolute_volume_error": absolute_error,
            "relative_volume_error_percent": relative_error_percent
        })

# Convert individual predictions into a DataFrame.
face_predictions_df = pd.DataFrame(prediction_records)

# ------------------------------------------------------------
# 4. CALCULATE TEST METRICS
# ------------------------------------------------------------

actual_y = face_predictions_df[
    "actual_normalized_volume"
].to_numpy()

predicted_y = face_predictions_df[
    "predicted_normalized_volume"
].to_numpy()

actual_volume = face_predictions_df[
    "actual_volume"
].to_numpy()

predicted_volume = face_predictions_df[
    "predicted_volume"
].to_numpy()

# Normalized-volume errors.
normalized_errors = predicted_y - actual_y

normalized_mae = float(
    np.mean(np.abs(normalized_errors))
)

normalized_rmse = float(
    np.sqrt(np.mean(normalized_errors ** 2))
)

# R-squared: 1 - residual sum of squares / total sum of squares.
ss_res = float(np.sum(normalized_errors ** 2))
ss_tot = float(np.sum((actual_y - np.mean(actual_y)) ** 2))

normalized_r2 = (
    float(1 - ss_res / ss_tot)
    if ss_tot > 0 else None
)

# Physical-volume errors.
volume_errors = predicted_volume - actual_volume

volume_mae = float(
    np.mean(np.abs(volume_errors))
)

volume_rmse = float(
    np.sqrt(np.mean(volume_errors ** 2))
)

volume_r2 = (
    float(
        1 - np.sum(volume_errors ** 2) /
        np.sum((actual_volume - np.mean(actual_volume)) ** 2)
    )
    if np.sum((actual_volume - np.mean(actual_volume)) ** 2) > 0
    else None
)

mean_relative_error = float(
    face_predictions_df[
        "relative_volume_error_percent"
    ].mean()
)

# ------------------------------------------------------------
# 5. SAVE TEST PREDICTIONS AND METRICS
# ------------------------------------------------------------

face_predictions_df.to_csv(
    FACE_PREDICTIONS_PATH,
    index=False
)

face_test_metrics = {
    "model": "FaceGCN",
    "checkpoint_epoch": int(checkpoint["epoch"]),
    "number_of_test_graphs": len(face_test_graphs),
    "normalized_volume_MAE": normalized_mae,
    "normalized_volume_RMSE": normalized_rmse,
    "normalized_volume_R2": normalized_r2,
    "physical_volume_MAE": volume_mae,
    "physical_volume_RMSE": volume_rmse,
    "physical_volume_R2": volume_r2,
    "mean_relative_volume_error_percent": mean_relative_error
}

with open(FACE_METRICS_PATH, "w") as file:
    json.dump(face_test_metrics, file, indent=4)

# ------------------------------------------------------------
# 6. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FACE GCN - FINAL TEST RESULTS")
print("=" * 60)

print("Test graphs:", len(face_test_graphs))

print("\nNORMALIZED VOLUME METRICS")
print("MAE :", normalized_mae)
print("RMSE:", normalized_rmse)
print("R²  :", normalized_r2)

print("\nPHYSICAL VOLUME METRICS")
print("MAE :", volume_mae)
print("RMSE:", volume_rmse)
print("R²  :", volume_r2)

print(
    "Mean relative volume error (%):",
    mean_relative_error
)

print("\nINDIVIDUAL TEST PREDICTIONS")
display(face_predictions_df)

print("\nSaved files:")
print(FACE_PREDICTIONS_PATH)
print(FACE_METRICS_PATH)

print("\nFaceGCN test evaluation completed.")


Loaded checkpoint from epoch: 4
Best validation MSE: 0.00032855491103115736

FACE GCN - FINAL TEST RESULTS
Test graphs: 15

NORMALIZED VOLUME METRICS
MAE : 0.020159143228859953
RMSE: 0.023578052519877574
R²  : -1.3328002315129162

PHYSICAL VOLUME METRICS
MAE : 18828356.730939887
RMSE: 40702745.69062293
R²  : -33.85645223105041
Mean relative volume error (%): 578.7869194162377

INDIVIDUAL TEST PREDICTIONS


,filename,actual_normalized_volume,predicted_normalized_volume,normalized_absolute_error,bounding_box_diagonal,actual_volume,predicted_volume,absolute_volume_error,relative_volume_error_percent
0,00000013_c2f4d27f35ed4c138caf5c18_step_000.step,0.062463,0.018511,0.043952,59.460583,1.313129e+04,3.891484e+03,9.239804e+03,70.364797
1,00000015_b08aa818955948c690fd9b6d_step_000.step,0.019105,0.020565,0.001460,280.508179,4.216787e+05,4.539071e+05,3.222841e+04,7.642883
2,00000048_efe12a13ac2e4ea7880a66d1_step_000.step,0.002841,0.044406,0.041565,1414.248901,8.036504e+06,1.256087e+08,1.175722e+08,1462.976917
3,00000049_aa208c45e8734584a147f595_step_000.step,0.018208,0.035888,0.017680,61.040611,4.141228e+03,8.162249e+03,4.021022e+03,97.097342
4,00000050_80d90bfdd2e74e709956122a_step_000.step,0.006982,0.020480,0.013498,134.933777,1.715210e+04,5.031409e+04,3.316199e+04,193.340758
5,00000251_66fe5f0992fd4065b4838668_step_000.step,0.000700,-0.002155,0.002855,1501.339722,2.368534e+06,-7.292146e+06,9.660680e+06,407.875998
6,00000248_e87554b09cdc48c3add39db9_step_000.step,0.008626,0.011569,0.002943,9.180809,6.674980e+00,8.952355e+00,2.277375e+00,34.118081
7,00000132_438507a38e404bfdbd51a16b_step_000.step,0.023331,-0.004271,0.027602,22.704969,2.730829e+02,-4.999037e+01,3.230733e+02,118.305928
8,00000083_9ea934d3400e45fb9d190941_step_000.step,0.028270,0.012759,0.015511,417.534363,2.057763e+06,9.287219e+05,1.129041e+06,54.867403
9,00000084_9a051df11afa4fc5a1d667cd_step_000.step,0.005894,0.024036,0.018142,1673.240356,2.761327e+07,1.126000e+08,8.498668e+07,307.774741



Saved files:
/content/drive/MyDrive/CAD_GNN_Project/training_results/face_gcn/face_test_predictions.csv
/content/drive/MyDrive/CAD_GNN_Project/training_results/face_gcn/face_test_metrics.json

FaceGCN test evaluation completed.


In [21]:
#19
# ============================================================
# STEP 3A: INITIALIZE EDGE GCN
# ============================================================
# Uses the existing edge graph dataset.
# Does not modify any original graph files.
# Does not start training.
# ============================================================

import torch
import torch.nn as nn
from torch_geometric.nn import GCNConv
from pathlib import Path

# ------------------------------------------------------------
# 1. REPRODUCIBILITY AND DEVICE
# ------------------------------------------------------------

EDGE_SEED = 42

torch.manual_seed(EDGE_SEED)

EDGE_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------------------
# 2. EXPERIMENTAL CONFIGURATION
# ------------------------------------------------------------

EDGE_INPUT_DIM = 4
EDGE_HIDDEN_DIM = 64
EDGE_DROPOUT = 0.2

EDGE_LEARNING_RATE = 0.001
EDGE_EPOCHS = 300
EDGE_PATIENCE = 30

# ------------------------------------------------------------
# 3. LOAD THE EXISTING EDGE GRAPH SPLITS
# ------------------------------------------------------------

edge_train_graphs = graph_splits["edge"]["train"]
edge_validation_graphs = graph_splits["edge"]["validation"]
edge_test_graphs = graph_splits["edge"]["test"]

assert len(edge_train_graphs) == 70
assert len(edge_validation_graphs) == 15
assert len(edge_test_graphs) == 15

# Confirm the expected node-feature dimension.
for graph in edge_train_graphs.values():
    assert graph.x.shape[1] == EDGE_INPUT_DIM

# ------------------------------------------------------------
# 4. DEFINE THE EDGE GCN ARCHITECTURE
# ------------------------------------------------------------

class EdgeGCN(nn.Module):

    def __init__(self):

        super().__init__()

        # Three graph convolution layers.
        self.conv1 = GCNConv(EDGE_INPUT_DIM, EDGE_HIDDEN_DIM)
        self.conv2 = GCNConv(EDGE_HIDDEN_DIM, EDGE_HIDDEN_DIM)
        self.conv3 = GCNConv(EDGE_HIDDEN_DIM, EDGE_HIDDEN_DIM)

        # Dropout probability.
        self.dropout = EDGE_DROPOUT

        # Regression head: 64 -> 32 -> 1.
        self.regressor = nn.Sequential(
            nn.Linear(EDGE_HIDDEN_DIM, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x, edge_index):

        # First graph convolution.
        x = self.conv1(x, edge_index)
        x = torch.relu(x)
        x = torch.nn.functional.dropout(
            x, p=self.dropout, training=self.training
        )

        # Second graph convolution.
        x = self.conv2(x, edge_index)
        x = torch.relu(x)
        x = torch.nn.functional.dropout(
            x, p=self.dropout, training=self.training
        )

        # Third graph convolution.
        x = self.conv3(x, edge_index)
        x = torch.relu(x)

        # Convert all edge-node embeddings into one graph embedding.
        graph_embedding = x.mean(dim=0, keepdim=True)

        # Predict normalized volume.
        prediction = self.regressor(graph_embedding)

        return prediction

# ------------------------------------------------------------
# 5. INITIALIZE AN INDEPENDENT MODEL
# ------------------------------------------------------------

edge_model = EdgeGCN().to(EDGE_DEVICE)

# Adam optimizer with the same learning rate as FaceGCN.
edge_optimizer = torch.optim.Adam(
    edge_model.parameters(),
    lr=EDGE_LEARNING_RATE
)

# Mean squared error regression loss.
edge_criterion = nn.MSELoss()

# ------------------------------------------------------------
# 6. CREATE A NEW DRIVE OUTPUT DIRECTORY
# ------------------------------------------------------------

EDGE_OUTPUT_DIR = (
    Path("/content/drive/MyDrive/CAD_GNN_Project")
    / "training_results"
    / "edge_gcn"
)

EDGE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EDGE_CHECKPOINT = EDGE_OUTPUT_DIR / "best_edge_gcn.pt"

# Protect existing results.
assert not EDGE_CHECKPOINT.exists(), (
    "EdgeGCN checkpoint already exists. Stop to protect previous work."
)

# ------------------------------------------------------------
# 7. DISPLAY INITIALIZATION SUMMARY
# ------------------------------------------------------------

parameter_count = sum(
    parameter.numel()
    for parameter in edge_model.parameters()
)

print("=" * 55)
print("EDGE GCN INITIALIZED")
print("=" * 55)

print("Device:", EDGE_DEVICE)
print("Training graphs:", len(edge_train_graphs))
print("Validation graphs:", len(edge_validation_graphs))
print("Test graphs:", len(edge_test_graphs))

print("Input dimension:", EDGE_INPUT_DIM)
print("Hidden dimension:", EDGE_HIDDEN_DIM)
print("GCN layers:", 3)
print("Regression head: 64 -> 32 -> 1")
print("Dropout:", EDGE_DROPOUT)
print("Learning rate:", EDGE_LEARNING_RATE)
print("Maximum epochs:", EDGE_EPOCHS)
print("Early stopping patience:", EDGE_PATIENCE)

print("\nModel parameters:", parameter_count)
print("Checkpoint location:", EDGE_CHECKPOINT)

print("\nEdgeGCN initialized. Training has NOT started.")


EDGE GCN INITIALIZED
Device: cpu
Training graphs: 70
Validation graphs: 15
Test graphs: 15
Input dimension: 4
Hidden dimension: 64
GCN layers: 3
Regression head: 64 -> 32 -> 1
Dropout: 0.2
Learning rate: 0.001
Maximum epochs: 300
Early stopping patience: 30

Model parameters: 10753
Checkpoint location: /content/drive/MyDrive/CAD_GNN_Project/training_results/edge_gcn/best_edge_gcn.pt

EdgeGCN initialized. Training has NOT started.


In [22]:
#20
# ============================================================
# STEP 3B: TRAIN EDGE GCN
# ============================================================

import copy
import time
import numpy as np
import pandas as pd
import torch

# ------------------------------------------------------------
# 1. INITIALIZE TRAINING TRACKING
# ------------------------------------------------------------

best_edge_val_loss = float("inf")
epochs_without_improvement = 0

edge_history = []

start_time = time.time()

# Protect previous results from accidental overwriting.
EDGE_HISTORY_PATH = EDGE_OUTPUT_DIR / "edge_training_history.csv"

assert not EDGE_CHECKPOINT.exists(), (
    "Checkpoint already exists. Stop to protect previous results."
)

assert not EDGE_HISTORY_PATH.exists(), (
    "Training history already exists. Stop to protect previous results."
)

# ------------------------------------------------------------
# 2. DEFINE ONE TRAINING EPOCH
# ------------------------------------------------------------

def train_edge_epoch():

    edge_model.train()

    total_loss = 0.0

    # Process each training CAD graph independently.
    for model_id, graph in edge_train_graphs.items():

        edge_optimizer.zero_grad()

        x = graph.x.float().to(EDGE_DEVICE)
        edge_index = graph.edge_index.long().to(EDGE_DEVICE)

        target = graph.y.float().reshape(1, 1).to(EDGE_DEVICE)

        # Forward pass: predict normalized volume.
        prediction = edge_model(x, edge_index)

        # Calculate squared prediction error.
        loss = edge_criterion(prediction, target)

        # Backpropagation: calculate gradients.
        loss.backward()

        # Update EdgeGCN parameters.
        edge_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(edge_train_graphs)


# ------------------------------------------------------------
# 3. DEFINE VALIDATION FUNCTION
# ------------------------------------------------------------

def validate_edge():

    edge_model.eval()

    total_loss = 0.0

    # No gradients or parameter updates during validation.
    with torch.no_grad():

        for model_id, graph in edge_validation_graphs.items():

            x = graph.x.float().to(EDGE_DEVICE)
            edge_index = graph.edge_index.long().to(EDGE_DEVICE)

            target = graph.y.float().reshape(1, 1).to(EDGE_DEVICE)

            prediction = edge_model(x, edge_index)

            loss = edge_criterion(prediction, target)

            total_loss += loss.item()

    return total_loss / len(edge_validation_graphs)


# ------------------------------------------------------------
# 4. TRAINING LOOP WITH EARLY STOPPING
# ------------------------------------------------------------

for epoch in range(1, EDGE_EPOCHS + 1):

    train_loss = train_edge_epoch()

    val_loss = validate_edge()

    # Stop if numerical instability produces invalid losses.
    if not np.isfinite(train_loss) or not np.isfinite(val_loss):
        raise RuntimeError(
            f"Non-finite loss detected at epoch {epoch}."
        )

    edge_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_loss": val_loss
    })

    # --------------------------------------------------------
    # SAVE CHECKPOINT WHEN VALIDATION IMPROVES
    # --------------------------------------------------------

    if val_loss < best_edge_val_loss:

        best_edge_val_loss = val_loss
        epochs_without_improvement = 0

        best_edge_state = copy.deepcopy(
            edge_model.state_dict()
        )

        torch.save({
            "epoch": epoch,
            "model_state_dict": best_edge_state,
            "optimizer_state_dict": edge_optimizer.state_dict(),
            "best_validation_loss": best_edge_val_loss,
            "seed": EDGE_SEED,
            "input_dim": EDGE_INPUT_DIM,
            "hidden_dim": EDGE_HIDDEN_DIM,
            "num_layers": 3,
            "dropout": EDGE_DROPOUT,
            "learning_rate": EDGE_LEARNING_RATE
        }, EDGE_CHECKPOINT)

    else:

        epochs_without_improvement += 1

    # Display progress every ten epochs and at the beginning.
    if epoch == 1 or epoch % 10 == 0:

        print(
            f"Epoch {epoch:03d}/{EDGE_EPOCHS} | "
            f"Train Loss: {train_loss:.8f} | "
            f"Validation Loss: {val_loss:.8f} | "
            f"Best Validation: {best_edge_val_loss:.8f}"
        )

    # Stop after the configured patience.
    if epochs_without_improvement >= EDGE_PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch}. "
            f"No validation improvement for {EDGE_PATIENCE} epochs."
        )
        break


# ------------------------------------------------------------
# 5. RESTORE BEST CHECKPOINT
# ------------------------------------------------------------

edge_model.load_state_dict(best_edge_state)
edge_model.eval()


# ------------------------------------------------------------
# 6. SAVE TRAINING HISTORY
# ------------------------------------------------------------

edge_history_df = pd.DataFrame(edge_history)

edge_history_df.to_csv(
    EDGE_HISTORY_PATH,
    index=False
)


# ------------------------------------------------------------
# 7. DISPLAY FINAL RESULTS
# ------------------------------------------------------------

elapsed_time = time.time() - start_time

best_epoch = int(
    edge_history_df.loc[
        edge_history_df["validation_loss"].idxmin(),
        "epoch"
    ]
)

print("\n" + "=" * 60)
print("EDGE GCN TRAINING COMPLETED")
print("=" * 60)

print("Epochs completed:", len(edge_history_df))
print("Best epoch:", best_epoch)
print("Best validation loss:", best_edge_val_loss)
print("Final training loss:", edge_history_df.iloc[-1]["train_loss"])
print("Elapsed time (minutes):", round(elapsed_time / 60, 2))

print("\nBest checkpoint:")
print(EDGE_CHECKPOINT)

print("\nTraining history:")
print(EDGE_HISTORY_PATH)

print("\nBest EdgeGCN weights restored.")
print("Test dataset has NOT been used.")


Epoch 001/300 | Train Loss: 0.00154825 | Validation Loss: 0.00037737 | Best Validation: 0.00037737
Epoch 010/300 | Train Loss: 0.00063213 | Validation Loss: 0.00046862 | Best Validation: 0.00035915
Epoch 020/300 | Train Loss: 0.00051100 | Validation Loss: 0.00052636 | Best Validation: 0.00035915
Epoch 030/300 | Train Loss: 0.00049491 | Validation Loss: 0.00063578 | Best Validation: 0.00035915

Early stopping at epoch 33. No validation improvement for 30 epochs.

EDGE GCN TRAINING COMPLETED
Epochs completed: 33
Best epoch: 3
Best validation loss: 0.0003591528587700547
Final training loss: 0.0003590850515816266
Elapsed time (minutes): 0.56

Best checkpoint:
/content/drive/MyDrive/CAD_GNN_Project/training_results/edge_gcn/best_edge_gcn.pt

Training history:
/content/drive/MyDrive/CAD_GNN_Project/training_results/edge_gcn/edge_training_history.csv

Best EdgeGCN weights restored.
Test dataset has NOT been used.


In [23]:
#21
# ============================================================
# STEP 3C: EDGE GCN - FINAL TEST EVALUATION
# ============================================================
# Loads the best EdgeGCN checkpoint.
# Evaluates the 15 untouched test graphs.
# Does not train or modify model weights.
#
# Saves:
#   edge_test_predictions.csv
#   edge_test_metrics.json
# ============================================================

import json
import numpy as np
import pandas as pd
import torch

# ------------------------------------------------------------
# 1. DEFINE OUTPUT PATHS
# ------------------------------------------------------------

EDGE_PREDICTIONS_PATH = (
    EDGE_OUTPUT_DIR / "edge_test_predictions.csv"
)

EDGE_METRICS_PATH = (
    EDGE_OUTPUT_DIR / "edge_test_metrics.json"
)

# Prevent accidental overwriting of previous results.
assert not EDGE_PREDICTIONS_PATH.exists(), (
    "Test predictions already exist. Stop to protect previous results."
)

assert not EDGE_METRICS_PATH.exists(), (
    "Test metrics already exist. Stop to protect previous results."
)

# ------------------------------------------------------------
# 2. LOAD THE BEST CHECKPOINT
# ------------------------------------------------------------

checkpoint = torch.load(
    EDGE_CHECKPOINT,
    map_location=EDGE_DEVICE,
    weights_only=False
)

edge_model.load_state_dict(checkpoint["model_state_dict"])
edge_model.eval()

print("Loaded checkpoint from epoch:", checkpoint["epoch"])
print("Best validation MSE:", checkpoint["best_validation_loss"])

# ------------------------------------------------------------
# 3. PREDICT ALL 15 TEST GRAPHS
# ------------------------------------------------------------

prediction_records = []

with torch.no_grad():

    for model_id, graph in edge_test_graphs.items():

        # Retrieve edge-node features and edge adjacency.
        x = graph.x.float().to(EDGE_DEVICE)
        edge_index = graph.edge_index.long().to(EDGE_DEVICE)

        # Predict normalized volume.
        predicted_y = edge_model(x, edge_index)

        predicted_y = float(predicted_y.item())

        # Retrieve actual target and physical geometry values.
        actual_y = float(graph.y.reshape(-1)[0].item())
        actual_volume = float(graph.volume.reshape(-1)[0].item())
        diagonal = float(graph.diag.reshape(-1)[0].item())

        # Convert normalized prediction to physical volume.
        predicted_volume = predicted_y * (diagonal ** 3)

        # Calculate prediction errors.
        absolute_error = abs(predicted_volume - actual_volume)

        relative_error_percent = (
            absolute_error / actual_volume
        ) * 100.0

        prediction_records.append({
            "filename": model_id,
            "actual_normalized_volume": actual_y,
            "predicted_normalized_volume": predicted_y,
            "normalized_absolute_error": abs(predicted_y - actual_y),
            "bounding_box_diagonal": diagonal,
            "actual_volume": actual_volume,
            "predicted_volume": predicted_volume,
            "absolute_volume_error": absolute_error,
            "relative_volume_error_percent": relative_error_percent
        })

# Store predictions for every test model.
edge_predictions_df = pd.DataFrame(prediction_records)

# ------------------------------------------------------------
# 4. CALCULATE NORMALIZED VOLUME METRICS
# ------------------------------------------------------------

actual_y = edge_predictions_df[
    "actual_normalized_volume"
].to_numpy()

predicted_y = edge_predictions_df[
    "predicted_normalized_volume"
].to_numpy()

normalized_errors = predicted_y - actual_y

normalized_mae = float(
    np.mean(np.abs(normalized_errors))
)

normalized_rmse = float(
    np.sqrt(np.mean(normalized_errors ** 2))
)

ss_res = float(np.sum(normalized_errors ** 2))
ss_tot = float(np.sum((actual_y - np.mean(actual_y)) ** 2))

normalized_r2 = (
    float(1 - ss_res / ss_tot)
    if ss_tot > 0 else None
)

# ------------------------------------------------------------
# 5. CALCULATE PHYSICAL VOLUME METRICS
# ------------------------------------------------------------

actual_volume = edge_predictions_df[
    "actual_volume"
].to_numpy()

predicted_volume = edge_predictions_df[
    "predicted_volume"
].to_numpy()

volume_errors = predicted_volume - actual_volume

volume_mae = float(
    np.mean(np.abs(volume_errors))
)

volume_rmse = float(
    np.sqrt(np.mean(volume_errors ** 2))
)

volume_ss_tot = float(
    np.sum((actual_volume - np.mean(actual_volume)) ** 2)
)

volume_r2 = (
    float(
        1 - np.sum(volume_errors ** 2) / volume_ss_tot
    )
    if volume_ss_tot > 0 else None
)

mean_relative_error = float(
    edge_predictions_df[
        "relative_volume_error_percent"
    ].mean()
)

# ------------------------------------------------------------
# 6. SAVE PREDICTIONS AND METRICS
# ------------------------------------------------------------

edge_predictions_df.to_csv(
    EDGE_PREDICTIONS_PATH,
    index=False
)

edge_test_metrics = {
    "model": "EdgeGCN",
    "checkpoint_epoch": int(checkpoint["epoch"]),
    "number_of_test_graphs": len(edge_test_graphs),
    "normalized_volume_MAE": normalized_mae,
    "normalized_volume_RMSE": normalized_rmse,
    "normalized_volume_R2": normalized_r2,
    "physical_volume_MAE": volume_mae,
    "physical_volume_RMSE": volume_rmse,
    "physical_volume_R2": volume_r2,
    "mean_relative_volume_error_percent": mean_relative_error
}

with open(EDGE_METRICS_PATH, "w") as file:
    json.dump(edge_test_metrics, file, indent=4)

# ------------------------------------------------------------
# 7. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("EDGE GCN - FINAL TEST RESULTS")
print("=" * 60)

print("Test graphs:", len(edge_test_graphs))

print("\nNORMALIZED VOLUME METRICS")
print("MAE :", normalized_mae)
print("RMSE:", normalized_rmse)
print("R²  :", normalized_r2)

print("\nPHYSICAL VOLUME METRICS")
print("MAE :", volume_mae)
print("RMSE:", volume_rmse)
print("R²  :", volume_r2)

print(
    "Mean relative volume error (%):",
    mean_relative_error
)

print("\nINDIVIDUAL TEST PREDICTIONS")
display(edge_predictions_df)

print("\nSaved files:")
print(EDGE_PREDICTIONS_PATH)
print(EDGE_METRICS_PATH)

print("\nEdgeGCN test evaluation completed.")


Loaded checkpoint from epoch: 3
Best validation MSE: 0.0003591528587700547

EDGE GCN - FINAL TEST RESULTS
Test graphs: 15

NORMALIZED VOLUME METRICS
MAE : 0.021234697953332216
RMSE: 0.02452163931355052
R²  : -1.5232523623706364

PHYSICAL VOLUME METRICS
MAE : 19720787.63670117
RMSE: 43167120.032866165
R²  : -38.2050415499965
Mean relative volume error (%): 838.9422265988677

INDIVIDUAL TEST PREDICTIONS


,filename,actual_normalized_volume,predicted_normalized_volume,normalized_absolute_error,bounding_box_diagonal,actual_volume,predicted_volume,absolute_volume_error,relative_volume_error_percent
0,00000013_c2f4d27f35ed4c138caf5c18_step_000.step,0.062463,0.026466,0.035997,59.460583,1.313129e+04,5.563795e+03,7.567492e+03,57.629479
1,00000015_b08aa818955948c690fd9b6d_step_000.step,0.019105,0.017943,0.001162,280.508179,4.216787e+05,3.960313e+05,2.564734e+04,6.082199
2,00000048_efe12a13ac2e4ea7880a66d1_step_000.step,0.002841,0.018956,0.016115,1414.248901,8.036504e+06,5.361862e+07,4.558212e+07,567.188389
3,00000049_aa208c45e8734584a147f595_step_000.step,0.018208,0.027699,0.009491,61.040611,4.141228e+03,6.299804e+03,2.158576e+03,52.124073
4,00000050_80d90bfdd2e74e709956122a_step_000.step,0.006982,0.027700,0.020719,134.933777,1.715210e+04,6.805245e+04,5.090036e+04,296.758816
5,00000251_66fe5f0992fd4065b4838668_step_000.step,0.000700,0.043508,0.042808,1501.339722,2.368534e+06,1.472338e+08,1.448653e+08,6116.244946
6,00000248_e87554b09cdc48c3add39db9_step_000.step,0.008626,0.028873,0.020247,9.180809,6.674980e+00,2.234263e+01,1.566765e+01,234.721983
7,00000132_438507a38e404bfdbd51a16b_step_000.step,0.023331,0.072531,0.049200,22.704969,2.730829e+02,8.489589e+02,5.758759e+02,210.879489
8,00000083_9ea934d3400e45fb9d190941_step_000.step,0.028270,0.010475,0.017794,417.534363,2.057763e+06,7.624977e+05,1.295265e+06,62.945314
9,00000084_9a051df11afa4fc5a1d667cd_step_000.step,0.005894,0.015917,0.010023,1673.240356,2.761327e+07,7.456613e+07,4.695285e+07,170.037251



Saved files:
/content/drive/MyDrive/CAD_GNN_Project/training_results/edge_gcn/edge_test_predictions.csv
/content/drive/MyDrive/CAD_GNN_Project/training_results/edge_gcn/edge_test_metrics.json

EdgeGCN test evaluation completed.


In [24]:
#22
# ============================================================
# STEP 4A: INITIALIZE VERTEX GCN
# ============================================================
# Uses the existing vertex graph dataset.
# Creates an independent model and optimizer.
# Does not start training or modify original graph files.
# ============================================================

import torch
import torch.nn as nn
from torch_geometric.nn import GCNConv
from pathlib import Path

# ------------------------------------------------------------
# 1. REPRODUCIBILITY AND DEVICE
# ------------------------------------------------------------

VERTEX_SEED = 42

torch.manual_seed(VERTEX_SEED)

VERTEX_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------------------
# 2. EXPERIMENTAL CONFIGURATION
# ------------------------------------------------------------

VERTEX_INPUT_DIM = 3
VERTEX_HIDDEN_DIM = 64
VERTEX_DROPOUT = 0.2

VERTEX_LEARNING_RATE = 0.001
VERTEX_EPOCHS = 300
VERTEX_PATIENCE = 30

# ------------------------------------------------------------
# 3. LOAD EXISTING VERTEX GRAPH SPLITS
# ------------------------------------------------------------

vertex_train_graphs = graph_splits["vertex"]["train"]
vertex_validation_graphs = graph_splits["vertex"]["validation"]
vertex_test_graphs = graph_splits["vertex"]["test"]

assert len(vertex_train_graphs) == 70
assert len(vertex_validation_graphs) == 15
assert len(vertex_test_graphs) == 15

# Confirm the expected feature dimension.
for graph in vertex_train_graphs.values():
    assert graph.x.shape[1] == VERTEX_INPUT_DIM

# ------------------------------------------------------------
# 4. DEFINE VERTEX GCN ARCHITECTURE
# ------------------------------------------------------------

class VertexGCN(nn.Module):

    def __init__(self):

        super().__init__()

        # Three graph convolution layers.
        self.conv1 = GCNConv(
            VERTEX_INPUT_DIM,
            VERTEX_HIDDEN_DIM
        )

        self.conv2 = GCNConv(
            VERTEX_HIDDEN_DIM,
            VERTEX_HIDDEN_DIM
        )

        self.conv3 = GCNConv(
            VERTEX_HIDDEN_DIM,
            VERTEX_HIDDEN_DIM
        )

        # Dropout probability.
        self.dropout = VERTEX_DROPOUT

        # Regression head: 64 -> 32 -> 1.
        self.regressor = nn.Sequential(
            nn.Linear(VERTEX_HIDDEN_DIM, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x, edge_index):

        # First graph convolution.
        x = self.conv1(x, edge_index)
        x = torch.relu(x)
        x = torch.nn.functional.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        # Second graph convolution.
        x = self.conv2(x, edge_index)
        x = torch.relu(x)
        x = torch.nn.functional.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        # Third graph convolution.
        x = self.conv3(x, edge_index)
        x = torch.relu(x)

        # Convert vertex embeddings into a graph embedding.
        graph_embedding = x.mean(
            dim=0,
            keepdim=True
        )

        # Predict normalized volume.
        prediction = self.regressor(graph_embedding)

        return prediction

# ------------------------------------------------------------
# 5. INITIALIZE INDEPENDENT MODEL AND OPTIMIZER
# ------------------------------------------------------------

vertex_model = VertexGCN().to(VERTEX_DEVICE)

vertex_optimizer = torch.optim.Adam(
    vertex_model.parameters(),
    lr=VERTEX_LEARNING_RATE
)

vertex_criterion = nn.MSELoss()

# ------------------------------------------------------------
# 6. CREATE DRIVE OUTPUT DIRECTORY
# ------------------------------------------------------------

VERTEX_OUTPUT_DIR = (
    Path("/content/drive/MyDrive/CAD_GNN_Project")
    / "training_results"
    / "vertex_gcn"
)

VERTEX_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

VERTEX_CHECKPOINT = (
    VERTEX_OUTPUT_DIR / "best_vertex_gcn.pt"
)

# Protect existing results.
assert not VERTEX_CHECKPOINT.exists(), (
    "VertexGCN checkpoint already exists. "
    "Stop to protect previous results."
)

# ------------------------------------------------------------
# 7. DISPLAY INITIALIZATION SUMMARY
# ------------------------------------------------------------

parameter_count = sum(
    parameter.numel()
    for parameter in vertex_model.parameters()
)

print("=" * 55)
print("VERTEX GCN INITIALIZED")
print("=" * 55)

print("Device:", VERTEX_DEVICE)
print("Training graphs:", len(vertex_train_graphs))
print("Validation graphs:", len(vertex_validation_graphs))
print("Test graphs:", len(vertex_test_graphs))

print("Input dimension:", VERTEX_INPUT_DIM)
print("Hidden dimension:", VERTEX_HIDDEN_DIM)
print("GCN layers:", 3)
print("Regression head: 64 -> 32 -> 1")
print("Dropout:", VERTEX_DROPOUT)
print("Learning rate:", VERTEX_LEARNING_RATE)
print("Maximum epochs:", VERTEX_EPOCHS)
print("Early stopping patience:", VERTEX_PATIENCE)

print("\nModel parameters:", parameter_count)
print("Checkpoint location:", VERTEX_CHECKPOINT)

print("\nVertexGCN initialized. Training has NOT started.")


VERTEX GCN INITIALIZED
Device: cpu
Training graphs: 70
Validation graphs: 15
Test graphs: 15
Input dimension: 3
Hidden dimension: 64
GCN layers: 3
Regression head: 64 -> 32 -> 1
Dropout: 0.2
Learning rate: 0.001
Maximum epochs: 300
Early stopping patience: 30

Model parameters: 10689
Checkpoint location: /content/drive/MyDrive/CAD_GNN_Project/training_results/vertex_gcn/best_vertex_gcn.pt

VertexGCN initialized. Training has NOT started.


In [25]:
#23

# ============================================================
# STEP 4B: TRAIN VERTEX GCN
# ============================================================

import copy
import time
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. TRAINING FUNCTION
# ------------------------------------------------------------

def train_vertex_epoch():

    vertex_model.train()
    total_loss = 0.0

    # Each graph is processed independently.
    for graph in vertex_train_graphs.values():

        graph = graph.to(VERTEX_DEVICE)

        vertex_optimizer.zero_grad()

        prediction = vertex_model(
            graph.x,
            graph.edge_index
        )

        target = graph.y.view(1, 1)

        loss = vertex_criterion(prediction, target)

        loss.backward()
        vertex_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(vertex_train_graphs)


# ------------------------------------------------------------
# 2. VALIDATION FUNCTION
# ------------------------------------------------------------

def validate_vertex():

    vertex_model.eval()
    total_loss = 0.0

    # Validation does not update model parameters.
    with torch.no_grad():

        for graph in vertex_validation_graphs.values():

            graph = graph.to(VERTEX_DEVICE)

            prediction = vertex_model(
                graph.x,
                graph.edge_index
            )

            target = graph.y.view(1, 1)

            loss = vertex_criterion(prediction, target)

            total_loss += loss.item()

    return total_loss / len(vertex_validation_graphs)


# ------------------------------------------------------------
# 3. TRAINING CONFIGURATION
# ------------------------------------------------------------

best_vertex_validation = float("inf")
best_vertex_epoch = 0
epochs_without_improvement = 0

vertex_history = []

start_time = time.time()

print("=" * 60)
print("VERTEX GCN TRAINING STARTED")
print("=" * 60)


# ------------------------------------------------------------
# 4. TRAINING LOOP WITH EARLY STOPPING
# ------------------------------------------------------------

for epoch in range(1, VERTEX_EPOCHS + 1):

    train_loss = train_vertex_epoch()
    validation_loss = validate_vertex()

    # Record both losses for later analysis.
    vertex_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_loss": validation_loss
    })

    # Save checkpoint whenever validation improves.
    if validation_loss < best_vertex_validation:

        best_vertex_validation = validation_loss
        best_vertex_epoch = epoch
        epochs_without_improvement = 0

        torch.save({
            "epoch": epoch,
            "model_state_dict": vertex_model.state_dict(),
            "optimizer_state_dict": vertex_optimizer.state_dict(),
            "validation_loss": validation_loss,
            "architecture": "VertexGCN",
            "seed": VERTEX_SEED
        }, VERTEX_CHECKPOINT)

    else:
        epochs_without_improvement += 1

    # Display progress every 10 epochs and at epoch 1.
    if epoch == 1 or epoch % 10 == 0:

        print(
            f"Epoch {epoch:03d}/{VERTEX_EPOCHS} | "
            f"Train Loss: {train_loss:.8f} | "
            f"Validation Loss: {validation_loss:.8f} | "
            f"Best Validation: {best_vertex_validation:.8f}"
        )

    # Early stopping.
    if epochs_without_improvement >= VERTEX_PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch}. "
            f"No validation improvement for "
            f"{VERTEX_PATIENCE} epochs."
        )
        break


# ------------------------------------------------------------
# 5. SAVE TRAINING HISTORY TO DRIVE
# ------------------------------------------------------------

vertex_history_df = pd.DataFrame(vertex_history)

VERTEX_HISTORY_PATH = (
    VERTEX_OUTPUT_DIR / "vertex_training_history.csv"
)

assert not VERTEX_HISTORY_PATH.exists(), (
    "Training history already exists. "
    "Stop to protect previous results."
)

vertex_history_df.to_csv(
    VERTEX_HISTORY_PATH,
    index=False
)


# ------------------------------------------------------------
# 6. RESTORE BEST MODEL CHECKPOINT
# ------------------------------------------------------------

checkpoint = torch.load(
    VERTEX_CHECKPOINT,
    map_location=VERTEX_DEVICE,
    weights_only=False
)

vertex_model.load_state_dict(
    checkpoint["model_state_dict"]
)


# ------------------------------------------------------------
# 7. TRAINING SUMMARY
# ------------------------------------------------------------

elapsed_time = (time.time() - start_time) / 60

print("\n" + "=" * 60)
print("VERTEX GCN TRAINING COMPLETED")
print("=" * 60)

print("Epochs completed:", len(vertex_history))
print("Best epoch:", best_vertex_epoch)
print("Best validation loss:", best_vertex_validation)

print(
    "Final training loss:",
    vertex_history[-1]["train_loss"]
)

print(f"Elapsed time: {elapsed_time:.2f} min")

print("Best checkpoint:", VERTEX_CHECKPOINT)
print("Training history:", VERTEX_HISTORY_PATH)

print("\nBest VertexGCN weights restored.")
print("Test dataset has NOT been used.")


VERTEX GCN TRAINING STARTED
Epoch 001/300 | Train Loss: 0.00232991 | Validation Loss: 0.00044988 | Best Validation: 0.00044988
Epoch 010/300 | Train Loss: 0.00080771 | Validation Loss: 0.00052931 | Best Validation: 0.00044988
Epoch 020/300 | Train Loss: 0.00073108 | Validation Loss: 0.00053669 | Best Validation: 0.00044988
Epoch 030/300 | Train Loss: 0.00066071 | Validation Loss: 0.00061223 | Best Validation: 0.00044988

Early stopping at epoch 31. No validation improvement for 30 epochs.

VERTEX GCN TRAINING COMPLETED
Epochs completed: 31
Best epoch: 1
Best validation loss: 0.0004498769156422592
Final training loss: 0.0006624687102888264
Elapsed time: 0.32 min
Best checkpoint: /content/drive/MyDrive/CAD_GNN_Project/training_results/vertex_gcn/best_vertex_gcn.pt
Training history: /content/drive/MyDrive/CAD_GNN_Project/training_results/vertex_gcn/vertex_training_history.csv

Best VertexGCN weights restored.
Test dataset has NOT been used.


In [26]:
#24
# ============================================================
# STEP 4C: TEST EVALUATION — VERTEX GCN
# ============================================================
# Uses the restored best VertexGCN checkpoint.
# Evaluates the 15 held-out test graphs exactly once.
# Saves predictions and metrics directly to Google Drive.
# ============================================================

import json
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ------------------------------------------------------------
# 1. PROTECT EXISTING RESULTS
# ------------------------------------------------------------

VERTEX_PREDICTIONS_PATH = (
    VERTEX_OUTPUT_DIR / "vertex_test_predictions.csv"
)

VERTEX_METRICS_PATH = (
    VERTEX_OUTPUT_DIR / "vertex_test_metrics.json"
)

assert not VERTEX_PREDICTIONS_PATH.exists(), (
    "Vertex test predictions already exist. Stopping."
)

assert not VERTEX_METRICS_PATH.exists(), (
    "Vertex test metrics already exist. Stopping."
)

# ------------------------------------------------------------
# 2. RESTORE BEST CHECKPOINT
# ------------------------------------------------------------

checkpoint = torch.load(
    VERTEX_CHECKPOINT,
    map_location=VERTEX_DEVICE,
    weights_only=False
)

vertex_model.load_state_dict(
    checkpoint["model_state_dict"]
)

vertex_model.eval()

# ------------------------------------------------------------
# 3. GENERATE TEST PREDICTIONS
# ------------------------------------------------------------

vertex_results = []

with torch.no_grad():

    for filename, graph in vertex_test_graphs.items():

        graph = graph.to(VERTEX_DEVICE)

        # Predict normalized volume.
        prediction = vertex_model(
            graph.x,
            graph.edge_index
        )

        predicted_normalized_volume = prediction.item()
        actual_normalized_volume = graph.y.item()

        # Recover physical volume using stored bbox diagonal.
        diagonal = graph.diag.item()

        predicted_volume = (
            predicted_normalized_volume * diagonal**3
        )

        actual_volume = graph.volume.item()

        vertex_results.append({
            "filename": filename,
            "actual_normalized_volume": actual_normalized_volume,
            "predicted_normalized_volume": predicted_normalized_volume,
            "normalized_absolute_error": abs(
                actual_normalized_volume -
                predicted_normalized_volume
            ),
            "bbox_diagonal": diagonal,
            "actual_volume": actual_volume,
            "predicted_volume": predicted_volume,
            "absolute_volume_error": abs(
                actual_volume - predicted_volume
            ),
            "relative_volume_error_percent": (
                abs(actual_volume - predicted_volume)
                / abs(actual_volume)
            ) * 100
        })

# ------------------------------------------------------------
# 4. CREATE PREDICTION TABLE
# ------------------------------------------------------------

vertex_predictions_df = pd.DataFrame(vertex_results)

y_true_normalized = (
    vertex_predictions_df["actual_normalized_volume"].to_numpy()
)

y_pred_normalized = (
    vertex_predictions_df["predicted_normalized_volume"].to_numpy()
)

y_true_volume = (
    vertex_predictions_df["actual_volume"].to_numpy()
)

y_pred_volume = (
    vertex_predictions_df["predicted_volume"].to_numpy()
)

# ------------------------------------------------------------
# 5. CALCULATE PERFORMANCE METRICS
# ------------------------------------------------------------

vertex_metrics = {

    "model": "VertexGCN",
    "checkpoint_epoch": int(checkpoint["epoch"]),
    "test_graphs": len(vertex_predictions_df),

    "normalized_MAE": float(
        mean_absolute_error(
            y_true_normalized,
            y_pred_normalized
        )
    ),

    "normalized_RMSE": float(
        np.sqrt(
            mean_squared_error(
                y_true_normalized,
                y_pred_normalized
            )
        )
    ),

    "normalized_R2": float(
        r2_score(
            y_true_normalized,
            y_pred_normalized
        )
    ),

    "physical_volume_MAE": float(
        mean_absolute_error(
            y_true_volume,
            y_pred_volume
        )
    ),

    "physical_volume_RMSE": float(
        np.sqrt(
            mean_squared_error(
                y_true_volume,
                y_pred_volume
            )
        )
    ),

    "physical_volume_R2": float(
        r2_score(
            y_true_volume,
            y_pred_volume
        )
    ),

    "mean_relative_volume_error_percent": float(
        vertex_predictions_df[
            "relative_volume_error_percent"
        ].mean()
    )
}

# ------------------------------------------------------------
# 6. SAVE RESULTS TO DRIVE
# ------------------------------------------------------------

vertex_predictions_df.to_csv(
    VERTEX_PREDICTIONS_PATH,
    index=False
)

with open(VERTEX_METRICS_PATH, "w") as file:
    json.dump(
        vertex_metrics,
        file,
        indent=4
    )

# ------------------------------------------------------------
# 7. DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 60)
print("VERTEX GCN — TEST RESULTS")
print("=" * 60)

for metric, value in vertex_metrics.items():
    print(f"{metric}: {value}")

print("\nPrediction sample:")
print(vertex_predictions_df.head(10).to_string(index=False))

print("\nSaved predictions:", VERTEX_PREDICTIONS_PATH)
print("Saved metrics:", VERTEX_METRICS_PATH)

print("\nVertexGCN test evaluation completed.")


VERTEX GCN — TEST RESULTS
model: VertexGCN
checkpoint_epoch: 1
test_graphs: 15
normalized_MAE: 0.018630409593849134
normalized_RMSE: 0.021457078709270144
normalized_R2: -0.9319811146712853
physical_volume_MAE: 25750956.423920415
physical_volume_RMSE: 51478546.68217816
physical_volume_R2: -54.755578186100855
mean_relative_volume_error_percent: 889.2343809454225

Prediction sample:
                                       filename  actual_normalized_volume  predicted_normalized_volume  normalized_absolute_error  bbox_diagonal  actual_volume  predicted_volume  absolute_volume_error  relative_volume_error_percent
00000013_c2f4d27f35ed4c138caf5c18_step_000.step                  0.062463                     0.026245                   0.036218      59.460583   1.313129e+04      5.517336e+03           7.613951e+03                      57.983278
00000015_b08aa818955948c690fd9b6d_step_000.step                  0.019105                     0.029262                   0.010157     280.508179   4.2167

In [27]:
#25

# ============================================================
# STEP 5A: INITIALIZE HETEROGENEOUS GCN
# ============================================================
# Preserves the original HeteroData node types and relations.
# Creates an independent model and optimizer.
# Does not start training or modify the graph dataset.
# ============================================================

import torch
import torch.nn as nn
from torch_geometric.nn import HeteroConv, GraphConv
from pathlib import Path

# ------------------------------------------------------------
# 1. REPRODUCIBILITY AND DEVICE
# ------------------------------------------------------------

HETERO_SEED = 42
torch.manual_seed(HETERO_SEED)

HETERO_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------------------
# 2. EXPERIMENTAL CONFIGURATION
# ------------------------------------------------------------

HETERO_HIDDEN_DIM = 64
HETERO_DROPOUT = 0.2

HETERO_LEARNING_RATE = 0.001
HETERO_EPOCHS = 300
HETERO_PATIENCE = 30

# Original heterogeneous node types.
HETERO_NODE_TYPES = ["face", "edge", "vertex"]

# Original directed relation types.
HETERO_EDGE_TYPES = [
    ("face", "boundary", "edge"),
    ("edge", "belongs_to", "face"),
    ("edge", "endpoint", "vertex"),
    ("vertex", "connected_by", "edge")
]

# ------------------------------------------------------------
# 3. LOAD EXISTING HETEROGENEOUS GRAPH SPLITS
# ------------------------------------------------------------

hetero_train_graphs = graph_splits["hetero"]["train"]
hetero_validation_graphs = graph_splits["hetero"]["validation"]
hetero_test_graphs = graph_splits["hetero"]["test"]

assert len(hetero_train_graphs) == 70
assert len(hetero_validation_graphs) == 15
assert len(hetero_test_graphs) == 15

# Confirm the original feature dimensions.
HETERO_INPUT_DIMS = {
    "face": 14,
    "edge": 4,
    "vertex": 3
}

for graph in hetero_train_graphs.values():

    assert set(graph.node_types) == set(HETERO_NODE_TYPES)

    for node_type, expected_dim in HETERO_INPUT_DIMS.items():

        assert graph[node_type].x.shape[1] == expected_dim

    assert set(graph.edge_types) == set(HETERO_EDGE_TYPES)

# ------------------------------------------------------------
# 4. DEFINE HETEROGENEOUS GCN ARCHITECTURE
# ------------------------------------------------------------

class HeterogeneousGCN(nn.Module):

    def __init__(self):

        super().__init__()

        # ----------------------------------------------------
        # Layer 1: Original feature dimensions -> 64
        # Each relation has its own GraphConv operator.
        # ----------------------------------------------------

        self.conv1 = HeteroConv({

            ("face", "boundary", "edge"): GraphConv(
                (14, 4), HETERO_HIDDEN_DIM
            ),

            ("edge", "belongs_to", "face"): GraphConv(
                (4, 14), HETERO_HIDDEN_DIM
            ),

            ("edge", "endpoint", "vertex"): GraphConv(
                (4, 3), HETERO_HIDDEN_DIM
            ),

            ("vertex", "connected_by", "edge"): GraphConv(
                (3, 4), HETERO_HIDDEN_DIM
            )

        }, aggr="sum")

        # ----------------------------------------------------
        # Layers 2 and 3: 64 -> 64
        # ----------------------------------------------------

        self.conv2 = HeteroConv({

            edge_type: GraphConv(
                (HETERO_HIDDEN_DIM, HETERO_HIDDEN_DIM),
                HETERO_HIDDEN_DIM
            )

            for edge_type in HETERO_EDGE_TYPES

        }, aggr="sum")

        self.conv3 = HeteroConv({

            edge_type: GraphConv(
                (HETERO_HIDDEN_DIM, HETERO_HIDDEN_DIM),
                HETERO_HIDDEN_DIM
            )

            for edge_type in HETERO_EDGE_TYPES

        }, aggr="sum")

        self.dropout = HETERO_DROPOUT

        # Regression head: 64 -> 32 -> 1.
        self.regressor = nn.Sequential(
            nn.Linear(HETERO_HIDDEN_DIM, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x_dict, edge_index_dict):

        # First heterogeneous message-passing layer.
        x_dict = self.conv1(x_dict, edge_index_dict)

        x_dict = {
            node_type: torch.relu(x)
            for node_type, x in x_dict.items()
        }

        x_dict = {
            node_type: torch.nn.functional.dropout(
                x,
                p=self.dropout,
                training=self.training
            )
            for node_type, x in x_dict.items()
        }

        # Second heterogeneous message-passing layer.
        x_dict = self.conv2(x_dict, edge_index_dict)

        x_dict = {
            node_type: torch.relu(x)
            for node_type, x in x_dict.items()
        }

        x_dict = {
            node_type: torch.nn.functional.dropout(
                x,
                p=self.dropout,
                training=self.training
            )
            for node_type, x in x_dict.items()
        }

        # Third heterogeneous message-passing layer.
        x_dict = self.conv3(x_dict, edge_index_dict)

        x_dict = {
            node_type: torch.relu(x)
            for node_type, x in x_dict.items()
        }

        # Mean-pool each node type independently.
        pooled_embeddings = [
            x_dict[node_type].mean(dim=0)
            for node_type in HETERO_NODE_TYPES
        ]

        # Average the three type-level graph embeddings.
        # Each node type contributes equally to the final
        # 64-dimensional graph representation.
        graph_embedding = torch.stack(
            pooled_embeddings,
            dim=0
        ).mean(dim=0, keepdim=True)

        # Predict normalized volume.
        prediction = self.regressor(graph_embedding)

        return prediction


# ------------------------------------------------------------
# 5. INITIALIZE INDEPENDENT MODEL AND OPTIMIZER
# ------------------------------------------------------------

hetero_model = HeterogeneousGCN().to(HETERO_DEVICE)

hetero_optimizer = torch.optim.Adam(
    hetero_model.parameters(),
    lr=HETERO_LEARNING_RATE
)

hetero_criterion = nn.MSELoss()

# ------------------------------------------------------------
# 6. CREATE DRIVE OUTPUT DIRECTORY
# ------------------------------------------------------------

HETERO_OUTPUT_DIR = (
    Path("/content/drive/MyDrive/CAD_GNN_Project")
    / "training_results"
    / "hetero_gcn"
)

HETERO_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

HETERO_CHECKPOINT = (
    HETERO_OUTPUT_DIR / "best_hetero_gcn.pt"
)

assert not HETERO_CHECKPOINT.exists(), (
    "Heterogeneous checkpoint already exists. "
    "Stop to protect previous results."
)

# ------------------------------------------------------------
# 7. DISPLAY INITIALIZATION SUMMARY
# ------------------------------------------------------------

parameter_count = sum(
    parameter.numel()
    for parameter in hetero_model.parameters()
)

print("=" * 60)
print("HETEROGENEOUS GCN INITIALIZED")
print("=" * 60)

print("Device:", HETERO_DEVICE)
print("Training graphs:", len(hetero_train_graphs))
print("Validation graphs:", len(hetero_validation_graphs))
print("Test graphs:", len(hetero_test_graphs))

print("Node types:", HETERO_NODE_TYPES)
print("Input dimensions:", HETERO_INPUT_DIMS)
print("Relations:", HETERO_EDGE_TYPES)

print("Message-passing layers:", 3)
print("Hidden dimension:", HETERO_HIDDEN_DIM)
print("Aggregation across relations: sum")
print("Pooling: equal average of three node-type means")
print("Regression head: 64 -> 32 -> 1")

print("Dropout:", HETERO_DROPOUT)
print("Learning rate:", HETERO_LEARNING_RATE)
print("Maximum epochs:", HETERO_EPOCHS)
print("Early stopping patience:", HETERO_PATIENCE)

print("\nModel parameters:", parameter_count)
print("Checkpoint location:", HETERO_CHECKPOINT)

print("\nHeterogeneous GCN initialized.")
print("Training has NOT started.")


HETEROGENEOUS GCN INITIALIZED
Device: cpu
Training graphs: 70
Validation graphs: 15
Test graphs: 15
Node types: ['face', 'edge', 'vertex']
Input dimensions: {'face': 14, 'edge': 4, 'vertex': 3}
Relations: [('face', 'boundary', 'edge'), ('edge', 'belongs_to', 'face'), ('edge', 'endpoint', 'vertex'), ('vertex', 'connected_by', 'edge')]
Message-passing layers: 3
Hidden dimension: 64
Aggregation across relations: sum
Pooling: equal average of three node-type means
Regression head: 64 -> 32 -> 1
Dropout: 0.2
Learning rate: 0.001
Maximum epochs: 300
Early stopping patience: 30

Model parameters: 71617
Checkpoint location: /content/drive/MyDrive/CAD_GNN_Project/training_results/hetero_gcn/best_hetero_gcn.pt

Heterogeneous GCN initialized.
Training has NOT started.


In [28]:
#26

# ============================================================
# STEP 5B: TRAIN HETEROGENEOUS GCN
# ============================================================

import time
import pandas as pd
import torch

# ------------------------------------------------------------
# 1. TRAINING FUNCTION
# ------------------------------------------------------------

def train_hetero_epoch():

    hetero_model.train()
    total_loss = 0.0

    for graph in hetero_train_graphs.values():

        graph = graph.to(HETERO_DEVICE)

        hetero_optimizer.zero_grad()

        prediction = hetero_model(
            graph.x_dict,
            graph.edge_index_dict
        )

        target = graph.y.view(1, 1)

        loss = hetero_criterion(prediction, target)

        loss.backward()
        hetero_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(hetero_train_graphs)


# ------------------------------------------------------------
# 2. VALIDATION FUNCTION
# ------------------------------------------------------------

def validate_hetero():

    hetero_model.eval()
    total_loss = 0.0

    with torch.no_grad():

        for graph in hetero_validation_graphs.values():

            graph = graph.to(HETERO_DEVICE)

            prediction = hetero_model(
                graph.x_dict,
                graph.edge_index_dict
            )

            target = graph.y.view(1, 1)

            loss = hetero_criterion(prediction, target)

            total_loss += loss.item()

    return total_loss / len(hetero_validation_graphs)


# ------------------------------------------------------------
# 3. TRAINING CONFIGURATION
# ------------------------------------------------------------

best_hetero_validation = float("inf")
best_hetero_epoch = 0
epochs_without_improvement = 0

hetero_history = []

start_time = time.time()

print("=" * 60)
print("HETEROGENEOUS GCN TRAINING STARTED")
print("=" * 60)


# ------------------------------------------------------------
# 4. TRAINING LOOP WITH EARLY STOPPING
# ------------------------------------------------------------

for epoch in range(1, HETERO_EPOCHS + 1):

    train_loss = train_hetero_epoch()
    validation_loss = validate_hetero()

    hetero_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_loss": validation_loss
    })

    # Save the best validation checkpoint.
    if validation_loss < best_hetero_validation:

        best_hetero_validation = validation_loss
        best_hetero_epoch = epoch
        epochs_without_improvement = 0

        torch.save({
            "epoch": epoch,
            "model_state_dict": hetero_model.state_dict(),
            "optimizer_state_dict": hetero_optimizer.state_dict(),
            "validation_loss": validation_loss,
            "architecture": "HeterogeneousGCN",
            "seed": HETERO_SEED
        }, HETERO_CHECKPOINT)

    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 10 == 0:

        print(
            f"Epoch {epoch:03d}/{HETERO_EPOCHS} | "
            f"Train Loss: {train_loss:.8f} | "
            f"Validation Loss: {validation_loss:.8f} | "
            f"Best Validation: {best_hetero_validation:.8f}"
        )

    # Stop when validation has not improved for 30 epochs.
    if epochs_without_improvement >= HETERO_PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch}. "
            f"No validation improvement for "
            f"{HETERO_PATIENCE} epochs."
        )
        break


# ------------------------------------------------------------
# 5. SAVE TRAINING HISTORY
# ------------------------------------------------------------

hetero_history_df = pd.DataFrame(hetero_history)

HETERO_HISTORY_PATH = (
    HETERO_OUTPUT_DIR / "hetero_training_history.csv"
)

assert not HETERO_HISTORY_PATH.exists(), (
    "Training history already exists. "
    "Stop to protect previous results."
)

hetero_history_df.to_csv(
    HETERO_HISTORY_PATH,
    index=False
)


# ------------------------------------------------------------
# 6. RESTORE BEST MODEL CHECKPOINT
# ------------------------------------------------------------

checkpoint = torch.load(
    HETERO_CHECKPOINT,
    map_location=HETERO_DEVICE,
    weights_only=False
)

hetero_model.load_state_dict(
    checkpoint["model_state_dict"]
)


# ------------------------------------------------------------
# 7. TRAINING SUMMARY
# ------------------------------------------------------------

elapsed_time = (time.time() - start_time) / 60

print("\n" + "=" * 60)
print("HETEROGENEOUS GCN TRAINING COMPLETED")
print("=" * 60)

print("Epochs completed:", len(hetero_history))
print("Best epoch:", best_hetero_epoch)
print("Best validation loss:", best_hetero_validation)

print(
    "Final training loss:",
    hetero_history[-1]["train_loss"]
)

print(f"Elapsed time: {elapsed_time:.2f} min")

print("Best checkpoint:", HETERO_CHECKPOINT)
print("Training history:", HETERO_HISTORY_PATH)

print("\nBest HeterogeneousGCN weights restored.")
print("Test dataset has NOT been used.")


HETEROGENEOUS GCN TRAINING STARTED
Epoch 001/300 | Train Loss: 27.69320556 | Validation Loss: 0.02256150 | Best Validation: 0.02256150
Epoch 010/300 | Train Loss: 476.03888084 | Validation Loss: 0.13420474 | Best Validation: 0.00294740
Epoch 020/300 | Train Loss: 17.67574150 | Validation Loss: 0.01531089 | Best Validation: 0.00294740
Epoch 030/300 | Train Loss: 4.23085461 | Validation Loss: 0.00192068 | Best Validation: 0.00122492
Epoch 040/300 | Train Loss: 0.02638489 | Validation Loss: 0.00212161 | Best Validation: 0.00122492
Epoch 050/300 | Train Loss: 0.05197850 | Validation Loss: 0.00122082 | Best Validation: 0.00122082
Epoch 060/300 | Train Loss: 0.00343158 | Validation Loss: 0.00160557 | Best Validation: 0.00122082
Epoch 070/300 | Train Loss: 0.07397591 | Validation Loss: 0.00103506 | Best Validation: 0.00103506
Epoch 080/300 | Train Loss: 0.25461146 | Validation Loss: 0.00076013 | Best Validation: 0.00076013
Epoch 090/300 | Train Loss: 0.10013775 | Validation Loss: 0.00078624 |

In [29]:
#27
# ============================================================
# STEP 5C: TEST EVALUATION — HETEROGENEOUS GCN
# ============================================================

import json
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ------------------------------------------------------------
# 1. PROTECT EXISTING RESULTS
# ------------------------------------------------------------

HETERO_PREDICTIONS_PATH = (
    HETERO_OUTPUT_DIR / "hetero_test_predictions.csv"
)

HETERO_METRICS_PATH = (
    HETERO_OUTPUT_DIR / "hetero_test_metrics.json"
)

assert not HETERO_PREDICTIONS_PATH.exists(), (
    "Heterogeneous test predictions already exist. Stopping."
)

assert not HETERO_METRICS_PATH.exists(), (
    "Heterogeneous test metrics already exist. Stopping."
)

# ------------------------------------------------------------
# 2. RESTORE BEST CHECKPOINT
# ------------------------------------------------------------

checkpoint = torch.load(
    HETERO_CHECKPOINT,
    map_location=HETERO_DEVICE,
    weights_only=False
)

hetero_model.load_state_dict(
    checkpoint["model_state_dict"]
)

hetero_model.eval()

# ------------------------------------------------------------
# 3. GENERATE TEST PREDICTIONS
# ------------------------------------------------------------

hetero_results = []

with torch.no_grad():

    for filename, graph in hetero_test_graphs.items():

        graph = graph.to(HETERO_DEVICE)

        # Predict normalized volume using all node types
        # and their original heterogeneous relations.
        prediction = hetero_model(
            graph.x_dict,
            graph.edge_index_dict
        )

        predicted_normalized_volume = prediction.item()
        actual_normalized_volume = graph.y.item()

        # Recover physical volume using stored diagonal.
        diagonal = graph.diag.item()

        predicted_volume = (
            predicted_normalized_volume * diagonal**3
        )

        actual_volume = graph.volume.item()

        hetero_results.append({

            "filename": filename,

            "actual_normalized_volume":
                actual_normalized_volume,

            "predicted_normalized_volume":
                predicted_normalized_volume,

            "normalized_absolute_error": abs(
                actual_normalized_volume -
                predicted_normalized_volume
            ),

            "bbox_diagonal": diagonal,

            "actual_volume": actual_volume,

            "predicted_volume": predicted_volume,

            "absolute_volume_error": abs(
                actual_volume - predicted_volume
            ),

            "relative_volume_error_percent": (
                abs(actual_volume - predicted_volume)
                / abs(actual_volume)
            ) * 100
        })

# ------------------------------------------------------------
# 4. CREATE PREDICTION TABLE
# ------------------------------------------------------------

hetero_predictions_df = pd.DataFrame(hetero_results)

y_true_normalized = (
    hetero_predictions_df[
        "actual_normalized_volume"
    ].to_numpy()
)

y_pred_normalized = (
    hetero_predictions_df[
        "predicted_normalized_volume"
    ].to_numpy()
)

y_true_volume = (
    hetero_predictions_df["actual_volume"].to_numpy()
)

y_pred_volume = (
    hetero_predictions_df["predicted_volume"].to_numpy()
)

# ------------------------------------------------------------
# 5. CALCULATE PERFORMANCE METRICS
# ------------------------------------------------------------

hetero_metrics = {

    "model": "HeterogeneousGCN",

    "checkpoint_epoch": int(checkpoint["epoch"]),

    "test_graphs": len(hetero_predictions_df),

    "normalized_MAE": float(
        mean_absolute_error(
            y_true_normalized,
            y_pred_normalized
        )
    ),

    "normalized_RMSE": float(
        np.sqrt(
            mean_squared_error(
                y_true_normalized,
                y_pred_normalized
            )
        )
    ),

    "normalized_R2": float(
        r2_score(
            y_true_normalized,
            y_pred_normalized
        )
    ),

    "physical_volume_MAE": float(
        mean_absolute_error(
            y_true_volume,
            y_pred_volume
        )
    ),

    "physical_volume_RMSE": float(
        np.sqrt(
            mean_squared_error(
                y_true_volume,
                y_pred_volume
            )
        )
    ),

    "physical_volume_R2": float(
        r2_score(
            y_true_volume,
            y_pred_volume
        )
    ),

    "mean_relative_volume_error_percent": float(
        hetero_predictions_df[
            "relative_volume_error_percent"
        ].mean()
    )
}

# ------------------------------------------------------------
# 6. SAVE RESULTS TO DRIVE
# ------------------------------------------------------------

hetero_predictions_df.to_csv(
    HETERO_PREDICTIONS_PATH,
    index=False
)

with open(HETERO_METRICS_PATH, "w") as file:

    json.dump(
        hetero_metrics,
        file,
        indent=4
    )

# ------------------------------------------------------------
# 7. DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 60)
print("HETEROGENEOUS GCN — TEST RESULTS")
print("=" * 60)

for metric, value in hetero_metrics.items():
    print(f"{metric}: {value}")

print("\nPrediction sample:")
print(
    hetero_predictions_df.head(10).to_string(index=False)
)

print("\nSaved predictions:", HETERO_PREDICTIONS_PATH)
print("Saved metrics:", HETERO_METRICS_PATH)

print("\nHeterogeneousGCN test evaluation completed.")


HETEROGENEOUS GCN — TEST RESULTS
model: HeterogeneousGCN
checkpoint_epoch: 112
test_graphs: 15
normalized_MAE: 0.23868766004452482
normalized_RMSE: 0.8410734931932484
normalized_R2: -2967.448923466129
physical_volume_MAE: 628668285.5806632
physical_volume_RMSE: 2377906403.635865
physical_volume_R2: -118965.66343848818
mean_relative_volume_error_percent: 8497.169459987022

Prediction sample:
                                       filename  actual_normalized_volume  predicted_normalized_volume  normalized_absolute_error  bbox_diagonal  actual_volume  predicted_volume  absolute_volume_error  relative_volume_error_percent
00000013_c2f4d27f35ed4c138caf5c18_step_000.step                  0.062463                     0.014106                   0.048357      59.460583   1.313129e+04      2.965416e+03           1.016587e+04                      77.417172
00000015_b08aa818955948c690fd9b6d_step_000.step                  0.019105                     0.011121                   0.007984     280.5081

In [30]:
# =====================================================================
# BASELINES: how well can simple models predict volume / diag^3?
# No neural network here. This tells us how much room a GNN has.
# =====================================================================
import numpy as np
import pandas as pd
import torch
import torch_geometric                       # needed so saved graphs can be loaded
from pathlib import Path
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

PROJECT_DIR = Path("/content/drive/MyDrive/CAD_GNN_Project")
GRAPH_DIR   = PROJECT_DIR / "graphs"

# ---------------------------------------------------------------------
# PART 1: Load the dataset, your split, and the saved graphs
# ---------------------------------------------------------------------
dataset = pd.read_csv(PROJECT_DIR / "dataset_100.csv")
split   = pd.read_csv(PROJECT_DIR / "audit_reports" / "shared_split_100.csv")

names = dataset["filename"].tolist()
assert set(split["filename"]) == set(names), "split and dataset differ"

face_g   = torch.load(GRAPH_DIR / "face_graphs_100.pt",   map_location="cpu", weights_only=False)
edge_g   = torch.load(GRAPH_DIR / "edge_graphs_100.pt",   map_location="cpu", weights_only=False)
vertex_g = torch.load(GRAPH_DIR / "vertex_graphs_100.pt", map_location="cpu", weights_only=False)

# ---------------------------------------------------------------------
# PART 2: Turn each model's graphs into a short list of summary numbers.
# Only numbers already inside the graphs are used (all scale-free).
# Nothing here uses the volume.
# ---------------------------------------------------------------------
def summarise(name):
    fx = face_g[name].x.numpy()        # 14 numbers per face
    ex = edge_g[name].x.numpy()        # 4 numbers per edge
    nv = vertex_g[name].x.shape[0]     # number of vertices
    nf, ne = len(fx), len(ex)

    area = fx[:, 6]                    # face area / diag^2
    total_area = area.sum()

    return {
        # how many parts
        "log_faces":    np.log(nf),
        "log_edges":    np.log(ne),
        "log_vertices": np.log(nv),
        # what kinds of faces (fractions add up to 1)
        "frac_plane":    fx[:, 0].mean(),
        "frac_cylinder": fx[:, 1].mean(),
        "frac_cone":     fx[:, 2].mean(),
        "frac_sphere":   fx[:, 3].mean(),
        "frac_torus":    fx[:, 4].mean(),
        "frac_other":    fx[:, 5].mean(),
        # how much surface, and how it is spread
        "total_area_norm":     total_area,
        "log_total_area_norm": np.log(total_area + 1e-12),
        "max_face_area":       area.max(),
        "mean_face_area":      area.mean(),
        # edges
        "edge_length_sum":     ex[:, 0].sum(),
        "frac_line_edges":     ex[:, 1].mean(),
        "frac_circle_edges":   ex[:, 2].mean(),
        "frac_other_edges":    ex[:, 3].mean(),
        # how spread out the faces are around the model's centre
        "centroid_spread":     fx[:, 11:14].std(axis=0).mean(),
    }

X_df = pd.DataFrame([summarise(n) for n in names], index=names)
y = np.array([float(face_g[n].y) for n in names])        # target: volume / diag^3
y_log = np.log(y)

assert np.isfinite(X_df.to_numpy()).all(), "non-finite summary feature"
assert (y > 0).all()
print("Summary features per model:", X_df.shape[1])
print("Target range:", y.min().round(5), "to", y.max().round(5))

X = X_df.to_numpy()
index_of = {n: i for i, n in enumerate(names)}

# ---------------------------------------------------------------------
# PART 3: The models to compare
# ---------------------------------------------------------------------
def fit_predict(kind, train_idx, test_idx):
    """Fit on train_idx, return predictions (in the original y units) for test_idx."""
    if kind == "mean":                      # always predict the training average
        return np.full(len(test_idx), y[train_idx].mean())
    if kind == "geometric_mean":            # average on the log scale
        return np.full(len(test_idx), np.exp(y_log[train_idx].mean()))
    if kind == "ridge_log":                 # straight-line model on the log target
        m = make_pipeline(StandardScaler(),
                          RidgeCV(alphas=np.logspace(-3, 3, 25)))
    elif kind == "forest_log":              # many small decision trees
        m = RandomForestRegressor(n_estimators=300, min_samples_leaf=2,
                                  random_state=0, n_jobs=-1)
    else:
        raise ValueError(kind)
    m.fit(X[train_idx], y_log[train_idx])
    return np.exp(m.predict(X[test_idx]))

KINDS = ["mean", "geometric_mean", "ridge_log", "forest_log"]

def metrics(true, pred):
    rel = np.abs(pred - true) / true * 100            # relative error in %
    # Because volume = y * diag^3 and diag is known, the relative error of the
    # volume equals the relative error of y. So this is also the volume error.
    return {
        "R2":                r2_score(true, pred),
        "R2_on_log":         r2_score(np.log(true), np.log(pred)),
        "MAE":               mean_absolute_error(true, pred),
        "median_rel_err_%":  float(np.median(rel)),
    }

# ---------------------------------------------------------------------
# PART 4: Your shared split (70 train / 15 validation / 15 test)
# ---------------------------------------------------------------------
idx = {s: np.array([index_of[n] for n in split.loc[split["split"] == s, "filename"]])
       for s in ("train", "validation", "test")}
print("Split sizes:", {k: len(v) for k, v in idx.items()})

rows = []
for kind in KINDS:
    for part in ("validation", "test"):
        pred = fit_predict(kind, idx["train"], idx[part])
        rows.append({"model": kind, "set": part, **metrics(y[idx[part]], pred)})

split_table = pd.DataFrame(rows)
print("\n===== SHARED SPLIT (fit on the 70 training models) =====")
print(split_table.round(4).to_string(index=False))

# ---------------------------------------------------------------------
# PART 5: Repeated 5-fold cross-validation over all 100 models.
# The three near-identical models (00000011, 00000012, 00000207) are
# kept together in one group so they can never leak across folds.
# ---------------------------------------------------------------------
group_of = {}
for n in names:
    group_of[n] = "dup" if n[:8] in ("00000011", "00000012", "00000207") else n
groups = sorted(set(group_of.values()))

N_REPEATS, N_FOLDS = 5, 5
cv_rows = []
for repeat in range(N_REPEATS):
    rng = np.random.default_rng(repeat)
    order = rng.permutation(len(groups))
    fold_of_group = {groups[g]: k % N_FOLDS for k, g in enumerate(order)}
    fold = np.array([fold_of_group[group_of[n]] for n in names])

    for f in range(N_FOLDS):
        test_idx  = np.where(fold == f)[0]
        train_idx = np.where(fold != f)[0]
        for kind in KINDS:
            pred = fit_predict(kind, train_idx, test_idx)
            cv_rows.append({"model": kind, "repeat": repeat, "fold": f,
                            **metrics(y[test_idx], pred)})

cv = pd.DataFrame(cv_rows)
cv_summary = (cv.groupby("model")[["R2", "R2_on_log", "median_rel_err_%"]]
                .agg(["mean", "std"]).round(3).loc[KINDS])

print("\n===== REPEATED 5-FOLD CV, ALL 100 MODELS (5 repeats x 5 folds) =====")
print(cv_summary.to_string())

# ---------------------------------------------------------------------
# PART 6: Save and compare with your GCNs
# ---------------------------------------------------------------------
out = PROJECT_DIR / "training_results" / "baseline_results.csv"
out.parent.mkdir(parents=True, exist_ok=True)
pd.concat([split_table.assign(source="shared_split"),
           cv.assign(source="repeated_cv")], ignore_index=True).to_csv(out, index=False)
print("\nSaved:", out)

print("\nYour GCN test results on the same 15 test models (normalized R2):")
print("  Face -1.33 | Edge -1.52 | Vertex -0.93 | Hetero -2967")
print("Compare with the 'test' rows of the shared-split table above.")

Summary features per model: 18
Target range: 0.00013 to 0.19245
Split sizes: {'train': 70, 'validation': 15, 'test': 15}

===== SHARED SPLIT (fit on the 70 training models) =====
         model        set      R2  R2_on_log    MAE  median_rel_err_%
          mean validation -0.0520    -0.0357 0.0158           57.5883
          mean       test -0.1891    -0.6861 0.0138          184.8605
geometric_mean validation -0.6416    -0.3552 0.0171           59.1123
geometric_mean       test -0.0780    -0.0466 0.0101           57.7297
     ridge_log validation -0.3238    -0.2609 0.0164           58.6575
     ridge_log       test  0.2747     0.4685 0.0073           47.8071
    forest_log validation -0.3149    -0.2397 0.0165           59.2419
    forest_log       test  0.2092     0.4853 0.0077           45.2500

===== REPEATED 5-FOLD CV, ALL 100 MODELS (5 repeats x 5 folds) =====
                   R2        R2_on_log        median_rel_err_%        
                 mean    std      mean    std     